In [ ]:
%cd /content
!git clone https://github.com/Shruti-sivaramakrishnan/indic-fake-voice-detection.git

/content
Cloning into 'indic-fake-voice-detection'...
remote: Enumerating objects: 357, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 357 (delta 36), reused 68 (delta 21), pack-reused 265 (from 1)
Receiving objects: 100% (357/357), 6.79 MiB | 14.36 MiB/s, done.
Resolving deltas: 100% (145/145), done.


In [ ]:
%cd /content/indic-fake-voice-detection
!git switch -c feature/long-call-analysis
!git status --short --branch

/content/indic-fake-voice-detection
Switched to a new branch 'feature/long-call-analysis'
## feature/long-call-analysis


In [ ]:
%cd /content/indic-fake-voice-detection

from pathlib import Path

for filename in [
    "voice_guard_core.py",
    "call_analysis.py",
    "tests/test_call_analysis.py",
    "checkpoints/best_model.pt",
]:
    path = Path(filename)
    print(f"{filename}: {'EXISTS' if path.exists() else 'MISSING'}")

/content/indic-fake-voice-detection
voice_guard_core.py: EXISTS
call_analysis.py: MISSING
tests/test_call_analysis.py: MISSING
checkpoints/best_model.pt: EXISTS


In [ ]:
from pathlib import Path

path = Path("voice_guard_core.py")
source = path.read_text()

In [ ]:
%cd /content/indic-fake-voice-detection

!grep -n -A8 "class CallInputPolicy" voice_guard_core.py

/content/indic-fake-voice-detection


In [ ]:
%cd /content/indic-fake-voice-detection
!head -80 voice_guard_core.py

/content/indic-fake-voice-detection
"""Dependency-light safety and decision helpers for Voice Guard.

This module deliberately does not import torch or Streamlit.  Keeping the input
policy and result policy here makes them cheap to unit-test and lets the web UI
validate an upload before the multi-gigabyte frontend is loaded.
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any, Dict

import numpy as np


SUPPORTED_LANGUAGES = (
    "Assamese",
    "Bengali",
    "Bodo",
    "Dogri",
    "English",
    "Gujarati",
    "Hindi",
    "Kannada",
    "Malayalam",
    "Manipuri",
    "Marathi",
    "Nepali",
    "Odia",
    "Sanskrit",
    "Tamil",
    "Telugu",
)


class AudioValidationError(ValueError):
    """A safe, user-displayable rejection of unsuitable input audio."""


@dataclass(frozen=True)
class AudioInputPolicy:
    """Limits that protect both result quality and quadratic GAT inference."""

    max_bytes: int = 25 * 1024 * 

In [ ]:
%cd /content/indic-fake-voice-detection

from pathlib import Path

path = Path("voice_guard_core.py")
source = path.read_text()

old = '''    min_active_seconds: float = 0.45


@dataclass(frozen=True)
class DecisionPolicy:
'''

new = '''    min_active_seconds: float = 0.45


@dataclass(frozen=True)
class CallInputPolicy:
    """Input limits for long phone-call recordings."""

    max_bytes: int = 200 * 1024 * 1024
    min_duration_seconds: float = 1.0
    max_duration_seconds: float = 900.0


@dataclass(frozen=True)
class DecisionPolicy:
'''

if source.count(old) != 1:
    raise RuntimeError("Insertion point mismatch. No changes made.")

path.write_text(source.replace(old, new, 1))
print("CallInputPolicy added successfully.")

/content/indic-fake-voice-detection
CallInputPolicy added successfully.


In [ ]:
from pathlib import Path

path = Path("call_analysis.py")

if path.exists():
    raise FileExistsError("call_analysis.py already exists. No changes made.")

source = '''"""Long-call speech detection and window segmentation for Voice Guard."""

from __future__ import annotations

from dataclasses import dataclass
from typing import List

import numpy as np


@dataclass(frozen=True)
class SpeechRegion:
    """Speech-active interval in the original recording timeline."""

    start_seconds: float
    end_seconds: float


@dataclass(frozen=True)
class AnalysisWindow:
    """A model-analysis window with its original recording timestamps."""

    start_seconds: float
    end_seconds: float
    start_sample: int
    end_sample: int


def detect_speech_regions(
    waveform: np.ndarray,
    sample_rate: int,
    frame_duration: float = 0.025,
    hop_duration: float = 0.010,
    merge_gap_seconds: float = 0.30,
    min_region_seconds: float = 0.15,
    energy_threshold: float | None = None,
) -> List[SpeechRegion]:
    """Find speech-active regions using a lightweight adaptive energy gate.

    This is an energy-based detector, not a trained voice activity detector.
    It preserves timestamps relative to the original recording.
    """
    samples = np.asarray(waveform, dtype=np.float32).reshape(-1)

    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive.")
    if samples.size == 0:
        return []
    if not np.isfinite(samples).all():
        raise ValueError("waveform contains non-finite values.")
    if frame_duration <= 0 or hop_duration <= 0:
        raise ValueError("Frame and hop durations must be positive.")

    frame_size = max(1, int(round(sample_rate * frame_duration)))
    hop_size = max(1, int(round(sample_rate * hop_duration)))

    if samples.size < frame_size:
        starts = [0]
    else:
        starts = range(0, samples.size - frame_size + 1, hop_size)

    frame_rms = np.asarray(
        [
            np.sqrt(np.mean(
                np.square(samples[start : start + frame_size], dtype=np.float64)
            ))
            for start in starts
        ],
        dtype=np.float64,
    )

    if energy_threshold is None:
        noise_floor = float(np.percentile(frame_rms, 20))
        peak_frame = float(np.max(frame_rms))
        threshold = max(1e-4, min(noise_floor * 2.0, peak_frame * 0.10))
    else:
        if energy_threshold < 0:
            raise ValueError("energy_threshold cannot be negative.")
        threshold = float(energy_threshold)

    active = frame_rms >= threshold
    active_indices = np.flatnonzero(active)
    if active_indices.size == 0:
        return []

    merge_frames = int(round(merge_gap_seconds / hop_duration))
    groups = []
    group_start = previous = int(active_indices[0])

    for index in active_indices[1:]:
        index = int(index)
        if index - previous - 1 > merge_frames:
            groups.append((group_start, previous))
            group_start = index
        previous = index
    groups.append((group_start, previous))

    regions = []
    for first_frame, last_frame in groups:
        start_sample = first_frame * hop_size
        end_sample = min(samples.size, last_frame * hop_size + frame_size)
        start_seconds = start_sample / float(sample_rate)
        end_seconds = end_sample / float(sample_rate)

        if end_seconds - start_seconds >= min_region_seconds:
            regions.append(
                SpeechRegion(
                    start_seconds=start_seconds,
                    end_seconds=end_seconds,
                )
            )

    return regions


def build_analysis_windows(
    regions: List[SpeechRegion],
    sample_rate: int,
    window_seconds: float = 6.0,
    hop_seconds: float = 3.0,
    min_trailing_seconds: float = 2.0,
) -> List[AnalysisWindow]:
    """Create overlapping analysis windows within speech regions."""
    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive.")
    if window_seconds <= 0 or hop_seconds <= 0 or min_trailing_seconds < 0:
        raise ValueError("Window, hop, and trailing durations must be valid.")

    windows = []
    for region in regions:
        if region.start_seconds < 0 or region.end_seconds < region.start_seconds:
            raise ValueError("Invalid speech region.")

        start = region.start_seconds
        while start < region.end_seconds:
            duration = min(window_seconds, region.end_seconds - start)
            if duration < min_trailing_seconds:
                break

            start_sample = int(round(start * sample_rate))
            end_sample = int(round((start + duration) * sample_rate))

            windows.append(
                AnalysisWindow(
                    start_seconds=start,
                    end_seconds=start + duration,
                    start_sample=start_sample,
                    end_sample=end_sample,
                )
            )
            start += hop_seconds

    return windows
'''

path.write_text(source)
print("Created call_analysis.py with speech-region detection and window segmentation.")

Created call_analysis.py with speech-region detection and window segmentation.


In [ ]:
from pathlib import Path

path = Path("call_analysis.py")
with path.open("a", encoding="utf-8") as file:
    file.write("\n\n" + '''@dataclass(frozen=True)
class WindowScore:
    """Fake-voice score for one analysis window."""

    start_seconds: float
    end_seconds: float
    raw_logit: float
    p_fake: float
    verdict: str


@dataclass(frozen=True)
class CallAnalysisReport:
    """Complete analysis results for one phone-call recording."""

    duration_seconds: float
    speech_duration_seconds: float
    windows_analysed: int
    flagged_windows: int
    flagged_share: float
    highest_p_fake: float
    highest_score_timestamp: float
    window_scores: List[WindowScore]
    flagged_ranges: List[SpeechRegion]
''')

print("Added WindowScore and CallAnalysisReport.")

Added WindowScore and CallAnalysisReport.


In [ ]:
from pathlib import Path

path = Path("call_analysis.py")

with path.open("a", encoding="utf-8") as file:
    file.write("\n\n" + '''def merge_flagged_ranges(
    window_scores: List[WindowScore],
) -> List[SpeechRegion]:
    """Merge overlapping or adjacent windows classified as fake."""
    flagged = sorted(
        (score for score in window_scores if score.verdict == "fake"),
        key=lambda score: score.start_seconds,
    )

    if not flagged:
        return []

    merged = []
    current_start = flagged[0].start_seconds
    current_end = flagged[0].end_seconds

    for score in flagged[1:]:
        if score.start_seconds <= current_end:
            current_end = max(current_end, score.end_seconds)
        else:
            merged.append(
                SpeechRegion(current_start, current_end)
            )
            current_start = score.start_seconds
            current_end = score.end_seconds

    merged.append(SpeechRegion(current_start, current_end))
    return merged
''')

print("Added merge_flagged_ranges().")

Added merge_flagged_ranges().


In [ ]:
from pathlib import Path

path = Path("call_analysis.py")

with path.open("a", encoding="utf-8") as file:
    file.write("\n\n" + '''def analyze_call(
    audio_bytes: bytes,
    checkpoint_dir: str | None = None,
    window_seconds: float = 6.0,
    hop_seconds: float = 3.0,
    progress_callback=None,
) -> CallAnalysisReport:
    """Analyse a long phone call using the existing Voice Guard model."""
    import model_pipeline

    if not audio_bytes:
        raise ValueError("No audio data was supplied.")

    waveform, original_sample_rate = model_pipeline.decode_audio_bytes(
        audio_bytes
    )

    waveform = np.asarray(waveform, dtype=np.float32).reshape(-1)

    if waveform.size == 0:
        raise ValueError("The decoded recording is empty.")
    if original_sample_rate <= 0:
        raise ValueError("Invalid audio sample rate.")
    if not np.isfinite(waveform).all():
        raise ValueError("The recording contains non-finite audio values.")

    duration_seconds = waveform.size / float(original_sample_rate)

    if duration_seconds > 900.0:
        raise ValueError("Call recordings cannot exceed 15 minutes.")
    if duration_seconds < 1.0:
        raise ValueError("The recording must be at least 1 second long.")

    waveform = model_pipeline.ensure_target_sample_rate(
        waveform,
        original_sample_rate,
        model_pipeline.TARGET_SAMPLE_RATE,
    )
    sample_rate = model_pipeline.TARGET_SAMPLE_RATE

    regions = detect_speech_regions(waveform, sample_rate)
    windows = build_analysis_windows(
        regions,
        sample_rate,
        window_seconds=window_seconds,
        hop_seconds=hop_seconds,
    )

    if not windows:
        raise ValueError(
            "No speech-active regions long enough for analysis were detected."
        )

    if checkpoint_dir is None:
        bundle = model_pipeline.load_inference_bundle()
    else:
        bundle = model_pipeline.load_inference_bundle(
            checkpoint_dir=checkpoint_dir
        )

    window_scores = []

    for index, window in enumerate(windows):
        segment = waveform[window.start_sample:window.end_sample]

        raw_logit, p_fake = model_pipeline.score_waveform(
            segment,
            sample_rate,
            bundle=bundle,
        )
        verdict = model_pipeline.verdict_from_p_fake(p_fake)

        window_scores.append(
            WindowScore(
                start_seconds=window.start_seconds,
                end_seconds=window.end_seconds,
                raw_logit=raw_logit,
                p_fake=p_fake,
                verdict=verdict,
            )
        )

        if progress_callback is not None:
            progress_callback((index + 1) / len(windows))

    flagged_windows = sum(
        score.verdict == "fake" for score in window_scores
    )

    flagged_share = flagged_windows / len(window_scores)

    highest_score = max(window_scores, key=lambda score: score.p_fake)

    speech_duration_seconds = sum(
        region.end_seconds - region.start_seconds
        for region in regions
    )

    return CallAnalysisReport(
        duration_seconds=duration_seconds,
        speech_duration_seconds=speech_duration_seconds,
        windows_analysed=len(window_scores),
        flagged_windows=flagged_windows,
        flagged_share=flagged_share,
        highest_p_fake=highest_score.p_fake,
        highest_score_timestamp=highest_score.start_seconds,
        window_scores=window_scores,
        flagged_ranges=merge_flagged_ranges(window_scores),
    )
''')

print("Added analyze_call().")

Added analyze_call().


In [ ]:
from pathlib import Path

path = Path("call_analysis.py")
content = path.read_text()

old = '''    if not audio_bytes:
        raise ValueError("No audio data was supplied.")

    waveform, original_sample_rate = model_pipeline.decode_audio_bytes(
'''

new = '''    from voice_guard_core import (
        CallInputPolicy,
        validate_encoded_audio_size,
    )

    validate_encoded_audio_size(
        audio_bytes,
        policy=CallInputPolicy(),
    )

    waveform, original_sample_rate = model_pipeline.decode_audio_bytes(
'''

if old not in content:
    raise RuntimeError("Expected analyze_call section was not found. No changes made.")

path.write_text(content.replace(old, new, 1))
print("Added the 200 MB call-upload validation before decoding.")

Added the 200 MB call-upload validation before decoding.


In [ ]:
%%writefile /content/indic-fake-voice-detection/tests/test_call_analysis.py
import numpy as np
import pytest

from call_analysis import (
    SpeechRegion,
    analyze_call,
    build_analysis_windows,
    detect_speech_regions,
)
from voice_guard_core import (
    AudioInputPolicy,
    CallInputPolicy,
    validate_encoded_audio_size,
)


def make_tone(duration, sample_rate=16000, amplitude=0.08):
    time = np.arange(int(duration * sample_rate), dtype=np.float32) / sample_rate
    return amplitude * np.sin(2 * np.pi * 220 * time)


def make_two_section_recording():
    first = make_tone(6.0)
    silence = np.zeros(int(0.8 * 16000), dtype=np.float32)
    second = make_tone(6.0)
    return np.concatenate([first, silence, second])


def patch_call_pipeline(monkeypatch, waveform, probabilities):
    import model_pipeline

    monkeypatch.setattr(
        model_pipeline, "decode_audio_bytes",
        lambda audio_bytes: (waveform, 16000),
    )
    monkeypatch.setattr(
        model_pipeline, "ensure_target_sample_rate",
        lambda samples, original_rate, target_rate: samples,
    )
    monkeypatch.setattr(
        model_pipeline, "load_inference_bundle",
        lambda **kwargs: object(),
    )

    scores = iter(probabilities)
    monkeypatch.setattr(
        model_pipeline, "score_waveform",
        lambda segment, sample_rate, bundle=None: (0.0, next(scores)),
    )


def test_silence_produces_no_speech_regions():
    silence = np.zeros(5 * 16000, dtype=np.float32)
    assert detect_speech_regions(silence, 16000) == []


def test_quiet_speech_is_detected():
    quiet_speech = make_tone(2.0, amplitude=0.003)
    regions = detect_speech_regions(quiet_speech, 16000)
    assert len(regions) == 1
    assert regions[0].end_seconds - regions[0].start_seconds >= 1.8


def test_silence_gap_creates_separate_speech_regions():
    waveform = np.concatenate([
        make_tone(2.0),
        np.zeros(int(0.8 * 16000), dtype=np.float32),
        make_tone(2.0),
    ])
    regions = detect_speech_regions(waveform, 16000)
    assert len(regions) == 2
    assert regions[0].end_seconds < regions[1].start_seconds
    assert 0.5 <= regions[1].start_seconds - regions[0].end_seconds <= 1.1


def test_windows_use_six_seconds_and_three_second_hop():
    windows = build_analysis_windows([SpeechRegion(0.0, 13.0)], 16000)
    assert [window.start_seconds for window in windows] == [0.0, 3.0, 6.0, 9.0]
    assert windows[0].end_seconds == 6.0
    assert windows[1].end_seconds == 9.0


def test_trailing_window_shorter_than_two_seconds_is_dropped():
    windows = build_analysis_windows([SpeechRegion(0.0, 7.0)], 16000)
    assert all(window.end_seconds - window.start_seconds >= 2.0 for window in windows)
    assert len(windows) == 2
    assert windows[-1].start_seconds == 3.0
    assert windows[-1].end_seconds == 7.0


def test_call_and_single_clip_upload_policies_remain_separate():
    assert CallInputPolicy().max_bytes == 200 * 1024 * 1024
    assert CallInputPolicy().max_duration_seconds == 900.0
    assert AudioInputPolicy().max_bytes == 25 * 1024 * 1024
    assert AudioInputPolicy().max_duration_seconds == 30.0


def test_call_upload_size_limit_is_enforced():
    from dataclasses import replace
    from voice_guard_core import AudioValidationError

    small_policy = replace(CallInputPolicy(), max_bytes=10)
    with pytest.raises(AudioValidationError, match="too large"):
        validate_encoded_audio_size(b"x" * 11, policy=small_policy)


def test_recording_over_fifteen_minutes_is_rejected(monkeypatch):
    import model_pipeline

    monkeypatch.setattr(
        model_pipeline, "decode_audio_bytes",
        lambda audio_bytes: (np.zeros(901, dtype=np.float32), 1),
    )

    def unexpected_resampling(*args, **kwargs):
        raise AssertionError("Resampling should not run for an overlong call.")

    monkeypatch.setattr(
        model_pipeline, "ensure_target_sample_rate",
        unexpected_resampling,
    )

    with pytest.raises(ValueError, match="15 minutes"):
        analyze_call(b"test-audio")


def test_mixed_call_reports_flagged_range_and_progress(monkeypatch):
    waveform = make_two_section_recording()
    patch_call_pipeline(monkeypatch, waveform, [0.1, 0.8])

    progress = []
    report = analyze_call(b"test-audio", progress_callback=progress.append)

    assert report.windows_analysed == 2
    assert report.flagged_windows == 1
    assert report.flagged_share == pytest.approx(0.5)
    assert len(report.flagged_ranges) == 1
    assert report.flagged_ranges[0].start_seconds > 6.5
    assert report.flagged_ranges[0].end_seconds > report.flagged_ranges[0].start_seconds
    assert report.highest_p_fake == pytest.approx(0.8)
    assert report.highest_score_timestamp == report.window_scores[1].start_seconds
    assert progress == pytest.approx([0.5, 1.0])


def test_all_real_call_has_no_flagged_ranges(monkeypatch):
    waveform = make_two_section_recording()
    patch_call_pipeline(monkeypatch, waveform, [0.1, 0.2])

    report = analyze_call(b"test-audio")

    assert report.windows_analysed == 2
    assert report.flagged_windows == 0
    assert report.flagged_share == 0.0
    assert report.flagged_ranges == []

Writing /content/indic-fake-voice-detection/tests/test_call_analysis.py


In [ ]:
%cd /content/indic-fake-voice-detection

from pathlib import Path

path = Path("tests/test_call_analysis.py")
content = path.read_text()

content = content.replace(
    "patch_call_pipeline(monkeypatch, waveform, [0.1, 0.8])",
    "patch_call_pipeline(monkeypatch, waveform, [0.1, 0.1, 0.8, 0.8])",
)
content = content.replace(
    "patch_call_pipeline(monkeypatch, waveform, [0.1, 0.2])",
    "patch_call_pipeline(monkeypatch, waveform, [0.1, 0.1, 0.2, 0.2])",
)

path.write_text(content)
print("Updated both test cases with four scores.")

/content/indic-fake-voice-detection
Updated both test cases with four scores.


In [ ]:
%cd /content/indic-fake-voice-detection

from pathlib import Path

path = Path("tests/test_call_analysis.py")
content = path.read_text()

content = content.replace(
    "assert report.windows_analysed == 2",
    "assert report.windows_analysed == 4",
)
content = content.replace(
    "assert report.flagged_windows == 1",
    "assert report.flagged_windows == 2",
)
content = content.replace(
    "assert report.flagged_share == pytest.approx(0.5)",
    "assert report.flagged_share == pytest.approx(0.5)",
)
content = content.replace(
    "assert progress == pytest.approx([0.5, 1.0])",
    "assert progress == pytest.approx([0.25, 0.5, 0.75, 1.0])",
)

path.write_text(content)
print("Updated window counts and progress assertions.")

/content/indic-fake-voice-detection
Updated window counts and progress assertions.


In [ ]:
%cd /content/indic-fake-voice-detection

from pathlib import Path

path = Path("tests/test_call_analysis.py")
content = path.read_text().replace(
    "assert report.highest_score_timestamp == report.window_scores[1].start_seconds",
    "assert report.highest_score_timestamp == report.window_scores[2].start_seconds",
)
path.write_text(content)

print("Timestamp assertion corrected.")

/content/indic-fake-voice-detection
Timestamp assertion corrected.


In [ ]:
!{sys.executable} -m pip install pypdf

In [ ]:
!{sys.executable} -m pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.4 MB/s  0:00:00


In [ ]:
import sys
!{sys.executable} -m py_compile call_analysis.py voice_guard_core.py
!{sys.executable} -m pytest -q

.................................                                        [100%]
33 passed in 2.10s


In [ ]:
!git status --short

 M voice_guard_core.py
?? call_analysis.py
?? tests/test_call_analysis.py


In [ ]:
!git add voice_guard_core.py call_analysis.py tests/test_call_analysis.py

In [ ]:
!git status

On branch feature/long-call-analysis
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   call_analysis.py
	new file:   tests/test_call_analysis.py
	modified:   voice_guard_core.py



In [ ]:
!git commit -m "Add long-call analysis core"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@dc40cbeefbf3.(none)')


In [ ]:
!git config user.name "Ashitha"
!git config user.email "rajashitha310@gmail.com"

In [ ]:
!git commit -m "Add long-call analysis core"

[feature/long-call-analysis 6af6647] Add long-call analysis core
 3 files changed, 490 insertions(+)
 create mode 100644 call_analysis.py
 create mode 100644 tests/test_call_analysis.py


In [ ]:
!git push -u origin feature/long-call-analysis

fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
import subprocess
from getpass import getpass

token = getpass("Enter your GitHub Personal Access Token: ")

subprocess.run(
    ["git", "config", "credential.helper", "cache"],
    check=True
)

credential = (
    "protocol=https\n"
    "host=github.com\n"
    "username=Ashitha098\n"
    f"password={token}\n\n"
)

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

del token
print("GitHub credentials configured for this Colab session.")

Enter your GitHub Personal Access Token: ··········
GitHub credentials configured for this Colab session.


In [ ]:
!git push -u origin feature/long-call-analysis

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 4.55 KiB | 1.14 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
remote: 
remote: Create a pull request for 'feature/long-call-analysis' on GitHub by visiting:
remote:      https://github.com/Shruti-sivaramakrishnan/indic-fake-voice-detection/pull/new/feature/long-call-analysis
remote: 
To https://github.com/Shruti-sivaramakrishnan/indic-fake-voice-detection.git
 * [new branch]      feature/long-call-analysis -> feature/long-call-analysis
branch 'feature/long-call-analysis' set up to track 'origin/feature/long-call-analysis'.


In [ ]:
from pathlib import Path

for file in [
    "app.py",
    "i18n.py",
    ".streamlit/config.toml",
    "README.md"
]:
    path = Path(file)
    print(f"{file}: {'FOUND' if path.exists() else 'NOT FOUND'}")

app.py: FOUND
i18n.py: FOUND
.streamlit/config.toml: FOUND
README.md: FOUND


In [ ]:
from pathlib import Path

print(Path("app.py").read_text())

"""Voice Guard Streamlit application.

Every analysis is bound to the SHA-256 hash of the selected upload. Replacing
the file clears the previous result and report before the page is rendered.
"""

from __future__ import annotations

import hashlib
import html
import logging
from pathlib import Path

try:
    import truststore

    truststore.inject_into_ssl()
except ImportError:
    pass

import streamlit as st

import codec_robustness
import evidence_report
import i18n
import ui_kit
from i18n import t


LOGGER = logging.getLogger("voice_guard.app")
MODEL_IMPORT_ERROR: Exception | None = None

try:
    import model_pipeline as mp
except Exception as exc:  # noqa: BLE001 - replaced with a safe public message
    mp = None
    MODEL_IMPORT_ERROR = exc
    LOGGER.exception("Model pipeline import failed")


st.set_page_config(
    page_title="Voice Guard · AI voice analysis",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="collapsed",
)
st.markdown(f"<style>{ui_kit.BASE_

In [ ]:
from pathlib import Path
import re

text = Path("app.py").read_text()
lines = text.splitlines()

print("Total lines:", len(lines))
print("\nFUNCTIONS, CLASSES AND SECTION HEADINGS:\n")

for number, line in enumerate(lines, start=1):
    if (
        re.match(r"\s*(def |class )", line)
        or re.match(r"\s*#\s*[A-Z]", line)
        or "st.title(" in line
        or "st.sidebar" in line
    ):
        print(f"{number}: {line.strip()}")

Total lines: 513

FUNCTIONS, CLASSES AND SECTION HEADINGS:

66: def current_lang() -> str:
70: def format_bytes(size: int) -> str:
78: def clear_analysis_result() -> None:
88: def sync_selected_upload(audio_bytes: bytes | None) -> str | None:
96: def run_analysis(audio_bytes: bytes):
102: def run_robustness_analysis(audio_bytes: bytes, original_result):
112: def safe_error_message(exc: Exception, lang: str) -> str:
123: def render_header(lang: str) -> None:
154: def render_hero(lang: str) -> None:
176: def render_process_card(lang: str) -> None:
204: def render_result(result, lang: str) -> None:
294: def render_robustness_report(report, lang: str) -> None:
356: def render_report_download(result, filename: str, lang: str) -> None:
383: def render_app() -> None:


In [ ]:
lines = Path("app.py").read_text().splitlines()

for i in range(382, len(lines)):
    print(f"{i+1}: {lines[i]}")

383: def render_app() -> None:
384:     lang = current_lang()
385:     render_header(lang)
386:     render_hero(lang)
387: 
388:     upload_col, process_col = st.columns([1.55, 0.85], gap="large")
389:     with upload_col:
390:         upload_panel = st.container(border=True)
391:         with upload_panel:
392:             st.markdown(
393:                 f"""
394:                 <div class="vg-section-kicker">{html.escape(t('workspace_kicker', lang))}</div>
395:                 <h2 class="vg-section-title">{html.escape(t('how_it_works_title', lang))}</h2>
396:                 <p class="vg-section-copy">{html.escape(t('how_it_works_body', lang))}</p>
397:                 """,
398:                 unsafe_allow_html=True,
399:             )
400:             uploaded = st.file_uploader(
401:                 t("upload_label", lang),
402:                 type=ACCEPTED_TYPES,
403:                 help=t("upload_help", lang),
404:                 label_visibility="collapsed",
405:         

In [ ]:
from pathlib import Path

text = Path("i18n.py").read_text()
print("Total lines:", len(text.splitlines()))

for i, line in enumerate(text.splitlines(), start=1):
    if line.strip().startswith(("def ", "LANGUAGES", "TRANSLATIONS", "MESSAGES")):
        print(f"{i}: {line.strip()}")

Total lines: 608
18: LANGUAGES = ["English", "Hindi", "Tamil", "Kannada"]
20: TRANSLATIONS: dict[str, dict[str, str]] = {
232: TRANSLATIONS.update({
489: TRANSLATIONS["pdf_heading"] = {
493: TRANSLATIONS["pdf_button"] = {
497: TRANSLATIONS["pdf_body"] = {
506: TRANSLATIONS.update({
602: def t(key: str, lang: str) -> str:


In [ ]:
lines = Path("i18n.py").read_text().splitlines()

print("TRANSLATION STRUCTURE:")
for i in range(17, 27):
    print(f"{i+1}: {lines[i]}")

print("\nTRANSLATION FUNCTION:")
for i in range(600, 608):
    print(f"{i+1}: {lines[i]}")

TRANSLATION STRUCTURE:
18: LANGUAGES = ["English", "Hindi", "Tamil", "Kannada"]
19: 
20: TRANSLATIONS: dict[str, dict[str, str]] = {
21:     "app_title": {
22:         "English": "Voice Guard", "Hindi": "Voice Guard",
23:         "Tamil": "Voice Guard", "Kannada": "Voice Guard",
24:     },
25:     "app_tagline": {
26:         "English": "AI-generated voice detection for the calls that try to catch you off guard.",
27:         "Hindi": "उन कॉल्स के लिए एआई-जनित आवाज़ की पहचान, जो आपको चौंकाने की कोशिश करती हैं।",

TRANSLATION FUNCTION:
601: 
602: def t(key: str, lang: str) -> str:
603:     """Look up a translated UI string. Falls back to English, then the raw key,
604:     so a missing translation degrades to readable text instead of a crash."""
605:     entry = TRANSLATIONS.get(key)
606:     if entry is None:
607:         return key
608:     return entry.get(lang) or entry.get("English") or key


In [ ]:
lines = Path("app.py").read_text().splitlines()

for i in range(0, 122):
    line = lines[i]
    if line.strip():
        print(f"{i+1}: {line}")

1: """Voice Guard Streamlit application.
3: Every analysis is bound to the SHA-256 hash of the selected upload. Replacing
4: the file clears the previous result and report before the page is rendered.
5: """
7: from __future__ import annotations
9: import hashlib
10: import html
11: import logging
12: from pathlib import Path
14: try:
15:     import truststore
17:     truststore.inject_into_ssl()
18: except ImportError:
19:     pass
21: import streamlit as st
23: import codec_robustness
24: import evidence_report
25: import i18n
26: import ui_kit
27: from i18n import t
30: LOGGER = logging.getLogger("voice_guard.app")
31: MODEL_IMPORT_ERROR: Exception | None = None
33: try:
34:     import model_pipeline as mp
35: except Exception as exc:  # noqa: BLE001 - replaced with a safe public message
36:     mp = None
37:     MODEL_IMPORT_ERROR = exc
38:     LOGGER.exception("Model pipeline import failed")
41: st.set_page_config(
42:     page_title="Voice Guard · AI voice analysis",
43:     page

In [ ]:
print(Path(".streamlit/config.toml").read_text())

# Forces Streamlit's OWN theme (native widget backgrounds/text -- file
# uploader, expander, selectbox, json viewer, etc.) to a fixed light theme
# regardless of the visitor's OS/browser dark-mode preference.
#
# This is the root-cause fix for a real bug: without this, Streamlit paints
# several of its own structural containers with ITS dark-theme background
# (near-black) when a visitor's browser prefers dark, while our own CSS
# assumed a light page -- producing genuinely unreadable text in specific
# native widgets under system dark mode. Confirmed via a real rendered-DOM
# audit (Playwright + Chromium), not assumed.
[theme]
base = "light"
primaryColor = "#665CF6"
backgroundColor = "#F6F7FB"
secondaryBackgroundColor = "#FFFFFF"
textColor = "#111827"
font = "sans-serif"

[server]
maxUploadSize = 25
maxMessageSize = 25



In [ ]:
from pathlib import Path

readme = Path("README.md").read_text()
print(readme[:7000])

# Voice Guard - Indic fake voice detection

Voice Guard is a research/demo pipeline for detecting synthetic speech across 16 Indian languages. It uses a frozen `facebook/wav2vec2-xls-r-300m` frontend and a trainable graph-attention classifier head, exposed through a Streamlit upload UI.

The head is **AASIST-inspired, not an implementation of the full AASIST architecture**. The current checkpoint is a historical v1 model and must not be treated as production-validated.

## Current status

- The upload path now rejects empty, silent, too-short, too-long, non-finite, and oversized audio before model inference.
- The UI returns `Likely genuine`, `Inconclusive`, or `Likely synthetic`, and labels its number as a model score rather than proof.
- Results and PDF downloads are bound to the exact SHA-256 hash of the currently selected audio.
- Automatic language identification is disabled. The removed SpeechBrain path did not correctly parse/resample all inputs and did not cover all 16 target l

In [ ]:
!git status --short

In [ ]:
!git branch --show-current

feature/long-call-analysis


In [ ]:
import os
from pathlib import Path

print("Current directory:", os.getcwd())
print("Project files:")
for name in ["app.py", "call_analysis.py", "codec_robustness.py",
             "model_pipeline.py", "voice_guard_core.py"]:
    print(f"{name}: {'FOUND' if Path(name).exists() else 'NOT FOUND'}")

Current directory: /content/indic-fake-voice-detection
Project files:
app.py: FOUND
call_analysis.py: FOUND
codec_robustness.py: FOUND
model_pipeline.py: FOUND
voice_guard_core.py: FOUND


In [ ]:
!python -m pytest -q tests

.................................                                        [100%]
33 passed in 2.51s


In [ ]:
from pathlib import Path

print(Path(".streamlit/config.toml").read_text())

# Forces Streamlit's OWN theme (native widget backgrounds/text -- file
# uploader, expander, selectbox, json viewer, etc.) to a fixed light theme
# regardless of the visitor's OS/browser dark-mode preference.
#
# This is the root-cause fix for a real bug: without this, Streamlit paints
# several of its own structural containers with ITS dark-theme background
# (near-black) when a visitor's browser prefers dark, while our own CSS
# assumed a light page -- producing genuinely unreadable text in specific
# native widgets under system dark mode. Confirmed via a real rendered-DOM
# audit (Playwright + Chromium), not assumed.
[theme]
base = "light"
primaryColor = "#665CF6"
backgroundColor = "#F6F7FB"
secondaryBackgroundColor = "#FFFFFF"
textColor = "#111827"
font = "sans-serif"

[server]
maxUploadSize = 25
maxMessageSize = 25



In [ ]:
from pathlib import Path

config_path = Path(".streamlit/config.toml")
config = config_path.read_text()

old = "maxUploadSize = 25"
new = "maxUploadSize = 200"

assert config.count(old) == 1, "Expected setting not found exactly once."

config_path.write_text(config.replace(old, new, 1))

print(config_path.read_text())

# Forces Streamlit's OWN theme (native widget backgrounds/text -- file
# uploader, expander, selectbox, json viewer, etc.) to a fixed light theme
# regardless of the visitor's OS/browser dark-mode preference.
#
# This is the root-cause fix for a real bug: without this, Streamlit paints
# several of its own structural containers with ITS dark-theme background
# (near-black) when a visitor's browser prefers dark, while our own CSS
# assumed a light page -- producing genuinely unreadable text in specific
# native widgets under system dark mode. Confirmed via a real rendered-DOM
# audit (Playwright + Chromium), not assumed.
[theme]
base = "light"
primaryColor = "#665CF6"
backgroundColor = "#F6F7FB"
secondaryBackgroundColor = "#FFFFFF"
textColor = "#111827"
font = "sans-serif"

[server]
maxUploadSize = 200
maxMessageSize = 25



In [ ]:
from pathlib import Path

readme = Path("README.md").read_text()
print(readme[:5000])

# Voice Guard - Indic fake voice detection

Voice Guard is a research/demo pipeline for detecting synthetic speech across 16 Indian languages. It uses a frozen `facebook/wav2vec2-xls-r-300m` frontend and a trainable graph-attention classifier head, exposed through a Streamlit upload UI.

The head is **AASIST-inspired, not an implementation of the full AASIST architecture**. The current checkpoint is a historical v1 model and must not be treated as production-validated.

## Current status

- The upload path now rejects empty, silent, too-short, too-long, non-finite, and oversized audio before model inference.
- The UI returns `Likely genuine`, `Inconclusive`, or `Likely synthetic`, and labels its number as a model score rather than proof.
- Results and PDF downloads are bound to the exact SHA-256 hash of the currently selected audio.
- Automatic language identification is disabled. The removed SpeechBrain path did not correctly parse/resample all inputs and did not cover all 16 target l

In [ ]:
!git status --short
!git diff -- .streamlit/config.toml

 M .streamlit/config.toml
diff --git a/.streamlit/config.toml b/.streamlit/config.toml
index 577579a..8448900 100644
--- a/.streamlit/config.toml
+++ b/.streamlit/config.toml
@@ -17,5 +17,5 @@ textColor = "#111827"
 font = "sans-serif"
 
 [server]
-maxUploadSize = 25
+maxUploadSize = 200
 maxMessageSize = 25


In [ ]:
from pathlib import Path

app = Path("app.py").read_text().splitlines()

for start, end in [(1, 115), (383, 513)]:
    print(f"\n--- app.py lines {start}-{end} ---")
    for i in range(start - 1, min(end, len(app))):
        print(f"{i+1}: {app[i]}")


--- app.py lines 1-115 ---
1: """Voice Guard Streamlit application.
2: 
3: Every analysis is bound to the SHA-256 hash of the selected upload. Replacing
4: the file clears the previous result and report before the page is rendered.
5: """
6: 
7: from __future__ import annotations
8: 
9: import hashlib
10: import html
11: import logging
12: from pathlib import Path
13: 
14: try:
15:     import truststore
16: 
17:     truststore.inject_into_ssl()
18: except ImportError:
19:     pass
20: 
21: import streamlit as st
22: 
23: import codec_robustness
24: import evidence_report
25: import i18n
26: import ui_kit
27: from i18n import t
28: 
29: 
30: LOGGER = logging.getLogger("voice_guard.app")
31: MODEL_IMPORT_ERROR: Exception | None = None
32: 
33: try:
34:     import model_pipeline as mp
35: except Exception as exc:  # noqa: BLE001 - replaced with a safe public message
36:     mp = None
37:     MODEL_IMPORT_ERROR = exc
38:     LOGGER.exception("Model pipeline import failed")
39: 
40: 
41: s

In [ ]:
import inspect
import call_analysis

print(inspect.signature(call_analysis.analyze_call))
print("\n--- Report structure ---")
print(inspect.getsource(call_analysis.CallAnalysisReport))
print("\n--- Window score structure ---")
print(inspect.getsource(call_analysis.WindowScore))

(audio_bytes: 'bytes', checkpoint_dir: 'str | None' = None, window_seconds: 'float' = 6.0, hop_seconds: 'float' = 3.0, progress_callback=None) -> 'CallAnalysisReport'

--- Report structure ---
@dataclass(frozen=True)
class CallAnalysisReport:
    """Complete analysis results for one phone-call recording."""

    duration_seconds: float
    speech_duration_seconds: float
    windows_analysed: int
    flagged_windows: int
    flagged_share: float
    highest_p_fake: float
    highest_score_timestamp: float
    window_scores: List[WindowScore]
    flagged_ranges: List[SpeechRegion]


--- Window score structure ---
@dataclass(frozen=True)
class WindowScore:
    """Fake-voice score for one analysis window."""

    start_seconds: float
    end_seconds: float
    raw_logit: float
    p_fake: float
    verdict: str



In [ ]:
import inspect
from call_analysis import analyze_call

print(inspect.getsource(analyze_call))

def analyze_call(
    audio_bytes: bytes,
    checkpoint_dir: str | None = None,
    window_seconds: float = 6.0,
    hop_seconds: float = 3.0,
    progress_callback=None,
) -> CallAnalysisReport:
    """Analyse a long phone call using the existing Voice Guard model."""
    import model_pipeline

    from voice_guard_core import (
        CallInputPolicy,
        validate_encoded_audio_size,
    )

    validate_encoded_audio_size(
        audio_bytes,
        policy=CallInputPolicy(),
    )

    waveform, original_sample_rate = model_pipeline.decode_audio_bytes(
        audio_bytes
    )

    waveform = np.asarray(waveform, dtype=np.float32).reshape(-1)

    if waveform.size == 0:
        raise ValueError("The decoded recording is empty.")
    if original_sample_rate <= 0:
        raise ValueError("Invalid audio sample rate.")
    if not np.isfinite(waveform).all():
        raise ValueError("The recording contains non-finite audio values.")

    duration_seconds = waveform.size / float

In [ ]:
import inspect
from voice_guard_core import CallInputPolicy

print(inspect.getsource(CallInputPolicy))

@dataclass(frozen=True)
class CallInputPolicy:
    """Input limits for long phone-call recordings."""

    max_bytes: int = 200 * 1024 * 1024
    min_duration_seconds: float = 1.0
    max_duration_seconds: float = 900.0



In [ ]:
from pathlib import Path

source = Path("voice_guard_core.py").read_text()

for line in source.splitlines():
    if "min_duration_seconds" in line or "CallInputPolicy" in line:
        print(repr(line))

'    min_duration_seconds: float = 1.0'
'class CallInputPolicy:'
'    min_duration_seconds: float = 1.0'
'    if duration < policy.min_duration_seconds:'
'            f"The clip is too short. Use at least {policy.min_duration_seconds:.0f} second "'


In [ ]:
from pathlib import Path
import importlib
import voice_guard_core

path = Path("voice_guard_core.py")
source = path.read_text()

class_start = source.index("class CallInputPolicy:")
old = "    min_duration_seconds: float = 1.0"

position = source.index(old, class_start)

source = source[:position] + source[position:].replace(
    old,
    "    min_duration_seconds: float = 30.0",
    1,
)

path.write_text(source)

importlib.reload(voice_guard_core)

print("Single Clip policy:", voice_guard_core.AudioInputPolicy())
print("Call Analysis policy:", voice_guard_core.CallInputPolicy())

Single Clip policy: AudioInputPolicy(max_bytes=26214400, min_duration_seconds=1.0, max_duration_seconds=30.0, min_rms=0.0001, min_peak=0.001, min_active_seconds=0.45)
Call Analysis policy: CallInputPolicy(max_bytes=209715200, min_duration_seconds=30.0, max_duration_seconds=900.0)


In [ ]:
from pathlib import Path

path = Path("call_analysis.py")
source = path.read_text()

for line in source.splitlines():
    if "duration_seconds <" in line or "at least 1 second" in line:
        print(line)

    if duration_seconds < 1.0:
        raise ValueError("The recording must be at least 1 second long.")


In [ ]:
from pathlib import Path

path = Path("call_analysis.py")
source = path.read_text()

old = '''    if duration_seconds < 1.0:
        raise ValueError("The recording must be at least 1 second long.")'''

new = '''    if duration_seconds < 30.0:
        raise ValueError("The recording must be at least 30 seconds long.")'''

assert source.count(old) == 1, "Expected validation block not found exactly once."

path.write_text(source.replace(old, new, 1))

print("Updated validation:")
for line in path.read_text().splitlines():
    if "duration_seconds <" in line or "at least 30 seconds" in line:
        print(line)

Updated validation:
    if duration_seconds < 30.0:
        raise ValueError("The recording must be at least 30 seconds long.")


In [ ]:
!python -m pytest -q tests

........FF.......................                                        [100%]
=================================== FAILURES ===================================
______________ test_mixed_call_reports_flagged_range_and_progress ______________

monkeypatch = <_pytest.monkeypatch.MonkeyPatch object at 0x7fea0fd25e00>

    def test_mixed_call_reports_flagged_range_and_progress(monkeypatch):
        waveform = make_two_section_recording()
        patch_call_pipeline(monkeypatch, waveform, [0.1, 0.1, 0.8, 0.8])
    
        progress = []
>       report = analyze_call(b"test-audio", progress_callback=progress.append)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

tests/test_call_analysis.py:132: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

audio_bytes = b'test-audio', checkpoint_dir = None, window_seconds = 6.0
hop_seconds = 3.0
progress_callback = <built-in method append of list object at 0x7fe9f4fe11c0>

    def analyze

In [ ]:
from pathlib import Path

path = Path("tests/test_call_analysis.py")
lines = path.read_text().splitlines()

for start, end in [(1, 90), (115, 165)]:
    print(f"\n--- Lines {start}-{end} ---")
    for number in range(start, min(end, len(lines)) + 1):
        print(f"{number}: {lines[number - 1]}")


--- Lines 1-90 ---
1: import numpy as np
2: import pytest
3: 
4: from call_analysis import (
5:     SpeechRegion,
6:     analyze_call,
7:     build_analysis_windows,
8:     detect_speech_regions,
9: )
10: from voice_guard_core import (
11:     AudioInputPolicy,
12:     CallInputPolicy,
13:     validate_encoded_audio_size,
14: )
15: 
16: 
17: def make_tone(duration, sample_rate=16000, amplitude=0.08):
18:     time = np.arange(int(duration * sample_rate), dtype=np.float32) / sample_rate
19:     return amplitude * np.sin(2 * np.pi * 220 * time)
20: 
21: 
22: def make_two_section_recording():
23:     first = make_tone(6.0)
24:     silence = np.zeros(int(0.8 * 16000), dtype=np.float32)
25:     second = make_tone(6.0)
26:     return np.concatenate([first, silence, second])
27: 
28: 
29: def patch_call_pipeline(monkeypatch, waveform, probabilities):
30:     import model_pipeline
31: 
32:     monkeypatch.setattr(
33:         model_pipeline, "decode_audio_bytes",
34:         lambda audio_bytes

In [ ]:
from pathlib import Path

path = Path("tests/test_call_analysis.py")
source = path.read_text()

old = "    silence = np.zeros(int(0.8 * 16000), dtype=np.float32)"
new = "    silence = np.zeros(int(18.8 * 16000), dtype=np.float32)"

assert source.count(old) == 1, "Expected fixture line not found exactly once."

path.write_text(source.replace(old, new, 1))

print("Updated test fixture:")
print("Two 6-second sections separated by 18.8 seconds of silence.")
print("Total recording duration: 30.8 seconds.")

Updated test fixture:
Two 6-second sections separated by 18.8 seconds of silence.
Total recording duration: 30.8 seconds.


In [ ]:
!python -m pytest -q tests

.................................                                        [100%]
33 passed in 2.76s


In [ ]:
from pathlib import Path

path = Path("i18n.py")
lines = path.read_text().splitlines()

print("--- Translation dictionary start ---")
for number in range(1, 65):
    print(f"{number}: {lines[number - 1]}")

print("\n--- Translation dictionary end ---")
for number in range(560, min(609, len(lines)) + 1):
    print(f"{number}: {lines[number - 1]}")

--- Translation dictionary start ---
1: """
2: Real interface translations for app.py -- English, Hindi, Tamil, Kannada.
3: 
4: Only interface chrome (labels, buttons, captions, headings) is translated.
5: `detected_language` in an AnalysisResult is a model output and is never
6: touched here -- it is displayed exactly as the pipeline returns it (currently
7: always "unknown", since there is no language classifier), regardless of which
8: interface language is selected. See the LANGUAGE_CAPTION string below, which
9: says so explicitly in the UI itself.
10: 
11: Translations are genuine, not placeholders, but were produced by an AI
12: assistant rather than a native-speaker reviewer -- worth a native-speaker
13: pass before this ships to real users, particularly for tone/register.
14: """
15: 
16: from __future__ import annotations
17: 
18: LANGUAGES = ["English", "Hindi", "Tamil", "Kannada"]
19: 
20: TRANSLATIONS: dict[str, dict[str, str]] = {
21:     "app_title": {
22:         "Engli

In [ ]:
from pathlib import Path

path = Path("app.py")
lines = path.read_text().splitlines()

for start, end in [(1, 65), (370, 430), (430, 515)]:
    print(f"\n--- app.py lines {start}-{end} ---")
    for number in range(start, min(end, len(lines)) + 1):
        print(f"{number}: {lines[number - 1]}")


--- app.py lines 1-65 ---
1: """Voice Guard Streamlit application.
2: 
3: Every analysis is bound to the SHA-256 hash of the selected upload. Replacing
4: the file clears the previous result and report before the page is rendered.
5: """
6: 
7: from __future__ import annotations
8: 
9: import hashlib
10: import html
11: import logging
12: from pathlib import Path
13: 
14: try:
15:     import truststore
16: 
17:     truststore.inject_into_ssl()
18: except ImportError:
19:     pass
20: 
21: import streamlit as st
22: 
23: import codec_robustness
24: import evidence_report
25: import i18n
26: import ui_kit
27: from i18n import t
28: 
29: 
30: LOGGER = logging.getLogger("voice_guard.app")
31: MODEL_IMPORT_ERROR: Exception | None = None
32: 
33: try:
34:     import model_pipeline as mp
35: except Exception as exc:  # noqa: BLE001 - replaced with a safe public message
36:     mp = None
37:     MODEL_IMPORT_ERROR = exc
38:     LOGGER.exception("Model pipeline import failed")
39: 
40: 
41: st

In [ ]:
from pathlib import Path

path = Path("call_analysis.py")
lines = path.read_text().splitlines()

for start, end in [(1, 75), (220, 310)]:
    print(f"\n--- call_analysis.py lines {start}-{end} ---")
    for number in range(start, min(end, len(lines)) + 1):
        print(f"{number}: {lines[number - 1]}")


--- call_analysis.py lines 1-75 ---
1: """Long-call speech detection and window segmentation for Voice Guard."""
2: 
3: from __future__ import annotations
4: 
5: from dataclasses import dataclass
6: from typing import List
7: 
8: import numpy as np
9: 
10: 
11: @dataclass(frozen=True)
12: class SpeechRegion:
13:     """Speech-active interval in the original recording timeline."""
14: 
15:     start_seconds: float
16:     end_seconds: float
17: 
18: 
19: @dataclass(frozen=True)
20: class AnalysisWindow:
21:     """A model-analysis window with its original recording timestamps."""
22: 
23:     start_seconds: float
24:     end_seconds: float
25:     start_sample: int
26:     end_sample: int
27: 
28: 
29: def detect_speech_regions(
30:     waveform: np.ndarray,
31:     sample_rate: int,
32:     frame_duration: float = 0.025,
33:     hop_duration: float = 0.010,
34:     merge_gap_seconds: float = 0.30,
35:     min_region_seconds: float = 0.15,
36:     energy_threshold: float | None = None,

In [ ]:
from pathlib import Path

lines = Path("call_analysis.py").read_text().splitlines()

for start, end in [(80, 145), (304, len(lines))]:
    print(f"\n--- call_analysis.py lines {start}-{end} ---")
    for number in range(start, min(end, len(lines)) + 1):
        print(f"{number}: {lines[number - 1]}")


--- call_analysis.py lines 80-145 ---
80: 
81:     active = frame_rms >= threshold
82:     active_indices = np.flatnonzero(active)
83:     if active_indices.size == 0:
84:         return []
85: 
86:     merge_frames = int(round(merge_gap_seconds / hop_duration))
87:     groups = []
88:     group_start = previous = int(active_indices[0])
89: 
90:     for index in active_indices[1:]:
91:         index = int(index)
92:         if index - previous - 1 > merge_frames:
93:             groups.append((group_start, previous))
94:             group_start = index
95:         previous = index
96:     groups.append((group_start, previous))
97: 
98:     regions = []
99:     for first_frame, last_frame in groups:
100:         start_sample = first_frame * hop_size
101:         end_sample = min(samples.size, last_frame * hop_size + frame_size)
102:         start_seconds = start_sample / float(sample_rate)
103:         end_seconds = end_sample / float(sample_rate)
104: 
105:         if end_seconds - st

In [ ]:
from pathlib import Path

path = Path("app.py")
source = path.read_text()

old_import = "import codec_robustness\n"
new_import = "import call_analysis\nimport codec_robustness\n"

assert source.count(old_import) == 1
source = source.replace(old_import, new_import, 1)

old_state = '''    "robustness_error": None,
}.items():'''

new_state = '''    "robustness_error": None,
    "call_selected_hash": None,
    "call_result": None,
    "call_result_hash": None,
    "call_result_error": None,
    "call_audio_filename": None,
}.items():'''

assert source.count(old_state) == 1
source = source.replace(old_state, new_state, 1)

path.write_text(source)

print("Call Analysis import and separate session-state keys added.")

Call Analysis import and separate session-state keys added.


In [ ]:
from pathlib import Path

path = Path("i18n.py")
source = path.read_text()

marker = "\n\n\ndef t(key: str, lang: str) -> str:"

translations = '''

TRANSLATIONS.update({
    "mode_label": {
        "English": "Analysis mode", "Hindi": "विश्लेषण मोड",
        "Tamil": "பகுப்பாய்வு முறை", "Kannada": "ವಿಶ್ಲೇಷಣೆ ವಿಧಾನ",
    },
    "single_clip_mode": {
        "English": "Single Clip", "Hindi": "एक क्लिप",
        "Tamil": "ஒற்றைக் கிளிப்", "Kannada": "ಒಂದು ಕ್ಲಿಪ್",
    },
    "call_analysis_mode": {
        "English": "Call Analysis", "Hindi": "कॉल विश्लेषण",
        "Tamil": "அழைப்பு பகுப்பாய்வு", "Kannada": "ಕರೆ ವಿಶ್ಲೇಷಣೆ",
    },
    "call_upload_label": {
        "English": "Upload a call recording", "Hindi": "कॉल रिकॉर्डिंग अपलोड करें",
        "Tamil": "அழைப்புப் பதிவைப் பதிவேற்றவும்", "Kannada": "ಕರೆ ರೆಕಾರ್ಡಿಂಗ್ ಅಪ್‌ಲೋಡ್ ಮಾಡಿ",
    },
    "call_upload_help": {
        "English": "WAV, MP3, FLAC, OGG or M4A. 30 seconds to 15 minutes. Maximum 200 MB.",
        "Hindi": "WAV, MP3, FLAC, OGG या M4A। 30 सेकंड से 15 मिनट। अधिकतम 200 MB।",
        "Tamil": "WAV, MP3, FLAC, OGG அல்லது M4A. 30 வினாடிகள் முதல் 15 நிமிடங்கள் வரை. அதிகபட்சம் 200 MB.",
        "Kannada": "WAV, MP3, FLAC, OGG ಅಥವಾ M4A. 30 ಸೆಕೆಂಡುಗಳಿಂದ 15 ನಿಮಿಷಗಳವರೆಗೆ. ಗರಿಷ್ಠ 200 MB.",
    },
    "call_analyze_button": {
        "English": "Analyse call", "Hindi": "कॉल का विश्लेषण करें",
        "Tamil": "அழைப்பைப் பகுப்பாய்வு செய்", "Kannada": "ಕರೆ ವಿಶ್ಲೇಷಿಸಿ",
    },
    "call_analyzing": {
        "English": "Analysing call recording...", "Hindi": "कॉल रिकॉर्डिंग का विश्लेषण हो रहा है...",
        "Tamil": "அழைப்புப் பதிவு பகுப்பாய்வு செய்யப்படுகிறது...", "Kannada": "ಕರೆ ರೆಕಾರ್ಡಿಂಗ್ ವಿಶ್ಲೇಷಿಸಲಾಗುತ್ತಿದೆ...",
    },
    "call_progress": {
        "English": "Analysing window {current} of {total}",
        "Hindi": "{total} में से {current} विंडो का विश्लेषण हो रहा है",
        "Tamil": "{total} சாளரங்களில் {current}வது பகுப்பாய்வு செய்யப்படுகிறது",
        "Kannada": "{total} ವಿಂಡೋಗಳಲ್ಲಿ {current}ನೇ ವಿಂಡೋ ವಿಶ್ಲೇಷಿಸಲಾಗುತ್ತಿದೆ",
    },
    "call_summary": {
        "English": "Call analysis summary", "Hindi": "कॉल विश्लेषण सारांश",
        "Tamil": "அழைப்புப் பகுப்பாய்வு சுருக்கம்", "Kannada": "ಕರೆ ವಿಶ್ಲೇಷಣೆ ಸಾರಾಂಶ",
    },
    "call_speech_duration": {
        "English": "Speech duration", "Hindi": "बोली की अवधि",
        "Tamil": "பேச்சு நேரம்", "Kannada": "ಮಾತಿನ ಅವಧಿ",
    },
    "call_windows": {
        "English": "Windows analysed", "Hindi": "विश्लेषित विंडो",
        "Tamil": "பகுப்பாய்வு செய்யப்பட்ட சாளரங்கள்", "Kannada": "ವಿಶ್ಲೇಷಿಸಿದ ವಿಂಡೋಗಳು",
    },
    "call_flagged_share": {
        "English": "Flagged share", "Hindi": "चिह्नित हिस्सेदारी",
        "Tamil": "குறிக்கப்பட்ட பங்கு", "Kannada": "ಗುರುತಿಸಲಾದ ಪಾಲು",
    },
    "call_highest_score": {
        "English": "Highest AI likelihood", "Hindi": "सबसे अधिक AI संभावना",
        "Tamil": "அதிகபட்ச AI சாத்தியம்", "Kannada": "ಅತ್ಯಧಿಕ AI ಸಾಧ್ಯತೆ",
    },
    "call_highest_time": {
        "English": "Highest score timestamp", "Hindi": "उच्चतम स्कोर का समय",
        "Tamil": "அதிகபட்ச மதிப்பெண் நேரம்", "Kannada": "ಅತ್ಯಧಿಕ ಅಂಕದ ಸಮಯ",
    },
    "call_timeline": {
        "English": "AI likelihood timeline", "Hindi": "AI संभावना समयरेखा",
        "Tamil": "AI சாத்திய காலவரிசை", "Kannada": "AI ಸಾಧ್ಯತೆ ಕಾಲರೇಖೆ",
    },
    "call_flagged_ranges": {
        "English": "Flagged ranges", "Hindi": "चिह्नित समय-सीमाएँ",
        "Tamil": "குறிக்கப்பட்ட நேரப் பகுதிகள்", "Kannada": "ಗುರುತಿಸಲಾದ ಸಮಯದ ವ್ಯಾಪ್ತಿಗಳು",
    },
    "call_start": {
        "English": "Start", "Hindi": "शुरुआत",
        "Tamil": "தொடக்கம்", "Kannada": "ಪ್ರಾರಂಭ",
    },
    "call_end": {
        "English": "End", "Hindi": "समाप्ति",
        "Tamil": "முடிவு", "Kannada": "ಅಂತ್ಯ",
    },
    "call_csv_button": {
        "English": "Download window results (CSV)", "Hindi": "विंडो परिणाम डाउनलोड करें (CSV)",
        "Tamil": "சாளர முடிவுகளைப் பதிவிறக்கு (CSV)", "Kannada": "ವಿಂಡೋ ಫಲಿತಾಂಶಗಳನ್ನು ಡೌನ್‌ಲೋಡ್ ಮಾಡಿ (CSV)",
    },
    "call_no_flagged": {
        "English": "No flagged ranges were found.", "Hindi": "कोई चिह्नित समय-सीमा नहीं मिली।",
        "Tamil": "குறிக்கப்பட்ட நேரப் பகுதிகள் எதுவும் இல்லை.", "Kannada": "ಯಾವುದೇ ಗುರುತಿಸಲಾದ ಸಮಯದ ವ್ಯಾಪ್ತಿಗಳು ಕಂಡುಬಂದಿಲ್ಲ.",
    },
    "call_limit_note": {
        "English": "Call analysis uses 6-second windows with 3-second overlap. Scores are model estimates, not proof that a voice is synthetic.",
        "Hindi": "कॉल विश्लेषण में 3 सेकंड के ओवरलैप के साथ 6 सेकंड की विंडो का उपयोग होता है। स्कोर मॉडल के अनुमान हैं, कृत्रिम आवाज़ का प्रमाण नहीं।",
        "Tamil": "அழைப்புப் பகுப்பாய்வு 3 வினாடி ஒன்றுடன் ஒன்று பொருந்தும் 6 வினாடி சாளரங்களைப் பயன்படுத்துகிறது. மதிப்பெண்கள் மாதிரி மதிப்பீடுகள் மட்டுமே; செயற்கைக் குரலுக்கான உறுதி அல்ல.",
        "Kannada": "ಕರೆ ವಿಶ್ಲೇಷಣೆಯು 3 ಸೆಕೆಂಡುಗಳ ಅತಿಕ್ರಮಣದೊಂದಿಗೆ 6 ಸೆಕೆಂಡುಗಳ ವಿಂಡೋಗಳನ್ನು ಬಳಸುತ್ತದೆ. ಅಂಕಗಳು ಮಾದರಿಯ ಅಂದಾಜುಗಳು ಮಾತ್ರ, ಕೃತಕ ಧ್ವನಿಯ ಪುರಾವೆಯಲ್ಲ.",
    },
})
'''

assert source.count(marker) == 1, "Translation insertion point not found exactly once."
path.write_text(source.replace(marker, translations + marker, 1))

import i18n
print("Added Call Analysis translations.")
print("Languages:", i18n.LANGUAGES)
print("New translation keys:", len([
    key for key in i18n.TRANSLATIONS if key.startswith("call_") or key.endswith("_mode") or key == "mode_label"
]))

Added Call Analysis translations.
Languages: ['English', 'Hindi', 'Tamil', 'Kannada']
New translation keys: 21


In [ ]:
from pathlib import Path

lines = Path("call_analysis.py").read_text().splitlines()

for number in range(155, 220):
    print(f"{number}: {lines[number - 1]}")

155: 
156: @dataclass(frozen=True)
157: class WindowScore:
158:     """Fake-voice score for one analysis window."""
159: 
160:     start_seconds: float
161:     end_seconds: float
162:     raw_logit: float
163:     p_fake: float
164:     verdict: str
165: 
166: 
167: @dataclass(frozen=True)
168: class CallAnalysisReport:
169:     """Complete analysis results for one phone-call recording."""
170: 
171:     duration_seconds: float
172:     speech_duration_seconds: float
173:     windows_analysed: int
174:     flagged_windows: int
175:     flagged_share: float
176:     highest_p_fake: float
177:     highest_score_timestamp: float
178:     window_scores: List[WindowScore]
179:     flagged_ranges: List[SpeechRegion]
180: 
181: 
182: def merge_flagged_ranges(
183:     window_scores: List[WindowScore],
184: ) -> List[SpeechRegion]:
185:     """Merge overlapping or adjacent windows classified as fake."""
186:     flagged = sorted(
187:         (score for score in window_scores if score.verdict

In [ ]:
from pathlib import Path

path = Path("app.py")
source = path.read_text()

marker = "\ndef render_app() -> None:"

new_function = '''

def format_call_time(seconds: float) -> str:
    total_seconds = max(0, int(seconds))
    minutes, seconds = divmod(total_seconds, 60)
    return f"{minutes:02d}:{seconds:02d}"


def render_call_analysis(lang: str) -> None:
    import csv
    import io
    import pandas as pd

    st.markdown(
        f"""
        <div class="vg-section-kicker">{html.escape(t("workspace_kicker", lang))}</div>
        <h2 class="vg-section-title">{html.escape(t("call_analysis_mode", lang))}</h2>
        """,
        unsafe_allow_html=True,
    )

    uploaded = st.file_uploader(
        t("call_upload_label", lang),
        type=ACCEPTED_TYPES,
        help=t("call_upload_help", lang),
        key="call_audio_upload",
    )

    audio_bytes = uploaded.getvalue() if uploaded is not None else None
    selected_hash = (
        hashlib.sha256(audio_bytes).hexdigest()
        if audio_bytes is not None
        else None
    )

    if selected_hash != st.session_state.call_selected_hash:
        st.session_state.call_selected_hash = selected_hash
        st.session_state.call_result = None
        st.session_state.call_result_hash = None
        st.session_state.call_result_error = None
        st.session_state.call_audio_filename = None

    if uploaded is not None and audio_bytes is not None:
        st.audio(audio_bytes)
        st.caption(
            f"{uploaded.name} · {format_bytes(len(audio_bytes))}"
        )

    analyze = st.button(
        t("call_analyze_button", lang),
        type="primary",
        use_container_width=True,
        disabled=uploaded is None,
        key="run_call_analysis",
    )

    if analyze and audio_bytes is not None:
        st.session_state.call_result = None
        st.session_state.call_result_hash = None
        st.session_state.call_result_error = None

        progress_bar = st.progress(0.0)
        progress_status = st.empty()

        def update_progress(value: float) -> None:
            value = min(1.0, max(0.0, float(value)))
            progress_bar.progress(value)
            progress_status.text(
                f"{t('call_analyzing', lang)} {value:.0%}"
            )

        try:
            with st.spinner(t("call_analyzing", lang)):
                report = call_analysis.analyze_call(
                    audio_bytes,
                    progress_callback=update_progress,
                )

            st.session_state.call_result = report
            st.session_state.call_result_hash = selected_hash
            st.session_state.call_audio_filename = uploaded.name
            progress_bar.progress(1.0)
            progress_status.empty()

        except Exception as exc:
            LOGGER.exception("Call analysis failed", exc_info=exc)
            st.session_state.call_result = None
            st.session_state.call_result_hash = None
            st.session_state.call_result_error = safe_error_message(
                exc, lang
            )
            progress_status.empty()

    if st.session_state.call_result_error:
        st.error(st.session_state.call_result_error)

    report = st.session_state.call_result
    result_is_current = (
        report is not None
        and selected_hash is not None
        and st.session_state.call_result_hash == selected_hash
    )

    if not result_is_current:
        return

    st.divider()
    st.subheader(t("call_summary", lang))

    metric_columns = st.columns(4)
    metric_columns[0].metric(
        t("call_speech_duration", lang),
        format_call_time(report.speech_duration_seconds),
    )
    metric_columns[1].metric(
        t("call_windows", lang),
        report.windows_analysed,
    )
    metric_columns[2].metric(
        t("call_flagged_share", lang),
        f"{report.flagged_share:.1%}",
    )
    metric_columns[3].metric(
        t("call_highest_score", lang),
        f"{report.highest_p_fake:.1%}",
    )

    st.caption(
        f"{t('call_highest_time', lang)}: "
        f"{format_call_time(report.highest_score_timestamp)}"
    )

    st.divider()
    st.subheader(t("call_timeline", lang))

    chart_data = pd.DataFrame([
        {
            "Time (seconds)": (
                score.start_seconds + score.end_seconds
            ) / 2,
            "AI likelihood": score.p_fake,
            "Lower threshold (0.35)": 0.35,
            "Upper threshold (0.65)": 0.65,
        }
        for score in report.window_scores
    ]).set_index("Time (seconds)")

    st.line_chart(chart_data, y_min=0.0, y_max=1.0)

    st.caption(t("call_limit_note", lang))

    st.divider()
    st.subheader(t("call_flagged_ranges", lang))

    if report.flagged_ranges:
        flagged_data = [
            {
                t("call_start", lang): format_call_time(
                    region.start_seconds
                ),
                t("call_end", lang): format_call_time(
                    region.end_seconds
                ),
            }
            for region in report.flagged_ranges
        ]
        st.dataframe(
            flagged_data,
            use_container_width=True,
            hide_index=True,
        )
    else:
        st.info(t("call_no_flagged", lang))

    csv_buffer = io.StringIO()
    writer = csv.DictWriter(
        csv_buffer,
        fieldnames=[
            "start_seconds",
            "end_seconds",
            "raw_logit",
            "p_fake",
            "verdict",
        ],
    )
    writer.writeheader()

    for score in report.window_scores:
        writer.writerow({
            "start_seconds": score.start_seconds,
            "end_seconds": score.end_seconds,
            "raw_logit": score.raw_logit,
            "p_fake": score.p_fake,
            "verdict": score.verdict,
        })

    safe_stem = Path(
        st.session_state.call_audio_filename or "call_recording"
    ).stem

    st.download_button(
        label=t("call_csv_button", lang),
        data=csv_buffer.getvalue(),
        file_name=f"{safe_stem}_call_analysis.csv",
        mime="text/csv",
        use_container_width=True,
    )

'''

assert source.count(marker) == 1, "Expected render_app insertion point not found."
path.write_text(source.replace(marker, new_function + marker, 1))

print("Call Analysis screen function added.")

Call Analysis screen function added.


In [ ]:
from pathlib import Path

path = Path("app.py")
source = path.read_text()

old = '''    render_header(lang)
    render_hero(lang)

    upload_col, process_col = st.columns([1.55, 0.85], gap="large")'''

new = '''    render_header(lang)
    render_hero(lang)

    mode = st.radio(
        t("mode_label", lang),
        options=["single", "call"],
        format_func=lambda value: (
            t("single_clip_mode", lang)
            if value == "single"
            else t("call_analysis_mode", lang)
        ),
        horizontal=True,
        key="analysis_mode",
    )

    if mode == "call":
        render_call_analysis(lang)
        st.markdown(
            f"""
            <footer class="vg-footer">
              <strong>{html.escape(t('footer_product', lang))}</strong>
              <span>{html.escape(t('privacy_note', lang))}</span>
            </footer>
            """,
            unsafe_allow_html=True,
        )
        return

    upload_col, process_col = st.columns([1.55, 0.85], gap="large")'''

assert source.count(old) == 1, "Expected render_app block not found exactly once."

path.write_text(source.replace(old, new, 1))

print("Mode switch connected.")
print("Single Clip remains the default mode.")
print("Call Analysis uses its separate screen.")

Mode switch connected.
Single Clip remains the default mode.
Call Analysis uses its separate screen.


In [ ]:
!python -m py_compile app.py i18n.py call_analysis.py

In [ ]:
!python -m pytest -q tests

.................................                                        [100%]
33 passed in 2.14s


In [ ]:
!grep -n -i streamlit requirements.txt pyproject.toml setup.py 2>/dev/null

requirements.txt:4:streamlit>=1.40,<2


In [ ]:
!python -m pip install "streamlit>=1.40,<2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 33.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 48.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [ ]:
import subprocess

log_file = open("/tmp/voice_guard_streamlit.log", "w")

process = subprocess.Popen(
    [
        "python", "-m", "streamlit", "run", "app.py",
        "--server.headless", "true",
        "--server.port", "8501",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("Streamlit process started.")
print("Process ID:", process.pid)

Streamlit process started.
Process ID: 21607


In [ ]:
!tail -n 25 /tmp/voice_guard_streamlit.log



2026-11-17 07:01:54.885 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.106.1.119:8501



In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501, height=900)

<IPython.core.display.Javascript object>

In [ ]:
import requests

response = requests.get("http://localhost:8501/_stcore/health", timeout=10)

print("Status code:", response.status_code)
print("Response:", response.text)

Status code: 200
Response: ok


In [ ]:
import os

print("Current directory:", os.getcwd())
print("Google Drive / Colab session ready!")

Current directory: /content
Google Drive / Colab session ready!


In [ ]:
import os

project_path = "/content/indic-fake-voice-detection"

if os.path.isdir(project_path):
    print("Project folder found!")
    print("Path:", project_path)
else:
    print("Project folder not found. We will clone it again.")

Project folder not found. We will clone it again.


In [ ]:
!git clone -b feature/long-call-analysis https://github.com/Shruti-sivaramakrishnan/indic-fake-voice-detection.git

Cloning into 'indic-fake-voice-detection'...
remote: Enumerating objects: 363, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 363 (delta 39), reused 73 (delta 23), pack-reused 265 (from 1)
Receiving objects: 100% (363/363), 6.79 MiB | 16.76 MiB/s, done.
Resolving deltas: 100% (148/148), done.


Start from here!!!!

In [ ]:
%cd /content/indic-fake-voice-detection
!git branch --show-current
!git status --short

/content/indic-fake-voice-detection
feature/long-call-analysis


In [ ]:
!python -m pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 15.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 14.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [streamlit]


In [ ]:
import streamlit
print("Streamlit version:", streamlit.__version__)

Streamlit version: 1.67.0


In [ ]:
import subprocess

log_file = open("/tmp/voice_guard_streamlit.log", "w")

process = subprocess.Popen(
    [
        "python", "-m", "streamlit", "run", "app.py",
        "--server.headless", "true",
        "--server.port", "8501",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("Voice Guard started. Process ID:", process.pid)

Voice Guard started. Process ID: 2344


In [ ]:
import requests

r = requests.get("http://localhost:8501/_stcore/health", timeout=10)
print(r.status_code, r.text)

200 ok


In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501, height=900)

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import output

print("Open this link in a new tab:")
print(output.eval_js("google.colab.kernel.proxyPort(8501)"))

Open this link in a new tab:
https://8501-m-s-usc1c3-cr2o5qyxo0ik-c.us-central1.prod.r.colab.dev


In [ ]:
!tail -n 30 /tmp/voice_guard_streamlit.log

2026-11-17 14:40:31.739 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-usc1c3-cr2o5qyxo0ik-c.us-central1.prod.r.colab.dev, host=m-s-usc1c3-cr2o5qyxo0ik.us-central1-c.c.codatalab-user-runtimes.internal:8007
2026-11-17 14:40:38.520 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-usc1c3-cr2o5qyxo0ik-c.us-central1.prod.r.colab.dev, host=m-s-usc1c3-cr2o5qyxo0ik.us-central1-c.c.codatalab-user-runtimes.internal:8007
2026-11-17 14:40:39.723 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-usc1c3-cr2o5qyxo0ik-c.us-central1.prod.r.colab.dev, host=m-s-usc1c3-cr2o5qyxo0ik.us-central1-c.c.codatalab-user-runtimes.internal:8007
2026-11-17 14:40:43.801 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-usc1c3-cr2o5qyxo0ik-c.us-central1.prod.r.colab.dev, host=m-s-usc1c3-cr2o5qyxo0ik.us-central1-c.c.codatalab-user-runtimes.in

In [ ]:
process.terminate()
print("Previous Streamlit server stopped.")

Previous Streamlit server stopped.


In [ ]:
import subprocess

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "python", "-m", "streamlit", "run", "app.py",
        "--server.headless=true",
        "--server.port=8501",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("Streamlit restarted. Process ID:", process.pid)

Streamlit restarted. Process ID: 3880


In [ ]:
import requests

r = requests.get("http://localhost:8501/_stcore/health", timeout=10)
print("Status:", r.status_code)
print("Response:", r.text)

Status: 200
Response: ok


In [ ]:
from google.colab import output

url = output.eval_js("google.colab.kernel.proxyPort(8501)")
print(url)

https://8501-m-s-usc1c3-cr2o5qyxo0ik-c.us-central1.prod.r.colab.dev


In [ ]:
!grep -n -B 8 -A 25 'analysis_mode' app.py

In [ ]:
!git log -1 --oneline

6af6647 (HEAD -> feature/long-call-analysis, origin/feature/long-call-analysis) Add long-call analysis core


In [ ]:
!git status --short
!grep -n "def render_app" app.py
!grep -n "def t(" i18n.py

383:def render_app() -> None:
602:def t(key: str, lang: str) -> str:


In [ ]:
!sed -n '383,455p' app.py

def render_app() -> None:
    lang = current_lang()
    render_header(lang)
    render_hero(lang)

    upload_col, process_col = st.columns([1.55, 0.85], gap="large")
    with upload_col:
        upload_panel = st.container(border=True)
        with upload_panel:
            st.markdown(
                f"""
                <div class="vg-section-kicker">{html.escape(t('workspace_kicker', lang))}</div>
                <h2 class="vg-section-title">{html.escape(t('how_it_works_title', lang))}</h2>
                <p class="vg-section-copy">{html.escape(t('how_it_works_body', lang))}</p>
                """,
                unsafe_allow_html=True,
            )
            uploaded = st.file_uploader(
                t("upload_label", lang),
                type=ACCEPTED_TYPES,
                help=t("upload_help", lang),
                label_visibility="collapsed",
                key="audio_upload",
            )
            audio_bytes = uploaded.getvalue() if uploaded is not None els

In [ ]:
!sed -n '455,520p' app.py

        render_process_card(lang)

    result_is_current = (
        st.session_state.result is not None
        and selected_hash is not None
        and st.session_state.result_hash == selected_hash
    )
    if result_is_current:
        st.markdown("<div style='height:1.2rem'></div>", unsafe_allow_html=True)
        render_result(st.session_state.result, lang)

        run_robustness = st.button(
            t("robustness_button", lang),
            use_container_width=True,
            key="run_robustness",
        )
        if run_robustness and audio_bytes is not None:
            st.session_state.robustness_error = None
            with st.spinner(t("robustness_running", lang)):
                try:
                    report = run_robustness_analysis(
                        audio_bytes,
                        st.session_state.result,
                    )
                    st.session_state.robustness_report = report
                    st.session_state.robustness_hash = se

In [ ]:
!sed -n '1,55p' app.py

"""Voice Guard Streamlit application.

Every analysis is bound to the SHA-256 hash of the selected upload. Replacing
the file clears the previous result and report before the page is rendered.
"""

from __future__ import annotations

import hashlib
import html
import logging
from pathlib import Path

try:
    import truststore

    truststore.inject_into_ssl()
except ImportError:
    pass

import streamlit as st

import codec_robustness
import evidence_report
import i18n
import ui_kit
from i18n import t


LOGGER = logging.getLogger("voice_guard.app")
MODEL_IMPORT_ERROR: Exception | None = None

try:
    import model_pipeline as mp
except Exception as exc:  # noqa: BLE001 - replaced with a safe public message
    mp = None
    MODEL_IMPORT_ERROR = exc
    LOGGER.exception("Model pipeline import failed")


st.set_page_config(
    page_title="Voice Guard · AI voice analysis",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="collapsed",
)
st.markdown(f"<style>{ui_kit.BASE_

In [ ]:
!sed -n '45,105p' app.py

    initial_sidebar_state="collapsed",
)
st.markdown(f"<style>{ui_kit.BASE_STYLES}</style>", unsafe_allow_html=True)

for key, default in {
    "ui_language": "English",
    "selected_audio_hash": None,
    "result": None,
    "result_hash": None,
    "result_error": None,
    "audio_filename": None,
    "robustness_report": None,
    "robustness_hash": None,
    "robustness_error": None,
}.items():
    st.session_state.setdefault(key, default)


ACCEPTED_TYPES = ["wav", "mp3", "flac", "ogg", "m4a"]


def current_lang() -> str:
    return st.session_state.get("ui_language", "English")


def format_bytes(size: int) -> str:
    if size < 1024:
        return f"{size} B"
    if size < 1024 * 1024:
        return f"{size / 1024:.1f} KB"
    return f"{size / (1024 * 1024):.1f} MB"


def clear_analysis_result() -> None:
    st.session_state.result = None
    st.session_state.result_hash = None
    st.session_state.result_error = None
    st.session_state.audio_filename = None
    st.session_

In [ ]:
!sed -n '570,630p' i18n.py

        "English": "No transformed condition completed successfully.",
        "Hindi": "कोई भी बदली हुई स्थिति सफलतापूर्वक पूरी नहीं हुई।",
        "Tamil": "மாற்றப்பட்ட எந்த நிலையும் வெற்றிகரமாக நிறைவடையவில்லை.",
        "Kannada": "ಯಾವುದೇ ಬದಲಾಯಿಸಿದ ಪರಿಸ್ಥಿತಿ ಯಶಸ್ವಿಯಾಗಿ ಪೂರ್ಣಗೊಂಡಿಲ್ಲ.",
    },
    "robustness_summary_changed": {
        "English": "The verdict changed in {count} of {total} successful transformed conditions.",
        "Hindi": "{total} सफल बदली हुई स्थितियों में से {count} में परिणाम बदल गया।",
        "Tamil": "{total} வெற்றிகரமான மாற்றப்பட்ட நிலைகளில் {count} இல் முடிவு மாறியது.",
        "Kannada": "{total} ಯಶಸ್ವಿ ಬದಲಾಯಿಸಿದ ಪರಿಸ್ಥಿತಿಗಳಲ್ಲಿ {count}ರಲ್ಲಿ ಫಲಿತಾಂಶ ಬದಲಾಗಿದೆ.",
    },
    "robustness_max_change": {
        "English": "Largest AI-likelihood change from the original: {value:.1f} percentage points.",
        "Hindi": "मूल ऑडियो से एआई-संभावना में सबसे बड़ा बदलाव: {value:.1f} प्रतिशत अंक।",
        "Tamil": "அசலிலிருந்து AI சாத்தியத்தின் மிகப்பெரிய மாற்றம்: 

In [ ]:
!sed -n '1,240p' call_analysis.py

"""Long-call speech detection and window segmentation for Voice Guard."""

from __future__ import annotations

from dataclasses import dataclass
from typing import List

import numpy as np


@dataclass(frozen=True)
class SpeechRegion:
    """Speech-active interval in the original recording timeline."""

    start_seconds: float
    end_seconds: float


@dataclass(frozen=True)
class AnalysisWindow:
    """A model-analysis window with its original recording timestamps."""

    start_seconds: float
    end_seconds: float
    start_sample: int
    end_sample: int


def detect_speech_regions(
    waveform: np.ndarray,
    sample_rate: int,
    frame_duration: float = 0.025,
    hop_duration: float = 0.010,
    merge_gap_seconds: float = 0.30,
    min_region_seconds: float = 0.15,
    energy_threshold: float | None = None,
) -> List[SpeechRegion]:
    """Find speech-active regions using a lightweight adaptive energy gate.

    This is an energy-based detector, not a trained voice activity de

In [ ]:
!sed -n '240,360p' call_analysis.py

    if original_sample_rate <= 0:
        raise ValueError("Invalid audio sample rate.")
    if not np.isfinite(waveform).all():
        raise ValueError("The recording contains non-finite audio values.")

    duration_seconds = waveform.size / float(original_sample_rate)

    if duration_seconds > 900.0:
        raise ValueError("Call recordings cannot exceed 15 minutes.")
    if duration_seconds < 1.0:
        raise ValueError("The recording must be at least 1 second long.")

    waveform = model_pipeline.ensure_target_sample_rate(
        waveform,
        original_sample_rate,
        model_pipeline.TARGET_SAMPLE_RATE,
    )
    sample_rate = model_pipeline.TARGET_SAMPLE_RATE

    regions = detect_speech_regions(waveform, sample_rate)
    windows = build_analysis_windows(
        regions,
        sample_rate,
        window_seconds=window_seconds,
        hop_seconds=hop_seconds,
    )

    if not windows:
        raise ValueError(
            "No speech-active regions long enough 

In [ ]:
!grep -n "call_analysis_mode\|call_upload_label\|mode_label" i18n.py

In [ ]:
!grep -n "TRANSLATIONS.update" i18n.py

232:TRANSLATIONS.update({
506:TRANSLATIONS.update({


In [ ]:
!sed -n '490,515p' i18n.py

    "English": "Automated Analysis Report", "Hindi": "स्वचालित विश्लेषण रिपोर्ट",
    "Tamil": "தானியங்கி பகுப்பாய்வு அறிக்கை", "Kannada": "ಸ್ವಯಂಚಾಲಿತ ವಿಶ್ಲೇಷಣಾ ವರದಿ",
}
TRANSLATIONS["pdf_button"] = {
    "English": "Download analysis report", "Hindi": "विश्लेषण रिपोर्ट डाउनलोड करें",
    "Tamil": "பகுப்பாய்வு அறிக்கையைப் பதிவிறக்கவும்", "Kannada": "ವಿಶ್ಲೇಷಣಾ ವರದಿಯನ್ನು ಡೌನ್‌ಲೋಡ್ ಮಾಡಿ",
}
TRANSLATIONS["pdf_body"] = {
    "English": "Download a shareable record of this automated result, its input hash, and its limitations.",
    "Hindi": "इस स्वचालित नतीजे, इनपुट हैश और इसकी सीमाओं का साझा रिकॉर्ड डाउनलोड करें।",
    "Tamil": "இந்த தானியங்கி முடிவு, உள்ளீட்டு hash மற்றும் அதன் வரம்புகளின் பகிரக்கூடிய பதிவைப் பதிவிறக்கவும்.",
    "Kannada": "ಈ ಸ್ವಯಂಚಾಲಿತ ಫಲಿತಾಂಶ, ಇನ್‌ಪುಟ್ hash ಮತ್ತು ಅದರ ಮಿತಿಗಳ ಹಂಚಬಹುದಾದ ದಾಖಲೆಯನ್ನು ಡೌನ್‌ಲೋಡ್ ಮಾಡಿ.",
}

# Transmission robustness comparison added to the upload workflow. Technical
# condition names (Opus/G.711/8 kHz) remain standards-based proper names.
TRANS

In [ ]:
from pathlib import Path

path = Path("i18n.py")
source = path.read_text(encoding="utf-8")

marker = "\n\ndef t(key: str, lang: str) -> str:"
assert source.count(marker) == 1, "Expected insertion point not found uniquely."

translations = '''

# Long-call analysis interface translations.
TRANSLATIONS.update({
    "mode_label": {
        "English": "Analysis Mode", "Hindi": "विश्लेषण मोड",
        "Tamil": "பகுப்பாய்வு முறை", "Kannada": "ವಿಶ್ಲೇಷಣಾ ವಿಧಾನ",
    },
    "single_clip_mode": {
        "English": "Single Clip", "Hindi": "एकल क्लिप",
        "Tamil": "ஒற்றை ஒலி", "Kannada": "ಒಂದು ಆಡಿಯೊ ಕ್ಲಿಪ್",
    },
    "call_analysis_mode": {
        "English": "Call Analysis", "Hindi": "कॉल विश्लेषण",
        "Tamil": "அழைப்பு பகுப்பாய்வு", "Kannada": "ಕರೆ ವಿಶ್ಲೇಷಣೆ",
    },
    "call_upload_label": {
        "English": "Upload a phone-call recording",
        "Hindi": "फ़ोन कॉल की रिकॉर्डिंग अपलोड करें",
        "Tamil": "தொலைபேசி அழைப்புப் பதிவேற்றம்",
        "Kannada": "ದೂರವಾಣಿ ಕರೆಯ ಧ್ವನಿಮುದ್ರಿಕೆಯನ್ನು ಅಪ್‌ಲೋಡ್ ಮಾಡಿ",
    },
    "call_upload_help": {
        "English": "WAV, MP3, FLAC, OGG or M4A. Long-call analysis supports recordings up to 15 minutes.",
        "Hindi": "WAV, MP3, FLAC, OGG या M4A। अधिकतम 15 मिनट की रिकॉर्डिंग।",
        "Tamil": "WAV, MP3, FLAC, OGG அல்லது M4A. அதிகபட்சம் 15 நிமிடப் பதிவு.",
        "Kannada": "WAV, MP3, FLAC, OGG ಅಥವಾ M4A. ಗರಿಷ್ಠ 15 ನಿಮಿಷಗಳ ಧ್ವನಿಮುದ್ರಿಕೆ.",
    },
    "call_analyze_button": {
        "English": "Analyze Call", "Hindi": "कॉल का विश्लेषण करें",
        "Tamil": "அழைப்பைப் பகுப்பாய்வு செய்", "Kannada": "ಕರೆಯನ್ನು ವಿಶ್ಲೇಷಿಸಿ",
    },
    "call_analyzing": {
        "English": "Analyzing call...", "Hindi": "कॉल का विश्लेषण हो रहा है...",
        "Tamil": "அழைப்பு பகுப்பாய்வு செய்யப்படுகிறது...",
        "Kannada": "ಕರೆಯನ್ನು ವಿಶ್ಲೇಷಿಸಲಾಗುತ್ತಿದೆ...",
    },
    "call_progress": {
        "English": "Analysis progress", "Hindi": "विश्लेषण की प्रगति",
        "Tamil": "பகுப்பாய்வு முன்னேற்றம்", "Kannada": "ವಿಶ್ಲೇಷಣೆಯ ಪ್ರಗತಿ",
    },
    "call_summary": {
        "English": "Call Analysis Summary", "Hindi": "कॉल विश्लेषण सारांश",
        "Tamil": "அழைப்புப் பகுப்பாய்வு சுருக்கம்",
        "Kannada": "ಕರೆ ವಿಶ್ಲೇಷಣೆಯ ಸಾರಾಂಶ",
    },
    "call_speech_duration": {
        "English": "Speech-active duration", "Hindi": "सक्रिय भाषण अवधि",
        "Tamil": "பேச்சு செயல்பாட்டு நேரம்", "Kannada": "ಮಾತಿನ ಸಕ್ರಿಯ ಅವಧಿ",
    },
    "call_windows": {
        "English": "Windows analyzed", "Hindi": "विश्लेषित विंडो",
        "Tamil": "பகுப்பாய்வு செய்யப்பட்ட பகுதிகள்",
        "Kannada": "ವಿಶ್ಲೇಷಿಸಿದ ವಿಭಾಗಗಳು",
    },
    "call_flagged_share": {
        "English": "Flagged share", "Hindi": "चिह्नित हिस्सा",
        "Tamil": "குறிக்கப்பட்ட பங்கு", "Kannada": "ಗುರುತಿಸಲಾದ ಪಾಲು",
    },
    "call_highest_score": {
        "English": "Highest AI likelihood", "Hindi": "सबसे अधिक AI संभावना",
        "Tamil": "அதிகபட்ச AI சாத்தியம்", "Kannada": "ಅತಿ ಹೆಚ್ಚು AI ಸಾಧ್ಯತೆ",
    },
    "call_highest_time": {
        "English": "Highest score timestamp", "Hindi": "उच्चतम स्कोर का समय",
        "Tamil": "அதிகபட்ச மதிப்பெண் நேரம்",
        "Kannada": "ಅತ್ಯಧಿಕ ಅಂಕದ ಸಮಯ",
    },
    "call_timeline": {
        "English": "AI Likelihood Timeline", "Hindi": "AI संभावना समयरेखा",
        "Tamil": "AI சாத்தியக்கூறு காலவரிசை",
        "Kannada": "AI ಸಾಧ್ಯತೆಯ ಕಾಲರೇಖೆ",
    },
    "call_flagged_ranges": {
        "English": "Flagged Time Ranges", "Hindi": "चिह्नित समय सीमाएँ",
        "Tamil": "குறிக்கப்பட்ட நேர இடைவெளிகள்",
        "Kannada": "ಗುರುತಿಸಲಾದ ಸಮಯದ ವ್ಯಾಪ್ತಿಗಳು",
    },
    "call_start": {
        "English": "Start", "Hindi": "प्रारंभ",
        "Tamil": "தொடக்கம்", "Kannada": "ಪ್ರಾರಂಭ",
    },
    "call_end": {
        "English": "End", "Hindi": "समाप्ति",
        "Tamil": "முடிவு", "Kannada": "ಅಂತ್ಯ",
    },
    "call_csv_button": {
        "English": "Download Window Results (CSV)",
        "Hindi": "विंडो परिणाम डाउनलोड करें (CSV)",
        "Tamil": "பகுதி முடிவுகளைப் பதிவிறக்கு (CSV)",
        "Kannada": "ವಿಭಾಗದ ಫಲಿತಾಂಶಗಳನ್ನು ಡೌನ್‌ಲೋಡ್ ಮಾಡಿ (CSV)",
    },
    "call_no_flagged": {
        "English": "No windows were classified as fake.",
        "Hindi": "किसी भी विंडो को नकली के रूप में वर्गीकृत नहीं किया गया।",
        "Tamil": "எந்தப் பகுதியும் போலியானது என வகைப்படுத்தப்படவில்லை.",
        "Kannada": "ಯಾವುದೇ ವಿಭಾಗವನ್ನು ನಕಲಿ ಎಂದು ವರ್ಗೀಕರಿಸಲಾಗಿಲ್ಲ.",
    },
    "call_limit_note": {
        "English": "The speech regions are estimated using audio energy, not a trained voice activity detector. Thresholds shown are 0.35 and 0.65.",
        "Hindi": "भाषण क्षेत्रों का अनुमान ऑडियो ऊर्जा से लगाया जाता है, प्रशिक्षित वॉइस एक्टिविटी डिटेक्टर से नहीं। दिखाई गई सीमाएँ 0.35 और 0.65 हैं।",
        "Tamil": "பயிற்சி பெற்ற குரல் செயல்பாட்டுக் கண்டறிவி அல்ல, ஒலி ஆற்றலைப் பயன்படுத்தியே பேச்சுப் பகுதிகள் கணிக்கப்படுகின்றன. காட்டப்படும் வரம்புகள் 0.35 மற்றும் 0.65.",
        "Kannada": "ತರಬೇತಿ ಪಡೆದ ಧ್ವನಿ ಚಟುವಟಿಕೆ ಪತ್ತೆಕಾರಕದ ಬದಲಿಗೆ ಆಡಿಯೊ ಶಕ್ತಿಯಿಂದ ಮಾತಿನ ಪ್ರದೇಶಗಳನ್ನು ಅಂದಾಜಿಸಲಾಗುತ್ತದೆ. ತೋರಿಸಿರುವ ಮಿತಿಗಳು 0.35 ಮತ್ತು 0.65.",
    },
})
'''

path.write_text(source.replace(marker, "\n" + translations + marker), encoding="utf-8")
print("Call Analysis translations added.")

Call Analysis translations added.


In [ ]:
!python -m py_compile i18n.py && grep -n '"mode_label"\|"call_csv_button"\|"call_limit_note"' i18n.py

605:    "mode_label": {
687:    "call_csv_button": {
699:    "call_limit_note": {


In [ ]:
from pathlib import Path

path = Path("app.py")
source = path.read_text(encoding="utf-8")

assert "import call_analysis" not in source, "Import already exists."
marker = "import codec_robustness\n"
assert source.count(marker) == 1, "Import insertion point not found uniquely."

path.write_text(
    source.replace(marker, "import call_analysis\n" + marker),
    encoding="utf-8",
)
print("Added call_analysis import.")

Added call_analysis import.


In [ ]:
!grep -n -B 2 -A 3 "import call_analysis" app.py

21-import streamlit as st
22-
23:import call_analysis
24-import codec_robustness
25-import evidence_report
26-import i18n


In [ ]:
from pathlib import Path

path = Path("app.py")
source = path.read_text(encoding="utf-8")

marker = '    "robustness_error": None,\n'
assert source.count(marker) == 1, "Session-state insertion point not found."

addition = '''    "call_selected_hash": None,
    "call_result": None,
    "call_result_hash": None,
    "call_result_error": None,
    "call_audio_filename": None,
'''

path.write_text(
    source.replace(marker, marker + addition),
    encoding="utf-8",
)
print("Added five Call Analysis session-state keys.")

Added five Call Analysis session-state keys.


In [ ]:
!python -m py_compile app.py && sed -n '42,75p' app.py

st.set_page_config(
    page_title="Voice Guard · AI voice analysis",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="collapsed",
)
st.markdown(f"<style>{ui_kit.BASE_STYLES}</style>", unsafe_allow_html=True)

for key, default in {
    "ui_language": "English",
    "selected_audio_hash": None,
    "result": None,
    "result_hash": None,
    "result_error": None,
    "audio_filename": None,
    "robustness_report": None,
    "robustness_hash": None,
    "robustness_error": None,
    "call_selected_hash": None,
    "call_result": None,
    "call_result_hash": None,
    "call_result_error": None,
    "call_audio_filename": None,
}.items():
    st.session_state.setdefault(key, default)


ACCEPTED_TYPES = ["wav", "mp3", "flac", "ogg", "m4a"]


def current_lang() -> str:
    return st.session_state.get("ui_language", "English")




In [ ]:
!grep -n "^def render_app" app.py

389:def render_app() -> None:


In [ ]:
from pathlib import Path

path = Path("app.py")
source = path.read_text(encoding="utf-8")

marker = "def render_app() -> None:\n"
assert source.count(marker) == 1, "render_app insertion point not found."

block = '''def format_call_time(seconds: float) -> str:
    total_seconds = max(0, int(seconds))
    minutes, seconds = divmod(total_seconds, 60)
    return f"{minutes:02d}:{seconds:02d}"


def render_call_analysis(lang: str) -> None:
    import csv
    import io

    import pandas as pd

    st.markdown(
        f"""
        <div class="vg-section-kicker">{html.escape(t("workspace_kicker", lang))}</div>
        <h2 class="vg-section-title">{html.escape(t("call_analysis_mode", lang))}</h2>
        """,
        unsafe_allow_html=True,
    )

    uploaded = st.file_uploader(
        t("call_upload_label", lang),
        type=ACCEPTED_TYPES,
        help=t("call_upload_help", lang),
        key="call_audio_upload",
    )

    audio_bytes = uploaded.getvalue() if uploaded is not None else None
    selected_hash = (
        hashlib.sha256(audio_bytes).hexdigest()
        if audio_bytes is not None
        else None
    )

    if selected_hash != st.session_state.call_selected_hash:
        st.session_state.call_selected_hash = selected_hash
        st.session_state.call_result = None
        st.session_state.call_result_hash = None
        st.session_state.call_result_error = None
        st.session_state.call_audio_filename = None

    if uploaded is not None and audio_bytes is not None:
        st.audio(audio_bytes)
        st.caption(f"{uploaded.name} · {format_bytes(len(audio_bytes))}")

    analyze = st.button(
        t("call_analyze_button", lang),
        type="primary",
        use_container_width=True,
        disabled=uploaded is None,
        key="run_call_analysis",
    )

    if analyze and audio_bytes is not None:
        st.session_state.call_result = None
        st.session_state.call_result_hash = None
        st.session_state.call_result_error = None

        progress_bar = st.progress(0.0)
        progress_status = st.empty()

        def update_progress(value: float) -> None:
            value = min(1.0, max(0.0, float(value)))
            progress_bar.progress(value)
            progress_status.text(
                f"{t('call_analyzing', lang)} {value:.0%}"
            )

        try:
            with st.spinner(t("call_analyzing", lang)):
                report = call_analysis.analyze_call(
                    audio_bytes,
                    progress_callback=update_progress,
                )

            st.session_state.call_result = report
            st.session_state.call_result_hash = selected_hash
            st.session_state.call_audio_filename = uploaded.name
            progress_bar.progress(1.0)
            progress_status.empty()

        except Exception as exc:  # noqa: BLE001
            LOGGER.exception("Call analysis failed", exc_info=exc)
            st.session_state.call_result = None
            st.session_state.call_result_hash = None
            st.session_state.call_result_error = safe_error_message(
                exc, lang
            )
            progress_status.empty()

    if st.session_state.call_result_error:
        st.error(st.session_state.call_result_error)

    report = st.session_state.call_result
    result_is_current = (
        report is not None
        and selected_hash is not None
        and st.session_state.call_result_hash == selected_hash
    )

    if not result_is_current:
        return

    st.divider()
    st.subheader(t("call_summary", lang))

    metric_columns = st.columns(4)
    metric_columns[0].metric(
        t("call_speech_duration", lang),
        format_call_time(report.speech_duration_seconds),
    )
    metric_columns[1].metric(
        t("call_windows", lang),
        report.windows_analysed,
    )
    metric_columns[2].metric(
        t("call_flagged_share", lang),
        f"{report.flagged_share:.1%}",
    )
    metric_columns[3].metric(
        t("call_highest_score", lang),
        f"{report.highest_p_fake:.1%}",
    )

    st.caption(
        f"{t('call_highest_time', lang)}: "
        f"{format_call_time(report.highest_score_timestamp)}"
    )

    st.divider()
    st.subheader(t("call_timeline", lang))

    chart_data = pd.DataFrame([
        {
            "Time (seconds)": (
                score.start_seconds + score.end_seconds
            ) / 2,
            "AI likelihood": score.p_fake,
            "Lower threshold (0.35)": 0.35,
            "Upper threshold (0.65)": 0.65,
        }
        for score in report.window_scores
    ]).set_index("Time (seconds)")

    st.line_chart(chart_data, y_min=0.0, y_max=1.0)
    st.caption(t("call_limit_note", lang))

    st.divider()
    st.subheader(t("call_flagged_ranges", lang))

    if report.flagged_ranges:
        flagged_data = [
            {
                t("call_start", lang): format_call_time(region.start_seconds),
                t("call_end", lang): format_call_time(region.end_seconds),
            }
            for region in report.flagged_ranges
        ]
        st.dataframe(
            flagged_data,
            use_container_width=True,
            hide_index=True,
        )
    else:
        st.info(t("call_no_flagged", lang))

    csv_buffer = io.StringIO()
    writer = csv.DictWriter(
        csv_buffer,
        fieldnames=[
            "start_seconds",
            "end_seconds",
            "raw_logit",
            "p_fake",
            "verdict",
        ],
    )
    writer.writeheader()

    for score in report.window_scores:
        writer.writerow({
            "start_seconds": score.start_seconds,
            "end_seconds": score.end_seconds,
            "raw_logit": score.raw_logit,
            "p_fake": score.p_fake,
            "verdict": score.verdict,
        })

    safe_stem = Path(
        st.session_state.call_audio_filename or "call_recording"
    ).stem

    st.download_button(
        label=t("call_csv_button", lang),
        data=csv_buffer.getvalue(),
        file_name=f"{safe_stem}_call_analysis.csv",
        mime="text/csv",
        use_container_width=True,
    )


'''

path.write_text(source.replace(marker, block + marker), encoding="utf-8")
print("Added Call Analysis rendering functions.")

Added Call Analysis rendering functions.


In [ ]:
!python -m py_compile app.py i18n.py call_analysis.py

In [ ]:
from pathlib import Path

path = Path("app.py")
source = path.read_text(encoding="utf-8")

marker = '''    render_header(lang)
    render_hero(lang)

    upload_col, process_col = st.columns([1.55, 0.85], gap="large")
'''

assert source.count(marker) == 1, "Mode selector insertion point not found."

replacement = '''    render_header(lang)
    render_hero(lang)

    mode = st.radio(
        t("mode_label", lang),
        options=["single", "call"],
        format_func=lambda value: (
            t("single_clip_mode", lang)
            if value == "single"
            else t("call_analysis_mode", lang)
        ),
        horizontal=True,
        key="analysis_mode",
    )

    if mode == "call":
        render_call_analysis(lang)
        st.markdown(
            f"""
            <footer class="vg-footer">
              <strong>{html.escape(t('footer_product', lang))}</strong>
              <span>{html.escape(t('privacy_note', lang))}</span>
            </footer>
            """,
            unsafe_allow_html=True,
        )
        return

    upload_col, process_col = st.columns([1.55, 0.85], gap="large")
'''

path.write_text(source.replace(marker, replacement), encoding="utf-8")
print("Connected Single Clip / Call Analysis mode selector.")

Connected Single Clip / Call Analysis mode selector.


In [ ]:
!python -m py_compile app.py i18n.py call_analysis.py

In [ ]:
import requests

response = requests.get("http://localhost:8501/_stcore/health")
print("Streamlit status:", response.status_code)
print("Response:", response.text)

Streamlit status: 200
Response: ok


In [ ]:
from pathlib import Path

path = Path("app.py")
source = path.read_text(encoding="utf-8")

old = 'st.line_chart(chart_data, y_min=0.0, y_max=1.0)'
new = 'st.line_chart(chart_data)'

assert source.count(old) == 1, "Expected chart line not found."

path.write_text(source.replace(old, new), encoding="utf-8")
print("Fixed Streamlit timeline chart compatibility.")

Fixed Streamlit timeline chart compatibility.


In [ ]:
%cd /content/indic-fake-voice-detection
!git branch --show-current
!git status --short

/content/indic-fake-voice-detection
feature/long-call-analysis
 M app.py
 M i18n.py


In [ ]:
!git diff --check
!git diff --stat

 app.py  | 232 ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 i18n.py | 106 +++++++++++++++++++++++++++++
 2 files changed, 338 insertions(+)


In [ ]:
!git add app.py i18n.py && git commit -m "Add Call Analysis Streamlit interface"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@9a9ade11756f.(none)')


In [ ]:
!git config user.name "Ashitha098"
!git config user.email "rajashitha310@gmail.com"
!git config --local --list

core.repositoryformatversion=0
core.filemode=true
core.bare=false
core.logallrefupdates=true
remote.origin.url=https://github.com/Shruti-sivaramakrishnan/indic-fake-voice-detection.git
remote.origin.fetch=+refs/heads/*:refs/remotes/origin/*
branch.feature/long-call-analysis.remote=origin
branch.feature/long-call-analysis.merge=refs/heads/feature/long-call-analysis
user.name=Ashitha098
user.email=rajashitha310@gmail.com


In [ ]:
!git commit -m "Add Call Analysis Streamlit interface"

[feature/long-call-analysis 7723489] Add Call Analysis Streamlit interface
 2 files changed, 338 insertions(+)


In [ ]:
!git push origin feature/long-call-analysis

fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
from getpass import getpass
import subprocess

token = getpass("Paste your GitHub token (input is hidden): ")

subprocess.run(
    ["git", "config", "credential.helper", "store"],
    check=True
)

subprocess.run(
    ["git", "credential", "approve"],
    input=f"protocol=https\nhost=github.com\nusername=Ashitha098\npassword={token}\n\n",
    text=True,
    check=True
)

result = subprocess.run(
    ["git", "push", "origin", "feature/long-call-analysis"],
    text=True
)

del token
print("Push exit code:", result.returncode)

Paste your GitHub token (input is hidden): ··········
Push exit code: 0


In [ ]:
!git status --short && git log -1 --oneline

7723489 (HEAD -> feature/long-call-analysis, origin/feature/long-call-analysis) Add Call Analysis Streamlit interface


From HERE !!!!

In [ ]:
!pwd

/content


In [ ]:
!ls -la /content/indic-fake-voice-detection

ls: cannot access '/content/indic-fake-voice-detection': No such file or directory


In [ ]:
!git clone -b feature/long-call-analysis https://github.com/Shruti-sivaramakrishnan/indic-fake-voice-detection.git

Cloning into 'indic-fake-voice-detection'...
remote: Enumerating objects: 367, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 367 (delta 42), reused 78 (delta 27), pack-reused 265 (from 1)
Receiving objects: 100% (367/367), 6.80 MiB | 25.30 MiB/s, done.
Resolving deltas: 100% (151/151), done.


In [ ]:
%cd /content/indic-fake-voice-detection
!git status --short
!git branch --show-current
!git log -1 --oneline

/content/indic-fake-voice-detection
feature/long-call-analysis
7723489 (HEAD -> feature/long-call-analysis, origin/feature/long-call-analysis) Add Call Analysis Streamlit interface


In [ ]:
!grep -n -E "def render_call_analysis|How the review works|def render_app|render_report_download" app.py

362:def render_report_download(result, filename: str, lang: str) -> None:
395:def render_call_analysis(lang: str) -> None:
590:def render_app() -> None:
728:        render_report_download(


In [ ]:
!sed -n '395,589p' app.py

def render_call_analysis(lang: str) -> None:
    import csv
    import io

    import pandas as pd

    st.markdown(
        f"""
        <div class="vg-section-kicker">{html.escape(t("workspace_kicker", lang))}</div>
        <h2 class="vg-section-title">{html.escape(t("call_analysis_mode", lang))}</h2>
        """,
        unsafe_allow_html=True,
    )

    uploaded = st.file_uploader(
        t("call_upload_label", lang),
        type=ACCEPTED_TYPES,
        help=t("call_upload_help", lang),
        key="call_audio_upload",
    )

    audio_bytes = uploaded.getvalue() if uploaded is not None else None
    selected_hash = (
        hashlib.sha256(audio_bytes).hexdigest()
        if audio_bytes is not None
        else None
    )

    if selected_hash != st.session_state.call_selected_hash:
        st.session_state.call_selected_hash = selected_hash
        st.session_state.call_result = None
        st.session_state.call_result_hash = None
        st.session_state.call_result_error = 

In [ ]:
!sed -n '590,750p' app.py

def render_app() -> None:
    lang = current_lang()
    render_header(lang)
    render_hero(lang)

    mode = st.radio(
        t("mode_label", lang),
        options=["single", "call"],
        format_func=lambda value: (
            t("single_clip_mode", lang)
            if value == "single"
            else t("call_analysis_mode", lang)
        ),
        horizontal=True,
        key="analysis_mode",
    )

    if mode == "call":
        render_call_analysis(lang)
        st.markdown(
            f"""
            <footer class="vg-footer">
              <strong>{html.escape(t('footer_product', lang))}</strong>
              <span>{html.escape(t('privacy_note', lang))}</span>
            </footer>
            """,
            unsafe_allow_html=True,
        )
        return

    upload_col, process_col = st.columns([1.55, 0.85], gap="large")
    with upload_col:
        upload_panel = st.container(border=True)
        with upload_panel:
            st.markdown(
                f"""


In [ ]:
!grep -n -A65 "def render_process_card" app.py

182:def render_process_card(lang: str) -> None:
183-    steps = (
184-        ("01", t("step_one_title", lang), t("step_one_body", lang)),
185-        ("02", t("step_two_title", lang), t("step_two_body", lang)),
186-        ("03", t("step_three_title", lang), t("step_three_body", lang)),
187-    )
188-    steps_html = "".join(
189-        (
190-            '<div class="vg-step">'
191-            f'<span class="vg-step-index">{index}</span>'
192-            f'<div><strong>{html.escape(title)}</strong><p>{html.escape(body)}</p></div>'
193-            "</div>"
194-        )
195-        for index, title, body in steps
196-    )
197-    st.markdown(
198-        f"""
199-        <aside class="vg-process-card">
200-          <h3>{html.escape(t('process_title', lang))}</h3>
201-          <p class="vg-process-intro">{html.escape(t('process_intro', lang))}</p>
202-          {steps_html}
203-          <div class="vg-privacy-strip">◈&nbsp; {html.escape(t('session_note', lang))}</div>
204-        <

In [ ]:

from pathlib import Path
import py_compile

app_path = Path("/content/indic-fake-voice-detection/app.py")

source = app_path.read_text(encoding="utf-8")

start = source.index("def render_call_analysis(lang: str) -> None:")
end = source.index("\ndef render_app()", start)

backup_path = app_path.with_name("app.py.before_call_layout.py")
backup_path.write_text(source, encoding="utf-8")

new_function = '''def render_call_analysis(lang: str) -> None:
    import csv
    import io

    import pandas as pd

    upload_col, process_col = st.columns([1.55, 0.85], gap="large")

    with upload_col:
        upload_panel = st.container(border=True)

        with upload_panel:
            st.markdown(
                f"""
                <div class="vg-section-kicker">{html.escape(t("workspace_kicker", lang))}</div>
                <h2 class="vg-section-title">{html.escape(t("call_analysis_mode", lang))}</h2>
                """,
                unsafe_allow_html=True,
            )

            uploaded = st.file_uploader(
                t("call_upload_label", lang),
                type=ACCEPTED_TYPES,
                help=t("call_upload_help", lang),
                key="call_audio_upload",
            )

            audio_bytes = uploaded.getvalue() if uploaded is not None else None

            selected_hash = (
                hashlib.sha256(audio_bytes).hexdigest()
                if audio_bytes is not None
                else None
            )

            if selected_hash != st.session_state.call_selected_hash:
                st.session_state.call_selected_hash = selected_hash
                st.session_state.call_result = None
                st.session_state.call_result_hash = None
                st.session_state.call_result_error = None
                st.session_state.call_audio_filename = None

            if uploaded is not None and audio_bytes is not None:
                st.audio(audio_bytes)
                st.caption(
                    f"{uploaded.name} · {format_bytes(len(audio_bytes))}"
                )

            analyze = st.button(
                t("call_analyze_button", lang),
                type="primary",
                use_container_width=True,
                disabled=uploaded is None,
                key="run_call_analysis",
            )

            if analyze and audio_bytes is not None:
                st.session_state.call_result = None
                st.session_state.call_result_hash = None
                st.session_state.call_result_error = None

                progress_bar = st.progress(0.0)
                progress_status = st.empty()

                def update_progress(value: float) -> None:
                    value = min(1.0, max(0.0, float(value)))
                    progress_bar.progress(value)
                    progress_status.text(
                        f"{t('call_analyzing', lang)} {value:.0%}"
                    )

                try:
                    with st.spinner(t("call_analyzing", lang)):
                        report = call_analysis.analyze_call(
                            audio_bytes,
                            progress_callback=update_progress,
                        )

                    st.session_state.call_result = report
                    st.session_state.call_result_hash = selected_hash
                    st.session_state.call_audio_filename = uploaded.name

                    progress_bar.progress(1.0)
                    progress_status.empty()

                except Exception as exc:  # noqa: BLE001
                    LOGGER.exception("Call analysis failed", exc_info=exc)
                    st.session_state.call_result = None
                    st.session_state.call_result_hash = None
                    st.session_state.call_result_error = safe_error_message(
                        exc, lang
                    )
                    progress_status.empty()

            if st.session_state.call_result_error:
                st.error(st.session_state.call_result_error)

    with process_col:
        render_process_card(lang)

    report = st.session_state.call_result

    result_is_current = (
        report is not None
        and selected_hash is not None
        and st.session_state.call_result_hash == selected_hash
    )

    if not result_is_current:
        return

    st.divider()
    st.subheader(t("call_summary", lang))

    metric_columns = st.columns(4)

    metric_columns[0].metric(
        t("call_speech_duration", lang),
        format_call_time(report.speech_duration_seconds),
    )

    metric_columns[1].metric(
        t("call_windows", lang),
        report.windows_analysed,
    )

    metric_columns[2].metric(
        t("call_flagged_share", lang),
        f"{report.flagged_share:.1%}",
    )

    metric_columns[3].metric(
        t("call_highest_score", lang),
        f"{report.highest_p_fake:.1%}",
    )

    st.caption(
        f"{t('call_highest_time', lang)}: "
        f"{format_call_time(report.highest_score_timestamp)}"
    )

    st.divider()
    st.subheader(t("call_timeline", lang))

    chart_data = pd.DataFrame([
        {
            "Time (seconds)": (
                score.start_seconds + score.end_seconds
            ) / 2,
            "AI likelihood": score.p_fake,
            "Lower threshold (0.35)": 0.35,
            "Upper threshold (0.65)": 0.65,
        }
        for score in report.window_scores
    ]).set_index("Time (seconds)")

    st.line_chart(chart_data)
    st.caption(t("call_limit_note", lang))

    st.divider()
    st.subheader(t("call_flagged_ranges", lang))

    if report.flagged_ranges:
        flagged_data = [
            {
                t("call_start", lang): format_call_time(
                    region.start_seconds
                ),
                t("call_end", lang): format_call_time(
                    region.end_seconds
                ),
            }
            for region in report.flagged_ranges
        ]

        st.dataframe(
            flagged_data,
            use_container_width=True,
            hide_index=True,
        )
    else:
        st.info(t("call_no_flagged", lang))

    csv_buffer = io.StringIO()

    writer = csv.DictWriter(
        csv_buffer,
        fieldnames=[
            "start_seconds",
            "end_seconds",
            "raw_logit",
            "p_fake",
            "verdict",
        ],
    )

    writer.writeheader()

    for score in report.window_scores:
        writer.writerow({
            "start_seconds": score.start_seconds,
            "end_seconds": score.end_seconds,
            "raw_logit": score.raw_logit,
            "p_fake": score.p_fake,
            "verdict": score.verdict,
        })

    safe_stem = Path(
        st.session_state.call_audio_filename or "call_recording"
    ).stem

    st.download_button(
        label=t("call_csv_button", lang),
        data=csv_buffer.getvalue(),
        file_name=f"{safe_stem}_call_analysis.csv",
        mime="text/csv",
        use_container_width=True,
    )
'''

updated_source = source[:start] + new_function + source[end:]

app_path.write_text(updated_source, encoding="utf-8")

py_compile.compile(str(app_path), doraise=True)

print("SUCCESS: Call Analysis layout updated.")
print("SUCCESS: Original app.py backup created.")
print("SUCCESS: Python syntax check passed.")


SUCCESS: Call Analysis layout updated.
SUCCESS: Original app.py backup created.
SUCCESS: Python syntax check passed.


In [ ]:
import subprocess
import time
from pathlib import Path

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "python", "-m", "streamlit", "run", "app.py",
        "--server.headless=true",
        "--server.port=8501",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

time.sleep(5)

print("Streamlit process running:", process.poll() is None)
print(Path("/content/streamlit.log").read_text()[-2000:])

Streamlit process running: False
/usr/bin/python3: No module named streamlit



In [ ]:
!ls -la | grep -E 'requirements|pyproject|environment'

-rw-r--r--  1 root root     34 Nov 18 05:32 requirements-kaggle.txt
-rw-r--r--  1 root root    187 Nov 18 05:32 requirements-training.txt
-rw-r--r--  1 root root    369 Nov 18 05:32 requirements.txt


In [ ]:
!cat requirements.txt

# Runtime dependencies for app.py and analyze_bytes(). Upper bounds prevent a
# new major release from silently changing the public Kaggle demo. Torch and
# torchaudio must resolve to the same release family.
streamlit>=1.40,<2
torch>=2.2,<3
torchaudio>=2.2,<3
transformers>=4.40,<6
numpy>=1.26,<3
soundfile>=0.12,<1
pydub>=0.25,<1
reportlab>=4.0,<5
truststore>=0.9,<1


In [ ]:
import importlib.metadata as metadata

packages = ["streamlit", "torch", "torchaudio", "transformers", "numpy", "soundfile", "pydub", "reportlab"]

for package in packages:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

streamlit: NOT INSTALLED
torch: 2.14.1+cpu
torchaudio: 2.11.0+cpu
transformers: 5.22.1
numpy: 2.4.6
soundfile: 0.14.0
pydub: 0.25.1
reportlab: NOT INSTALLED


In [ ]:
!pip install "streamlit>=1.40,<2" "reportlab>=4.0,<5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 92.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 99.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [streamlit]


In [ ]:
import streamlit
import reportlab

print("Streamlit:", streamlit.__version__)
print("ReportLab:", reportlab.Version)
print("Dependencies imported successfully.")

Streamlit: 1.67.0
ReportLab: 4.5.1
Dependencies imported successfully.


In [ ]:
import subprocess
import time
import requests

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "python", "-m", "streamlit", "run", "app.py",
        "--server.headless=true",
        "--server.port=8501",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

time.sleep(8)

print("Process running:", process.poll() is None)

try:
    response = requests.get("http://localhost:8501/_stcore/health")
    print("Health check:", response.status_code, response.text)
except Exception as error:
    print("Health check failed:", error)

Process running: True
Health check: 200 ok


In [ ]:
from google.colab import output

print(output.eval_js("google.colab.kernel.proxyPort(8501)"))

https://8501-m-s-use4a8-16g00xd535zsg-a.us-east4.prod.r.colab.dev


In [ ]:
!grep -n -A35 -B10 "def t(" i18n.py

698-    },
699-    "call_limit_note": {
700-        "English": "The speech regions are estimated using audio energy, not a trained voice activity detector. Thresholds shown are 0.35 and 0.65.",
701-        "Hindi": "भाषण क्षेत्रों का अनुमान ऑडियो ऊर्जा से लगाया जाता है, प्रशिक्षित वॉइस एक्टिविटी डिटेक्टर से नहीं। दिखाई गई सीमाएँ 0.35 और 0.65 हैं।",
702-        "Tamil": "பயிற்சி பெற்ற குரல் செயல்பாட்டுக் கண்டறிவி அல்ல, ஒலி ஆற்றலைப் பயன்படுத்தியே பேச்சுப் பகுதிகள் கணிக்கப்படுகின்றன. காட்டப்படும் வரம்புகள் 0.35 மற்றும் 0.65.",
703-        "Kannada": "ತರಬೇತಿ ಪಡೆದ ಧ್ವನಿ ಚಟುವಟಿಕೆ ಪತ್ತೆಕಾರಕದ ಬದಲಿಗೆ ಆಡಿಯೊ ಶಕ್ತಿಯಿಂದ ಮಾತಿನ ಪ್ರದೇಶಗಳನ್ನು ಅಂದಾಜಿಸಲಾಗುತ್ತದೆ. ತೋರಿಸಿರುವ ಮಿತಿಗಳು 0.35 ಮತ್ತು 0.65.",
704-    },
705-})
706-
707-
708:def t(key: str, lang: str) -> str:
709-    """Look up a translated UI string. Falls back to English, then the raw key,
710-    so a missing translation degrades to readable text instead of a crash."""
711-    entry = TRANSLATIONS.get(key)
712-    if entry is None:
713-        re

In [ ]:
import i18n

test_keys = [
    "mode_label",
    "call_analysis_mode",
    "call_summary",
    "call_speech_duration",
    "call_windows",
    "call_flagged_share",
    "call_highest_score",
    "call_timeline",
]

print("Translation test:\n")

for key in test_keys:
    print(
        f"{key}:",
        "FOUND" if key in i18n.TRANSLATIONS else "MISSING",
        "| English:", i18n.t(key, "English"),
        "| Hindi:", i18n.t(key, "Hindi"),
    )

Translation test:

mode_label: FOUND | English: Analysis Mode | Hindi: विश्लेषण मोड
call_analysis_mode: FOUND | English: Call Analysis | Hindi: कॉल विश्लेषण
call_summary: FOUND | English: Call Analysis Summary | Hindi: कॉल विश्लेषण सारांश
call_speech_duration: FOUND | English: Speech-active duration | Hindi: सक्रिय भाषण अवधि
call_windows: FOUND | English: Windows analyzed | Hindi: विश्लेषित विंडो
call_flagged_share: FOUND | English: Flagged share | Hindi: चिह्नित हिस्सा
call_highest_score: FOUND | English: Highest AI likelihood | Hindi: सबसे अधिक AI संभावना
call_timeline: FOUND | English: AI Likelihood Timeline | Hindi: AI संभावना समयरेखा


In [ ]:
import ast
import i18n

source = open("app.py", encoding="utf-8").read()
tree = ast.parse(source)

function = next(
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "render_call_analysis"
)

keys = sorted({
    node.args[0].value
    for node in ast.walk(function)
    if isinstance(node, ast.Call)
    and isinstance(node.func, ast.Name)
    and node.func.id == "t"
    and node.args
    and isinstance(node.args[0], ast.Constant)
    and isinstance(node.args[0].value, str)
})

print("CALL ANALYSIS TRANSLATION AUDIT")
print("-" * 40)

for key in keys:
    status = "FOUND" if key in i18n.TRANSLATIONS else "MISSING"
    print(f"{status}: {key}")

print("-" * 40)
print("Total keys checked:", len(keys))

CALL ANALYSIS TRANSLATION AUDIT
----------------------------------------
FOUND: call_analysis_mode
FOUND: call_analyze_button
FOUND: call_analyzing
FOUND: call_csv_button
FOUND: call_end
FOUND: call_flagged_ranges
FOUND: call_flagged_share
FOUND: call_highest_score
FOUND: call_highest_time
FOUND: call_limit_note
FOUND: call_no_flagged
FOUND: call_speech_duration
FOUND: call_start
FOUND: call_summary
FOUND: call_timeline
FOUND: call_upload_help
FOUND: call_upload_label
FOUND: call_windows
FOUND: workspace_kicker
----------------------------------------
Total keys checked: 19


In [ ]:
!grep -n -E "from i18n|import i18n|def current_lang|language|selected_language" app.py | head -60

26:import i18n
28:from i18n import t
51:    "ui_language": "English",
72:def current_lang() -> str:
73:    return st.session_state.get("ui_language", "English")
123:    if isinstance(exc, ValueError) and str(exc).startswith("Unsupported language hint"):
130:    brand_col, language_col = st.columns([5.3, 1.15], gap="medium")
151:    with language_col:
153:            t("interface_language_label", lang),
155:            key="ui_language",
165:            t("chip_languages", lang),


In [ ]:
!sed -n '65,85p' app.py
!sed -n '125,165p' app.py


}.items():
    st.session_state.setdefault(key, default)


ACCEPTED_TYPES = ["wav", "mp3", "flac", "ogg", "m4a"]


def current_lang() -> str:
    return st.session_state.get("ui_language", "English")


def format_bytes(size: int) -> str:
    if size < 1024:
        return f"{size} B"
    if size < 1024 * 1024:
        return f"{size / 1024:.1f} KB"
    return f"{size / (1024 * 1024):.1f} MB"


def clear_analysis_result() -> None:
    st.session_state.result = None
    LOGGER.exception("Audio analysis failed", exc_info=exc)
    return t("analysis_failed_body", lang)


def render_header(lang: str) -> None:
    brand_col, language_col = st.columns([5.3, 1.15], gap="medium")
    with brand_col:
        st.markdown(
            f"""
            <div class="vg-topbar">
              <div>
                <div class="vg-brand">
                  <span class="vg-brand-mark" aria-hidden="true">
                    <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.8">
  

In [ ]:
%cd /content/indic-fake-voice-detection

!grep -n -E "def analyze_call|def build_analysis_windows|def score_waveform|highest_score_timestamp|window_scores|p_fake" call_analysis.py

/content/indic-fake-voice-detection
116:def build_analysis_windows(
163:    p_fake: float
176:    highest_p_fake: float
177:    highest_score_timestamp: float
178:    window_scores: List[WindowScore]
183:    window_scores: List[WindowScore],
187:        (score for score in window_scores if score.verdict == "fake"),
212:def analyze_call(
279:    window_scores = []
284:        raw_logit, p_fake = model_pipeline.score_waveform(
289:        verdict = model_pipeline.verdict_from_p_fake(p_fake)
291:        window_scores.append(
296:                p_fake=p_fake,
305:        score.verdict == "fake" for score in window_scores
308:    flagged_share = flagged_windows / len(window_scores)
310:    highest_score = max(window_scores, key=lambda score: score.p_fake)
320:        windows_analysed=len(window_scores),
323:        highest_p_fake=highest_score.p_fake,
324:        highest_score_timestamp=highest_score.start_seconds,
325:        window_scores=window_scores,
326:        flagged_ranges=merge_f

In [ ]:
%cd /content/indic-fake-voice-detection

print("===== CALL ANALYSIS SCORING =====")
!sed -n '212,330p' call_analysis.py

print("\n===== MODEL SCORING FUNCTION =====")
!grep -R -n -A35 -B10 "def score_waveform" .

/content/indic-fake-voice-detection
===== CALL ANALYSIS SCORING =====
def analyze_call(
    audio_bytes: bytes,
    checkpoint_dir: str | None = None,
    window_seconds: float = 6.0,
    hop_seconds: float = 3.0,
    progress_callback=None,
) -> CallAnalysisReport:
    """Analyse a long phone call using the existing Voice Guard model."""
    import model_pipeline

    from voice_guard_core import (
        CallInputPolicy,
        validate_encoded_audio_size,
    )

    validate_encoded_audio_size(
        audio_bytes,
        policy=CallInputPolicy(),
    )

    waveform, original_sample_rate = model_pipeline.decode_audio_bytes(
        audio_bytes
    )

    waveform = np.asarray(waveform, dtype=np.float32).reshape(-1)

    if waveform.size == 0:
        raise ValueError("The decoded recording is empty.")
    if original_sample_rate <= 0:
        raise ValueError("Invalid audio sample rate.")
    if not np.isfinite(waveform).all():
        raise ValueError("The recording contains no

In [ ]:
%cd /content/indic-fake-voice-detection

print("===== SPEECH REGION DETECTION =====")
!sed -n '1,115p' call_analysis.py

print("\n===== WINDOW CREATION =====")
!sed -n '116,160p' call_analysis.py

print("\n===== AUDIO FEATURE EXTRACTION =====")
!grep -n -A65 -B5 "def extract_features" model_pipeline.py

/content/indic-fake-voice-detection
===== SPEECH REGION DETECTION =====
"""Long-call speech detection and window segmentation for Voice Guard."""

from __future__ import annotations

from dataclasses import dataclass
from typing import List

import numpy as np


@dataclass(frozen=True)
class SpeechRegion:
    """Speech-active interval in the original recording timeline."""

    start_seconds: float
    end_seconds: float


@dataclass(frozen=True)
class AnalysisWindow:
    """A model-analysis window with its original recording timestamps."""

    start_seconds: float
    end_seconds: float
    start_sample: int
    end_sample: int


def detect_speech_regions(
    waveform: np.ndarray,
    sample_rate: int,
    frame_duration: float = 0.025,
    hop_duration: float = 0.010,
    merge_gap_seconds: float = 0.30,
    min_region_seconds: float = 0.15,
    energy_threshold: float | None = None,
) -> List[SpeechRegion]:
    """Find speech-active regions using a lightweight adaptive energy gate

In [ ]:
%cd /content/indic-fake-voice-detection

print("===== SAMPLE RATE AND RESAMPLING =====")
!grep -n -A45 -B5 "def ensure_target_sample_rate" model_pipeline.py

print("\n===== MODEL FRONTEND CONFIGURATION =====")
!grep -n -E "TARGET_SAMPLE_RATE|CHECKPOINT_DIR|def load_frontend|AutoFeatureExtractor|Wav2Vec2FeatureExtractor" model_pipeline.py | head -60

/content/indic-fake-voice-detection
===== SAMPLE RATE AND RESAMPLING =====
455-
456-    decoded = audio.get_all_samples()
457-    return decoded.data.numpy().squeeze(), decoded.sample_rate
458-
459-
460:def ensure_target_sample_rate(
461-    waveform: np.ndarray,
462-    sample_rate: int,
463-    target_sample_rate: int = TARGET_SAMPLE_RATE,
464-) -> np.ndarray:
465-    """
466-    Guard against a silent correctness bug: XLS-R expects 16 kHz, and feeding it
467-    another rate degrades features without raising anything.
468-    """
469-    if sample_rate == target_sample_rate:
470-        return waveform
471-
472-    import torchaudio
473-
474-    resampled = torchaudio.functional.resample(
475-        torch.from_numpy(waveform).float(),
476-        orig_freq=sample_rate,
477-        new_freq=target_sample_rate,
478-    )
479-    return resampled.numpy()
480-
481-
482-# ============================================================================
483-# SECTION 5 — Frozen frontend
484-#

In [ ]:
%cd /content/indic-fake-voice-detection

print("===== FRONTEND LOADING =====")
!sed -n '491,532p' model_pipeline.py

print("\n===== AUDIO DECODING =====")
!sed -n '420,460p' model_pipeline.py

print("\n===== INFERENCE TEMPERATURE =====")
!grep -n -A25 -B15 "temperature" model_pipeline.py | tail -100

/content/indic-fake-voice-detection
===== FRONTEND LOADING =====
def load_frontend(
    model_name: str = FRONTEND_MODEL_NAME,
    device: Optional[str] = None,
) -> Tuple[Any, nn.Module]:
    """
    Load and freeze the wav2vec2-XLS-R-300M frontend. Cached after first call.

    Returns (feature_extractor, frozen_frontend). Verified to freeze to
    0 trainable parameters.
    """
    from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model

    device = device or DEVICE

    cache_key = (model_name, str(device))
    if cache_key in _FRONTEND_CACHE:
        return _FRONTEND_CACHE[cache_key]

    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
    frontend = Wav2Vec2Model.from_pretrained(model_name)  #"Load an already-trained version of this model."

    for param in frontend.parameters(): #Go through all the parameters of wav2vec2 and don't calculate/update gradients for them.
        param.requires_grad = False
    frontend.eval()  #puts wav2vec2 into

In [ ]:
%cd /content/indic-fake-voice-detection

print("===== INFERENCE BUNDLE LOADING =====")
!sed -n '2920,3035p' model_pipeline.py

print("\n===== CHECKPOINT FILES =====")
!find checkpoints -maxdepth 2 -type f -printf '%p\n' | head -40

/content/indic-fake-voice-detection
===== INFERENCE BUNDLE LOADING =====
            waveform_array = waveform_array.reshape((-1, audio.channels)).mean(axis=1)

        max_val = 1 << (8 * audio.sample_width - 1)
        waveform_array = waveform_array / max_val

        return waveform_array, audio.frame_rate
    except Exception as exc:  # noqa: BLE001
        last_error = exc

    raise RuntimeError(
        "Could not decode the supplied audio bytes with soundfile, torchaudio, or pydub. "
        f"Last error: {type(last_error).__name__}: {last_error}"
    )


@dataclass
class InferenceBundle:
    """Everything needed to score a clip, loaded once and cached."""

    head: nn.Module
    temperature: float
    checkpoint_path: str
    checkpoint_epoch: int
    config: Dict[str, Any]
    summary_path: Optional[str]
    calibration_ece_before: Optional[float]
    calibration_ece_after: Optional[float]
    validation_eer_threshold_bonafide: Optional[float]


# Keyed cache so tests and s

In [ ]:
%cd /content/indic-fake-voice-detection

import json

with open("checkpoints/step6_summary.json", "r") as f:
    summary = json.load(f)

print("Temperature:", summary.get("temperature"))
print("Calibration ECE before:", summary.get("ece_before"))
print("Calibration ECE after:", summary.get("ece_after"))
print("Validation EER:", summary.get("val_eer"))
print("Validation threshold:", summary.get("val_eer_threshold_bonafide_score"))

/content/indic-fake-voice-detection
Temperature: 1.260030746459961
Calibration ECE before: 0.016343042426937262
Calibration ECE after: 0.018292837797954584
Validation EER: 0.00996784565916397
Validation threshold: None


In [ ]:
from google.colab import files
import call_analysis

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload exactly one podcast audio file.")

filename, audio_bytes = next(iter(uploaded.items()))

print("Analysing:", filename)

report = call_analysis.analyze_call(audio_bytes)

print("\n===== PODCAST ANALYSIS =====")
print("Duration:", report.duration_seconds)
print("Speech-active duration:", report.speech_duration_seconds)
print("Windows analysed:", report.windows_analysed)
print("Highest AI likelihood:", f"{report.highest_p_fake:.2%}")
print("Highest score timestamp:", report.highest_score_timestamp)

print("\n===== INDIVIDUAL WINDOW SCORES =====")
for score in report.window_scores:
    print(
        f"{score.start_seconds:.2f}s - {score.end_seconds:.2f}s | "
        f"AI likelihood: {score.p_fake:.2%} | "
        f"Verdict: {score.verdict} | "
        f"Raw logit: {score.raw_logit:.4f}"
    )

Saving WhatsApp Audio 2026-11-16 at 7.27.32 PM.mp3 to WhatsApp Audio 2026-11-16 at 7.27.32 PM.mp3
Analysing: WhatsApp Audio 2026-11-16 at 7.27.32 PM.mp3
Inference bundle loaded: /content/indic-fake-voice-detection/checkpoints/best_model.pt (epoch 11), temperature=1.2600


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== PODCAST ANALYSIS =====
Duration: 31.872
Speech-active duration: 27.330000000000002
Windows analysed: 8
Highest AI likelihood: 86.54%
Highest score timestamp: 24.95

===== INDIVIDUAL WINDOW SCORES =====
1.91s - 7.91s | AI likelihood: 0.19% | Verdict: real | Raw logit: -7.9179
4.91s - 10.91s | AI likelihood: 0.80% | Verdict: real | Raw logit: -6.0692
7.91s - 11.36s | AI likelihood: 0.23% | Verdict: real | Raw logit: -7.6560
11.87s - 14.87s | AI likelihood: 12.71% | Verdict: real | Raw logit: -2.4283
17.03s - 22.75s | AI likelihood: 1.50% | Verdict: real | Raw logit: -5.2757
20.03s - 22.75s | AI likelihood: 35.74% | Verdict: uncertain | Raw logit: -0.7394
24.95s - 27.68s | AI likelihood: 86.54% | Verdict: fake | Raw logit: 2.3446
27.98s - 31.86s | AI likelihood: 0.54% | Verdict: real | Raw logit: -6.5612


In [ ]:
%cd /content/indic-fake-voice-detection

import numpy as np
import model_pipeline
import call_analysis

filename = "WhatsApp Audio 2026-11-16 at 7.27.32 PM.mp3"

with open(filename, "rb") as f:
    audio_bytes = f.read()

waveform, original_sr = model_pipeline.decode_audio_bytes(audio_bytes)
waveform = np.asarray(waveform, dtype=np.float32).reshape(-1)

waveform = model_pipeline.ensure_target_sample_rate(
    waveform, original_sr, model_pipeline.TARGET_SAMPLE_RATE
)

sr = model_pipeline.TARGET_SAMPLE_RATE
regions = call_analysis.detect_speech_regions(waveform, sr)
windows = call_analysis.build_analysis_windows(regions, sr)

print("===== DETECTED SPEECH REGIONS =====")
for i, region in enumerate(regions, 1):
    print(
        f"{i}. {region.start_seconds:.2f}s - "
        f"{region.end_seconds:.2f}s "
        f"({region.end_seconds - region.start_seconds:.2f}s)"
    )

print("\n===== WINDOW AUDIO LEVELS =====")
for i, window in enumerate(windows, 1):
    segment = waveform[window.start_sample:window.end_sample]
    rms = float(np.sqrt(np.mean(segment ** 2)))
    peak = float(np.max(np.abs(segment)))

    print(
        f"{i}. {window.start_seconds:.2f}s - {window.end_seconds:.2f}s | "
        f"Duration: {len(segment)/sr:.2f}s | "
        f"RMS: {rms:.5f} | Peak: {peak:.5f}"
    )

/content/indic-fake-voice-detection
===== DETECTED SPEECH REGIONS =====
1. 0.02s - 1.16s (1.14s)
2. 1.91s - 11.36s (9.45s)
3. 11.87s - 14.87s (3.00s)
4. 15.39s - 16.48s (1.09s)
5. 17.03s - 22.75s (5.71s)
6. 23.82s - 24.16s (0.34s)
7. 24.95s - 27.68s (2.73s)
8. 27.98s - 31.86s (3.88s)

===== WINDOW AUDIO LEVELS =====
1. 1.91s - 7.91s | Duration: 6.00s | RMS: 0.06280 | Peak: 0.36265
2. 4.91s - 10.91s | Duration: 6.00s | RMS: 0.06531 | Peak: 0.40554
3. 7.91s - 11.36s | Duration: 3.44s | RMS: 0.06672 | Peak: 0.40554
4. 11.87s - 14.87s | Duration: 3.00s | RMS: 0.07464 | Peak: 0.37265
5. 17.03s - 22.75s | Duration: 5.71s | RMS: 0.07809 | Peak: 0.52105
6. 20.03s - 22.75s | Duration: 2.71s | RMS: 0.07800 | Peak: 0.52105
7. 24.95s - 27.68s | Duration: 2.73s | RMS: 0.06850 | Peak: 0.40018
8. 27.98s - 31.86s | Duration: 3.88s | RMS: 0.08214 | Peak: 0.41569


In [ ]:
%cd /content/indic-fake-voice-detection

import numpy as np
import model_pipeline

sr = model_pipeline.TARGET_SAMPLE_RATE
bundle = model_pipeline.load_inference_bundle()

def diagnostic_score(start, end, label):
    segment = waveform[int(start * sr):int(end * sr)].copy()

    raw_logit, p_fake = model_pipeline.score_waveform(
        segment, sr, bundle=bundle
    )

    print(
        f"{label}: {start:.2f}s-{end:.2f}s | "
        f"{len(segment)/sr:.2f}s | "
        f"AI likelihood: {p_fake:.2%} | "
        f"Raw logit: {raw_logit:.4f}"
    )

print("===== CONTEXT SENSITIVITY TEST =====")

diagnostic_score(24.95, 27.68, "Original flagged segment")
diagnostic_score(22.95, 28.95, "Six-second context")
diagnostic_score(23.82, 29.82, "Context from previous speech boundary")
diagnostic_score(24.00, 30.00, "Shifted six-second context")

/content/indic-fake-voice-detection
===== CONTEXT SENSITIVITY TEST =====
Original flagged segment: 24.95s-27.68s | 2.73s | AI likelihood: 86.60% | Raw logit: 2.3512
Six-second context: 22.95s-28.95s | 6.00s | AI likelihood: 0.60% | Raw logit: -6.4350
Context from previous speech boundary: 23.82s-29.82s | 6.00s | AI likelihood: 13.15% | Raw logit: -2.3784
Shifted six-second context: 24.00s-30.00s | 6.00s | AI likelihood: 18.99% | Raw logit: -1.8280


In [ ]:
%cd /content/indic-fake-voice-detection

print("===== VERDICT LOGIC =====")
!grep -n -A45 -B10 "def verdict_from_p_fake" model_pipeline.py

print("\n===== MINIMUM AUDIO DURATION RULES =====")
!grep -n -E "min_duration|min_seconds|minimum.*duration|MIN.*DURATION" model_pipeline.py call_analysis.py

/content/indic-fake-voice-detection
===== VERDICT LOGIC =====
2849-
2850-# --- Frozen decision thresholds -------------------------------------------
2851-# Compatibility aliases for callers that imported these names. The canonical
2852-# policy and its provenance now live in voice_guard_core.py. The v1 values are
2853-# preserved until a v2 validation run can derive replacement operating points;
2854-# the already-observed v1 test set must never be used to tune them.
2855-VERDICT_REAL_MAX_P_FAKE = DEFAULT_DECISION_POLICY.real_max_p_fake
2856-VERDICT_FAKE_MIN_P_FAKE = DEFAULT_DECISION_POLICY.fake_min_p_fake
2857-
2858-
2859:def verdict_from_p_fake(
2860-    p_fake: float,
2861-    real_max: float = VERDICT_REAL_MAX_P_FAKE,
2862-    fake_min: float = VERDICT_FAKE_MIN_P_FAKE,
2863-) -> str:
2864-    if (
2865-        real_max == DEFAULT_DECISION_POLICY.real_max_p_fake
2866-        and fake_min == DEFAULT_DECISION_POLICY.fake_min_p_fake
2867-    ):
2868-        return verdict_from_probabi

In [ ]:
%cd /content/indic-fake-voice-detection

print("===== DECISION POLICY =====")
!grep -n -A90 -B15 "class DecisionPolicy\|DEFAULT_DECISION_POLICY\|def verdict_from_probability" voice_guard_core.py

print("\n===== AUDIO INPUT POLICY =====")
!grep -n -A65 -B10 "class CallInputPolicy\|DEFAULT_AUDIO_POLICY" voice_guard_core.py

/content/indic-fake-voice-detection
===== DECISION POLICY =====
48-    min_rms: float = 1e-4
49-    min_peak: float = 1e-3
50-    min_active_seconds: float = 0.45
51-
52-
53-@dataclass(frozen=True)
54-class CallInputPolicy:
55-    """Input limits for long phone-call recordings."""
56-
57-    max_bytes: int = 200 * 1024 * 1024
58-    min_duration_seconds: float = 1.0
59-    max_duration_seconds: float = 900.0
60-
61-
62-@dataclass(frozen=True)
63:class DecisionPolicy:
64-    """Three-way abstention policy operating on calibrated P(fake)."""
65-
66-    policy_id: str = "legacy-symmetric-abstention-v1"
67-    real_max_p_fake: float = 0.35
68-    fake_min_p_fake: float = 0.65
69-    provenance: str = (
70-        "Legacy symmetric abstention band; not selected on the frozen test set. "
71-        "Replace only from validation data under the v2 protocol."
72-    )
73-
74-    def __post_init__(self) -> None:
75-        if not 0.0 <= self.real_max_p_fake < self.fake_min_p_fake <= 1.0:
76-    

In [ ]:
%cd /content/indic-fake-voice-detection

import inspect
import voice_guard_core as v

print("DECISION POLICY:")
print(v.DEFAULT_DECISION_POLICY)

print("\nAUDIO INPUT POLICY:")
print(v.DEFAULT_AUDIO_POLICY)

print("\nVERDICT FUNCTION:")
print(inspect.getsource(v.verdict_from_probability))

/content/indic-fake-voice-detection
DECISION POLICY:
DecisionPolicy(policy_id='legacy-symmetric-abstention-v1', real_max_p_fake=0.35, fake_min_p_fake=0.65, provenance='Legacy symmetric abstention band; not selected on the frozen test set. Replace only from validation data under the v2 protocol.')

AUDIO INPUT POLICY:
AudioInputPolicy(max_bytes=26214400, min_duration_seconds=1.0, max_duration_seconds=30.0, min_rms=0.0001, min_peak=0.001, min_active_seconds=0.45)

VERDICT FUNCTION:
def verdict_from_probability(
    p_fake: float,
    policy: DecisionPolicy = DEFAULT_DECISION_POLICY,
) -> str:
    if not math.isfinite(p_fake) or not 0.0 <= p_fake <= 1.0:
        raise ValueError("p_fake must be a finite probability in [0, 1].")
    if p_fake < policy.real_max_p_fake:
        return "real"
    if p_fake > policy.fake_min_p_fake:
        return "fake"
    return "uncertain"



In [ ]:
%cd /content/indic-fake-voice-detection

import inspect
import call_analysis

print("WINDOW CREATION:")
print(inspect.getsource(call_analysis.build_analysis_windows))

print("\nSCORING LOGIC IN analyze_call():")
source = inspect.getsource(call_analysis.analyze_call).splitlines()

for i, line in enumerate(source):
    if any(term in line for term in [
        "build_analysis_windows",
        "score_waveform",
        "window.start",
        "window.end",
        "window_scores.append"
    ]):
        start = max(0, i - 3)
        end = min(len(source), i + 5)
        print("\n".join(source[start:end]))
        print("---")

/content/indic-fake-voice-detection
WINDOW CREATION:
def build_analysis_windows(
    regions: List[SpeechRegion],
    sample_rate: int,
    window_seconds: float = 6.0,
    hop_seconds: float = 3.0,
    min_trailing_seconds: float = 2.0,
) -> List[AnalysisWindow]:
    """Create overlapping analysis windows within speech regions."""
    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive.")
    if window_seconds <= 0 or hop_seconds <= 0 or min_trailing_seconds < 0:
        raise ValueError("Window, hop, and trailing durations must be valid.")

    windows = []
    for region in regions:
        if region.start_seconds < 0 or region.end_seconds < region.start_seconds:
            raise ValueError("Invalid speech region.")

        start = region.start_seconds
        while start < region.end_seconds:
            duration = min(window_seconds, region.end_seconds - start)
            if duration < min_trailing_seconds:
                break

            start_sample

In [ ]:
import numpy as np

print("SHORT-WINDOW CONTEXT TEST")
print("-" * 55)

for i, window in enumerate(report.window_scores):
    duration = window.end_seconds - window.start_seconds

    if duration >= 5.99:
        continue

    center = (window.start_seconds + window.end_seconds) / 2
    start = max(0.0, min(center - 3.0, len(waveform) / 16000 - 6.0))
    end = min(start + 6.0, len(waveform) / 16000)

    segment = waveform[int(start * 16000):int(end * 16000)]

    _, context_score = model_pipeline.score_waveform(
        segment, 16000, bundle=bundle
    )

    print(
        f"{window.start_seconds:.2f}-{window.end_seconds:.2f}s "
        f"({duration:.2f}s): "
        f"original={window.p_fake * 100:.2f}% | "
        f"context={context_score * 100:.2f}%"
    )

SHORT-WINDOW CONTEXT TEST
-------------------------------------------------------
7.91-11.36s (3.45s): original=0.23% | context=0.26%
11.87-14.87s (3.00s): original=12.71% | context=0.17%
17.03-22.75s (5.71s): original=1.50% | context=1.94%
20.03-22.75s (2.71s): original=35.74% | context=0.20%
24.95-27.68s (2.73s): original=86.54% | context=0.31%
27.98-31.86s (3.88s): original=0.54% | context=10.57%


In [ ]:
print("ALL WINDOWS: ORIGINAL VS 6-SECOND CONTEXT")
print("-" * 65)

duration_total = len(waveform) / 16000

for w in report.window_scores:
    center = (w.start_seconds + w.end_seconds) / 2
    start = max(0.0, min(center - 3.0, duration_total - 6.0))
    end = min(start + 6.0, duration_total)

    segment = waveform[int(start * 16000):int(end * 16000)]
    _, score = model_pipeline.score_waveform(
        segment, 16000, bundle=bundle
    )

    print(
        f"{w.start_seconds:5.2f}-{w.end_seconds:5.2f}s | "
        f"{(w.end_seconds-w.start_seconds):4.2f}s | "
        f"Original: {w.p_fake*100:6.2f}% | "
        f"Context: {score*100:6.2f}%"
    )

ALL WINDOWS: ORIGINAL VS 6-SECOND CONTEXT
-----------------------------------------------------------------
 1.91- 7.91s | 6.00s | Original:   0.19% | Context:   0.19%
 4.91-10.91s | 6.00s | Original:   0.80% | Context:   0.80%
 7.91-11.36s | 3.45s | Original:   0.23% | Context:   0.26%
11.87-14.87s | 3.00s | Original:  12.71% | Context:   0.17%
17.03-22.75s | 5.71s | Original:   1.50% | Context:   1.94%
20.03-22.75s | 2.71s | Original:  35.74% | Context:   0.20%
24.95-27.68s | 2.73s | Original:  86.54% | Context:   0.31%
27.98-31.86s | 3.88s | Original:   0.54% | Context:  10.57%


In [ ]:
%cd /content/indic-fake-voice-detection

import inspect
import subprocess
import call_analysis

print("GIT STATUS")
print(subprocess.run(
    ["git", "status", "--short"],
    capture_output=True, text=True
).stdout or "No uncommitted changes")

print("\nANALYZE_CALL IMPLEMENTATION")
print(inspect.getsource(call_analysis.analyze_call))

print("\nWINDOW CREATION DEFAULTS")
print(inspect.signature(call_analysis.build_analysis_windows))

/content/indic-fake-voice-detection
GIT STATUS
 M app.py
?? "WhatsApp Audio 2026-11-16 at 7.27.32 PM.mp3"
?? app.py.before_call_layout.py


ANALYZE_CALL IMPLEMENTATION
def analyze_call(
    audio_bytes: bytes,
    checkpoint_dir: str | None = None,
    window_seconds: float = 6.0,
    hop_seconds: float = 3.0,
    progress_callback=None,
) -> CallAnalysisReport:
    """Analyse a long phone call using the existing Voice Guard model."""
    import model_pipeline

    from voice_guard_core import (
        CallInputPolicy,
        validate_encoded_audio_size,
    )

    validate_encoded_audio_size(
        audio_bytes,
        policy=CallInputPolicy(),
    )

    waveform, original_sample_rate = model_pipeline.decode_audio_bytes(
        audio_bytes
    )

    waveform = np.asarray(waveform, dtype=np.float32).reshape(-1)

    if waveform.size == 0:
        raise ValueError("The decoded recording is empty.")
    if original_sample_rate <= 0:
        raise ValueError("Invalid audio sample r

In [ ]:
%cd /content/indic-fake-voice-detection

import inspect
import call_analysis

print("WINDOW SCORE STRUCTURE:")
print(inspect.getsource(call_analysis.WindowScore))

print("\nFLAGGED RANGE MERGING:")
print(inspect.getsource(call_analysis.merge_flagged_ranges))

/content/indic-fake-voice-detection
WINDOW SCORE STRUCTURE:
@dataclass(frozen=True)
class WindowScore:
    """Fake-voice score for one analysis window."""

    start_seconds: float
    end_seconds: float
    raw_logit: float
    p_fake: float
    verdict: str


FLAGGED RANGE MERGING:
def merge_flagged_ranges(
    window_scores: List[WindowScore],
) -> List[SpeechRegion]:
    """Merge overlapping or adjacent windows classified as fake."""
    flagged = sorted(
        (score for score in window_scores if score.verdict == "fake"),
        key=lambda score: score.start_seconds,
    )

    if not flagged:
        return []

    merged = []
    current_start = flagged[0].start_seconds
    current_end = flagged[0].end_seconds

    for score in flagged[1:]:
        if score.start_seconds <= current_end:
            current_end = max(current_end, score.end_seconds)
        else:
            merged.append(
                SpeechRegion(current_start, current_end)
            )
            curren

In [ ]:
%cd /content/indic-fake-voice-detection

from pathlib import Path
import py_compile
import importlib
import call_analysis

source_path = Path("call_analysis.py")
backup_path = Path("call_analysis.py.before_context_windows.py")
audio_path = Path("WhatsApp Audio 2026-11-16 at 7.27.32 PM.mp3")

original_source = source_path.read_text()

old_scoring = '''    for index, window in enumerate(windows):
        segment = waveform[window.start_sample:window.end_sample]

        raw_logit, p_fake = model_pipeline.score_waveform(
            segment,
            sample_rate,
            bundle=bundle,
        )
        verdict = model_pipeline.verdict_from_p_fake(p_fake)

        window_scores.append(
            WindowScore(
                start_seconds=window.start_seconds,
                end_seconds=window.end_seconds,
                raw_logit=raw_logit,
                p_fake=p_fake,
                verdict=verdict,
            )
        )
'''

new_scoring = '''    for index, window in enumerate(windows):
        segment_start = window.start_seconds
        segment_end = window.end_seconds
        segment_duration = segment_end - segment_start

        # Expand short Call Analysis windows using surrounding audio.
        # Single Clip and the shared model pipeline are not modified.
        if segment_duration < window_seconds and duration_seconds >= window_seconds:
            center = (segment_start + segment_end) / 2.0
            segment_start = max(
                0.0,
                min(
                    center - window_seconds / 2.0,
                    duration_seconds - window_seconds,
                ),
            )
            segment_end = segment_start + window_seconds

        start_sample = int(round(segment_start * sample_rate))
        end_sample = int(round(segment_end * sample_rate))
        segment = waveform[start_sample:end_sample]

        raw_logit, p_fake = model_pipeline.score_waveform(
            segment,
            sample_rate,
            bundle=bundle,
        )
        verdict = model_pipeline.verdict_from_p_fake(p_fake)

        window_scores.append(
            WindowScore(
                start_seconds=segment_start,
                end_seconds=segment_end,
                raw_logit=raw_logit,
                p_fake=p_fake,
                verdict=verdict,
            )
        )
'''

if not audio_path.exists():
    raise FileNotFoundError(f"Podcast test file not found: {audio_path}")

if old_scoring not in original_source:
    raise RuntimeError("Expected scoring code was not found. No changes made.")

if not backup_path.exists():
    backup_path.write_text(original_source)

try:
    updated_source = original_source.replace(old_scoring, new_scoring, 1)
    source_path.write_text(updated_source)

    py_compile.compile(str(source_path), doraise=True)

    importlib.reload(call_analysis)

    audio_bytes = audio_path.read_bytes()
    test_report = call_analysis.analyze_call(audio_bytes)

    print("CHANGE AND PODCAST TEST COMPLETED")
    print("--------------------------------")
    print(f"Duration: {test_report.duration_seconds:.2f}s")
    print(f"Windows analysed: {test_report.windows_analysed}")
    print(f"Flagged windows: {test_report.flagged_windows}")
    print(f"Flagged share: {test_report.flagged_share * 100:.2f}%")
    print(f"Highest AI likelihood: {test_report.highest_p_fake * 100:.2f}%")
    print(f"Highest score timestamp: {test_report.highest_score_timestamp:.2f}s")

    print("\nUPDATED WINDOW SCORES")
    for w in test_report.window_scores:
        print(
            f"{w.start_seconds:.2f}-{w.end_seconds:.2f}s | "
            f"{(w.end_seconds-w.start_seconds):.2f}s | "
            f"{w.p_fake*100:.2f}% | {w.verdict}"
        )

    print("\nBackup:", backup_path)
    print("Only call_analysis.py was edited.")

except Exception:
    source_path.write_text(original_source)
    importlib.reload(call_analysis)
    print("Test failed. Original call_analysis.py has been restored.")
    raise

/content/indic-fake-voice-detection
CHANGE AND PODCAST TEST COMPLETED
--------------------------------
Duration: 31.87s
Windows analysed: 8
Flagged windows: 0
Flagged share: 0.00%
Highest AI likelihood: 10.57%
Highest score timestamp: 25.87s

UPDATED WINDOW SCORES
1.91-7.91s | 6.00s | 0.19% | real
4.91-10.91s | 6.00s | 0.80% | real
6.63-12.63s | 6.00s | 0.26% | real
10.37-16.37s | 6.00s | 0.17% | real
16.89-22.89s | 6.00s | 1.94% | real
18.39-24.39s | 6.00s | 0.20% | real
23.31-29.31s | 6.00s | 0.31% | real
25.87-31.87s | 6.00s | 10.57% | real

Backup: call_analysis.py.before_context_windows.py
Only call_analysis.py was edited.


In [ ]:
%cd /content/indic-fake-voice-detection

import py_compile
import subprocess

py_compile.compile("call_analysis.py", doraise=True)
py_compile.compile("app.py", doraise=True)

print("Python compilation: PASSED")

print("\nChanged files:")
print(subprocess.run(
    ["git", "status", "--short"],
    capture_output=True,
    text=True
).stdout)

print("\nSingle Clip shared pipeline files:")
for filename in ["model_pipeline.py", "voice_guard_core.py"]:
    result = subprocess.run(
        ["git", "diff", "--", filename],
        capture_output=True,
        text=True
    )
    print(filename, "UNCHANGED" if not result.stdout else "MODIFIED")

/content/indic-fake-voice-detection
Python compilation: PASSED

Changed files:
 M app.py
 M call_analysis.py
?? "WhatsApp Audio 2026-11-16 at 7.27.32 PM.mp3"
?? app.py.before_call_layout.py
?? call_analysis.py.before_context_windows.py


Single Clip shared pipeline files:
model_pipeline.py UNCHANGED
voice_guard_core.py UNCHANGED


In [ ]:
import requests

try:
    response = requests.get("http://127.0.0.1:8501/_stcore/health", timeout=5)
    print("Streamlit status:", response.status_code)
    print("Response:", response.text)
except Exception as e:
    print("Streamlit is not responding:", e)

Streamlit status: 200
Response: ok


In [ ]:
from google.colab import output
print(output.eval_js("google.colab.kernel.proxyPort(8501)"))

https://8501-m-s-use4a8-16g00xd535zsg-a.us-east4.prod.r.colab.dev


In [ ]:
!pgrep -af "streamlit run app.py"

1381 python3 -m streamlit run app.py --server.headless=true --server.port=8501 --server.enableCORS=false --server.enableXsrfProtection=false


In [ ]:
!kill 1381

In [ ]:
!pgrep -af "streamlit run app.py"

In [ ]:
%cd /content/indic-fake-voice-detection
!streamlit run app.py --server.headless=true --server.port=8501 --server.enableCORS=false --server.enableXsrfProtection=false

/content/indic-fake-voice-detection


2026-11-18 06:30:19.731 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.182.242.201:8501

Inference bundle loaded: /content/indic-fake-voice-detection/checkpoints/best_model.pt (epoch 11), temperature=1.2600
Loading weights: 100% 422/422 [00:00<00:00, 18426.33it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loa

In [ ]:
from google.colab import output
print(output.eval_js("google.colab.kernel.proxyPort(8501)"))

https://8501-m-s-use4a8-16g00xd535zsg-a.us-east4.prod.r.colab.dev


In [ ]:
import requests

try:
    r = requests.get("http://127.0.0.1:8501/_stcore/health", timeout=10)
    print("Status:", r.status_code)
    print("Response:", r.text)
except Exception as e:
    print("Connection error:", e)

Connection error: HTTPConnectionPool(host='127.0.0.1', port=8501): Max retries exceeded with url: /_stcore/health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7aea0b1d9090>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [ ]:
%cd /content/indic-fake-voice-detection

import subprocess
import time
import requests

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.headless=true",
        "--server.port=8501",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(8)

try:
    response = requests.get(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=10
    )
    print("Streamlit status:", response.status_code)
    print("Response:", response.text)
except Exception as e:
    print("Server did not start:", e)
    print(open("/content/streamlit.log").read()[-3000:])

/content/indic-fake-voice-detection
Streamlit status: 200
Response: ok


In [ ]:
from google.colab import output
print(output.eval_js("google.colab.kernel.proxyPort(8501)"))

https://8501-m-s-use4a8-16g00xd535zsg-a.us-east4.prod.r.colab.dev


In [ ]:
%cd /content/indic-fake-voice-detection

!grep -n -E "def render_report_download|def .*report|reportlab|PDF|Download Window Results" app.py evidence_report.py

/content/indic-fake-voice-detection
app.py:300:def render_robustness_report(report, lang: str) -> None:
app.py:362:def render_report_download(result, filename: str, lang: str) -> None:
app.py:366:        LOGGER.exception("PDF report generation failed", exc_info=exc)
evidence_report.py:10:from reportlab.lib import colors
evidence_report.py:11:from reportlab.lib.enums import TA_CENTER
evidence_report.py:12:from reportlab.lib.pagesizes import A4
evidence_report.py:13:from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
evidence_report.py:14:from reportlab.lib.units import mm
evidence_report.py:15:from reportlab.pdfgen.canvas import Canvas
evidence_report.py:16:from reportlab.platypus import (
evidence_report.py:60:def generate_analysis_report(result: Any, filename: str = "uploaded_audio") -> bytes:
evidence_report.py:61:    """Return a polished PDF record without representing the result as proof."""
evidence_report.py:274:def generate_evidence_report(result: Any, filename:

In [ ]:
%cd /content/indic-fake-voice-detection

print("CURRENT BRANCH:")
!git branch --show-current

print("\nCHANGE SUMMARY:")
!git diff --stat

print("\nUNCOMMITTED FILES:")
!git status --short

print("\nPYTHON VALIDATION:")
!python -m py_compile app.py call_analysis.py
print("Compilation passed.")

/content/indic-fake-voice-detection
CURRENT BRANCH:
feature/long-call-analysis

CHANGE SUMMARY:
 app.py           | 166 +++++++++++++++++++++++++++++++------------------------
 call_analysis.py |  25 ++++++++-
 2 files changed, 117 insertions(+), 74 deletions(-)

UNCOMMITTED FILES:
 M app.py
 M call_analysis.py
?? "WhatsApp Audio 2026-11-16 at 7.27.32 PM.mp3"
?? app.py.before_call_layout.py
?? call_analysis.py.before_context_windows.py

PYTHON VALIDATION:
Compilation passed.


In [ ]:
%cd /content/indic-fake-voice-detection

!git add app.py call_analysis.py
!git status --short

/content/indic-fake-voice-detection
M  app.py
M  call_analysis.py
?? "WhatsApp Audio 2026-11-16 at 7.27.32 PM.mp3"
?? app.py.before_call_layout.py
?? call_analysis.py.before_context_windows.py


In [ ]:
%cd /content/indic-fake-voice-detection

!git diff --cached --stat
!git diff --cached --check

/content/indic-fake-voice-detection
 app.py           | 166 +++++++++++++++++++++++++++++++------------------------
 call_analysis.py |  25 ++++++++-
 2 files changed, 117 insertions(+), 74 deletions(-)


In [ ]:
%cd /content/indic-fake-voice-detection

!git commit -m "Fix Call Analysis layout and short-window scoring"

/content/indic-fake-voice-detection
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@ba9c16b0e368.(none)')


In [ ]:
%cd /content/indic-fake-voice-detection

!git config user.name "Ashitha"
!git config user.email "rajashitha310@gmail.com"

print("Git author configured:")
!git config user.name
!git config user.email

/content/indic-fake-voice-detection
Git author configured:
Ashitha
rajashitha310@gmail.com


In [ ]:
%cd /content/indic-fake-voice-detection

!git commit -m "Fix Call Analysis layout and short-window scoring"

/content/indic-fake-voice-detection
[feature/long-call-analysis 4ec3ecd] Fix Call Analysis layout and short-window scoring
 2 files changed, 117 insertions(+), 74 deletions(-)


In [ ]:
%cd /content/indic-fake-voice-detection

!git push origin feature/long-call-analysis

/content/indic-fake-voice-detection
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
%cd /content/indic-fake-voice-detection

import getpass
import os
import subprocess
import tempfile

token = getpass.getpass("Enter your GitHub Personal Access Token: ")

with tempfile.NamedTemporaryFile(mode="w", delete=False) as f:
    f.write("""#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")
    askpass_path = f.name

os.chmod(askpass_path, 0o700)

env = os.environ.copy()
env["GIT_ASKPASS"] = askpass_path
env["GITHUB_TOKEN"] = token
env["GIT_TERMINAL_PROMPT"] = "0"

try:
    result = subprocess.run(
        ["git", "push", "origin", "feature/long-call-analysis"],
        capture_output=True,
        text=True,
        env=env
    )
    print(result.stdout)
    print(result.stderr)
    print("Push exit code:", result.returncode)
finally:
    os.remove(askpass_path)
    token = None

/content/indic-fake-voice-detection
Enter your GitHub Personal Access Token: ··········

To https://github.com/Shruti-sivaramakrishnan/indic-fake-voice-detection.git
   7723489..4ec3ecd  feature/long-call-analysis -> feature/long-call-analysis

Push exit code: 0


In [1]:
import os

print("Current directory:", os.getcwd())
print("Repository exists:", os.path.exists("/content/indic-fake-voice-detection"))

Current directory: /content
Repository exists: False


In [2]:
!git clone -b feature/long-call-analysis https://github.com/Shruti-sivaramakrishnan/indic-fake-voice-detection.git

Cloning into 'indic-fake-voice-detection'...
remote: Enumerating objects: 371, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 371 (delta 45), reused 79 (delta 27), pack-reused 265 (from 1)
Receiving objects: 100% (371/371), 6.80 MiB | 12.54 MiB/s, done.
Resolving deltas: 100% (154/154), done.


In [3]:
%cd /content/indic-fake-voice-detection

print("BRANCH:")
!git branch --show-current

print("\nLATEST COMMIT:")
!git log -1 --oneline

print("\nPROJECT FILES:")
!ls

/content/indic-fake-voice-detection
BRANCH:
feature/long-call-analysis

LATEST COMMIT:
4ec3ecd (HEAD -> feature/long-call-analysis, origin/feature/long-call-analysis) Fix Call Analysis layout and short-window scoring

PROJECT FILES:
 app.py				  data_pipeline
 call_analysis.py		  docs
 checkpoints			  evaluate_test_split.py
 codec_robustness.py		  evaluation
 confusion_matrix_Assamese.png	  Evaluation.gitkeep
 confusion_matrix_Bengali.png	 'Evidence report'
 confusion_matrix_Bodo.png	  evidence_report.py
 confusion_matrix_Dogri.png	  i18n.py
 confusion_matrix_English.png	  kaggle
 confusion_matrix_Gujarati.png	  model_pipeline_artifacts
 confusion_matrix_Hindi.png	  model_pipeline.py
 confusion_matrix_Kannada.png	  README.md
 confusion_matrix_Malayalam.png   requirements-kaggle.txt
 confusion_matrix_Manipuri.png	  requirements-training.txt
 confusion_matrix_Marathi.png	  requirements.txt
 confusion_matrix_Nepali.png	  run_validation_ablations.py
 confusion_matrix_Odia.png	  test_analyz

In [4]:
%cd /content/indic-fake-voice-detection

print("REPORT DOWNLOAD FUNCTION:")
!sed -n '340,390p' app.py

print("\nPDF REPORT GENERATION FUNCTIONS:")
!grep -n -E "^def generate_|^def render_" evidence_report.py

print("\nREPORT DEPENDENCIES:")
!grep -n -i "reportlab\|streamlit\|torch\|transformers" requirements.txt

/content/indic-fake-voice-detection
REPORT DOWNLOAD FUNCTION:
            st.warning(t("robustness_no_comparisons", lang))
        elif report.verdict_change_count:
            st.warning(
                t("robustness_summary_changed", lang).format(
                    count=report.verdict_change_count,
                    total=report.transformed_success_count,
                )
            )
        else:
            st.success(t("robustness_summary_stable", lang))

        st.caption(
            t("robustness_max_change", lang).format(
                value=report.max_absolute_delta * 100
            )
        )
        if report.failed_conditions:
            failed = ", ".join(condition.label for condition in report.failed_conditions)
            st.warning(t("robustness_partial_failure", lang).format(conditions=failed))
        st.caption(t("robustness_limit", lang))


def render_report_download(result, filename: str, lang: str) -> None:
    try:
        report_pdf = evidence_r

In [5]:
%cd /content/indic-fake-voice-detection

print("PDF FUNCTION CALLS:")
!grep -n -B 5 -A 5 "render_report_download" app.py

print("\nANALYSIS REPORT GENERATOR:")
!sed -n '1,150p' evidence_report.py

print("\nEVIDENCE REPORT GENERATOR:")
!sed -n '260,340p' evidence_report.py

/content/indic-fake-voice-detection
PDF FUNCTION CALLS:
357-            failed = ", ".join(condition.label for condition in report.failed_conditions)
358-            st.warning(t("robustness_partial_failure", lang).format(conditions=failed))
359-        st.caption(t("robustness_limit", lang))
360-
361-
362:def render_report_download(result, filename: str, lang: str) -> None:
363-    try:
364-        report_pdf = evidence_report.generate_analysis_report(result, filename)
365-    except Exception as exc:  # noqa: BLE001
366-        LOGGER.exception("PDF report generation failed", exc_info=exc)
367-        st.warning(t("pdf_error", lang))
--
747-        if st.session_state.robustness_error:
748-            st.warning(st.session_state.robustness_error)
749-        if robustness_is_current:
750-            render_robustness_report(st.session_state.robustness_report, lang)
751-
752:        render_report_download(
753-            st.session_state.result,
754-            st.session_state.audio

In [6]:
%cd /content/indic-fake-voice-detection

print("CALL ANALYSIS FUNCTION:")
!grep -n "def render_call_analysis" app.py

print("\nCALL ANALYSIS PDF/CSV SECTION:")
!grep -n -B 12 -A 25 "call_csv_button" app.py

print("\nCALL ANALYSIS REPORT DATA STRUCTURE:")
!sed -n '1,180p' call_analysis.py

print("\nEXISTING PDF REPORT CONTENT:")
!sed -n '140,275p' evidence_report.py

/content/indic-fake-voice-detection
CALL ANALYSIS FUNCTION:
395:def render_call_analysis(lang: str) -> None:

CALL ANALYSIS PDF/CSV SECTION:
595-            "start_seconds": score.start_seconds,
596-            "end_seconds": score.end_seconds,
597-            "raw_logit": score.raw_logit,
598-            "p_fake": score.p_fake,
599-            "verdict": score.verdict,
600-        })
601-
602-    safe_stem = Path(
603-        st.session_state.call_audio_filename or "call_recording"
604-    ).stem
605-
606-    st.download_button(
607:        label=t("call_csv_button", lang),
608-        data=csv_buffer.getvalue(),
609-        file_name=f"{safe_stem}_call_analysis.csv",
610-        mime="text/csv",
611-        use_container_width=True,
612-    )
613-
614-def render_app() -> None:
615-    lang = current_lang()
616-    render_header(lang)
617-    render_hero(lang)
618-
619-    mode = st.radio(
620-        t("mode_label", lang),
621-        options=["single", "call"],
622-        format_fu

In [7]:
%cd /content/indic-fake-voice-detection

print("1. Call Analysis function location:")
!grep -n "def render_call_analysis" app.py

print("\n2. CSV download location:")
!grep -n -B 5 -A 12 "call_csv_button" app.py

print("\n3. Call Analysis report fields:")
!grep -n -A 45 "class CallAnalysisReport" call_analysis.py

/content/indic-fake-voice-detection
1. Call Analysis function location:
395:def render_call_analysis(lang: str) -> None:

2. CSV download location:
602-    safe_stem = Path(
603-        st.session_state.call_audio_filename or "call_recording"
604-    ).stem
605-
606-    st.download_button(
607:        label=t("call_csv_button", lang),
608-        data=csv_buffer.getvalue(),
609-        file_name=f"{safe_stem}_call_analysis.csv",
610-        mime="text/csv",
611-        use_container_width=True,
612-    )
613-
614-def render_app() -> None:
615-    lang = current_lang()
616-    render_header(lang)
617-    render_hero(lang)
618-
619-    mode = st.radio(

3. Call Analysis report fields:
168:class CallAnalysisReport:
169-    """Complete analysis results for one phone-call recording."""
170-
171-    duration_seconds: float
172-    speech_duration_seconds: float
173-    windows_analysed: int
174-    flagged_windows: int
175-    flagged_share: float
176-    highest_p_fake: float
177-    highes

In [8]:
%cd /content/indic-fake-voice-detection

print("WINDOW SCORE FIELDS:")
!sed -n '145,180p' call_analysis.py

print("\nCALL ANALYSIS DOWNLOAD BLOCK:")
!sed -n '575,613p' app.py

print("\nEXISTING PDF TRANSLATION LABELS:")
!grep -n '"pdf_heading"\|"pdf_body"\|"pdf_button"' i18n.py | head -12

/content/indic-fake-voice-detection
WINDOW SCORE FIELDS:
                    start_seconds=start,
                    end_seconds=start + duration,
                    start_sample=start_sample,
                    end_sample=end_sample,
                )
            )
            start += hop_seconds

    return windows


@dataclass(frozen=True)
class WindowScore:
    """Fake-voice score for one analysis window."""

    start_seconds: float
    end_seconds: float
    raw_logit: float
    p_fake: float
    verdict: str


@dataclass(frozen=True)
class CallAnalysisReport:
    """Complete analysis results for one phone-call recording."""

    duration_seconds: float
    speech_duration_seconds: float
    windows_analysed: int
    flagged_windows: int
    flagged_share: float
    highest_p_fake: float
    highest_score_timestamp: float
    window_scores: List[WindowScore]
    flagged_ranges: List[SpeechRegion]


CALL ANALYSIS DOWNLOAD BLOCK:
    else:
        st.info(t("call_no_flagged", l

In [9]:
%cd /content/indic-fake-voice-detection

print("APP IMPORTS:")
!sed -n '1,45p' app.py

print("\nPDF MODULE END:")
!tail -35 evidence_report.py

/content/indic-fake-voice-detection
APP IMPORTS:
"""Voice Guard Streamlit application.

Every analysis is bound to the SHA-256 hash of the selected upload. Replacing
the file clears the previous result and report before the page is rendered.
"""

from __future__ import annotations

import hashlib
import html
import logging
from pathlib import Path

try:
    import truststore

    truststore.inject_into_ssl()
except ImportError:
    pass

import streamlit as st

import call_analysis
import codec_robustness
import evidence_report
import i18n
import ui_kit
from i18n import t


LOGGER = logging.getLogger("voice_guard.app")
MODEL_IMPORT_ERROR: Exception | None = None

try:
    import model_pipeline as mp
except Exception as exc:  # noqa: BLE001 - replaced with a safe public message
    mp = None
    MODEL_IMPORT_ERROR = exc
    LOGGER.exception("Model pipeline import failed")


st.set_page_config(
    page_title="Voice Guard · AI voice analysis",
    page_icon="🛡️",
    layout="wide",

PDF 

In [10]:
%cd /content/indic-fake-voice-detection

from pathlib import Path
import py_compile

app_path = Path("app.py")
pdf_path = Path("evidence_report.py")

app_text = app_path.read_text()
pdf_text = pdf_path.read_text()

old_download = '''    csv_buffer = io.StringIO()

    writer = csv.DictWriter(
        csv_buffer,
        fieldnames=[
            "start_seconds",
            "end_seconds",
            "raw_logit",
            "p_fake",
            "verdict",
        ],
    )

    writer.writeheader()

    for score in report.window_scores:
        writer.writerow({
            "start_seconds": score.start_seconds,
            "end_seconds": score.end_seconds,
            "raw_logit": score.raw_logit,
            "p_fake": score.p_fake,
            "verdict": score.verdict,
        })

    safe_stem = Path(
        st.session_state.call_audio_filename or "call_recording"
    ).stem

    st.download_button(
        label=t("call_csv_button", lang),
        data=csv_buffer.getvalue(),
        file_name=f"{safe_stem}_call_analysis.csv",
        mime="text/csv",
        use_container_width=True,
    )
'''

new_download = '''    safe_stem = Path(
        st.session_state.call_audio_filename or "call_recording"
    ).stem

    try:
        call_report_pdf = evidence_report.generate_call_analysis_report(
            report,
            st.session_state.call_audio_filename or "call_recording",
        )
    except Exception as exc:  # noqa: BLE001
        LOGGER.exception("Call Analysis PDF generation failed", exc_info=exc)
        st.warning(t("pdf_error", lang))
    else:
        st.download_button(
            label=t("pdf_button", lang),
            data=call_report_pdf,
            file_name=f"voice_guard_{safe_stem}_call_analysis.pdf",
            mime="application/pdf",
            use_container_width=True,
        )
'''

assert old_download in app_text, "Could not find the exact CSV block. No files changed."
assert "def generate_call_analysis_report(" not in pdf_text, "Call Analysis PDF generator already exists."

pdf_function = '''

def generate_call_analysis_report(report: Any, filename: str = "call_recording") -> bytes:
    """Create a PDF report for a full phone-call analysis."""
    buffer = BytesIO()
    doc = SimpleDocTemplate(
        buffer,
        pagesize=A4,
        rightMargin=18 * mm,
        leftMargin=18 * mm,
        topMargin=14 * mm,
        bottomMargin=19 * mm,
        title="Voice Guard Call Analysis Report",
        author="Voice Guard",
        subject="Automated call-level AI voice analysis",
    )

    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        "CallReportTitle",
        parent=styles["Title"],
        alignment=TA_CENTER,
        fontName="Helvetica-Bold",
        fontSize=18,
        leading=21,
        textColor=INK,
        spaceAfter=5,
    )
    subtitle_style = ParagraphStyle(
        "CallReportSubtitle",
        parent=styles["Normal"],
        alignment=TA_CENTER,
        fontSize=9,
        leading=12,
        textColor=MUTED,
        spaceAfter=10,
    )
    heading_style = ParagraphStyle(
        "CallReportHeading",
        parent=styles["Heading2"],
        fontName="Helvetica-Bold",
        fontSize=12,
        leading=15,
        textColor=INK,
        spaceBefore=9,
        spaceAfter=5,
        keepWithNext=True,
    )
    body_style = ParagraphStyle(
        "CallReportBody",
        parent=styles["BodyText"],
        fontSize=9,
        leading=13,
        textColor=INK,
    )
    small_style = ParagraphStyle(
        "CallReportSmall",
        parent=body_style,
        fontSize=7.5,
        leading=9.5,
        wordWrap="CJK",
    )
    warning_style = ParagraphStyle(
        "CallReportWarning",
        parent=body_style,
        fontName="Helvetica-Bold",
        textColor=AMBER_TEXT,
    )

    def seconds(value: float) -> str:
        total = max(0, int(value))
        return f"{total // 60:02d}:{total % 60:02d}"

    def make_cell(value: Any, style=small_style) -> Paragraph:
        return Paragraph(_safe(value), style)

    story = [
        Paragraph("Voice Guard", title_style),
        Paragraph("Phone Call Analysis Report", subtitle_style),
        HRFlowable(width="100%", thickness=1, color=YELLOW),
        Spacer(1, 8),
        Paragraph("Recording summary", heading_style),
    ]

    summary_rows = [
        ("Audio file", Path(filename).name),
        ("Recording duration", f"{report.duration_seconds:.2f} seconds"),
        ("Estimated speech duration", f"{report.speech_duration_seconds:.2f} seconds"),
        ("Analysis windows", str(report.windows_analysed)),
        ("Flagged windows", str(report.flagged_windows)),
        ("Flagged share", f"{report.flagged_share:.1%}"),
        ("Highest AI likelihood score", f"{report.highest_p_fake:.1%}"),
        ("Timestamp of highest score", seconds(report.highest_score_timestamp)),
    ]

    summary_table = Table(
        [[make_cell(k), make_cell(v)] for k, v in summary_rows],
        colWidths=[52 * mm, 115 * mm],
        hAlign="LEFT",
    )
    summary_table.setStyle(
        TableStyle([
            ("BACKGROUND", (0, 0), (0, -1), CREAM),
            ("FONTNAME", (0, 0), (0, -1), "Helvetica-Bold"),
            ("GRID", (0, 0), (-1, -1), 0.5, BORDER),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("LEFTPADDING", (0, 0), (-1, -1), 7),
            ("RIGHTPADDING", (0, 0), (-1, -1), 7),
            ("TOPPADDING", (0, 0), (-1, -1), 5),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ])
    )
    story.append(summary_table)

    story.append(Paragraph("Window-by-window results", heading_style))
    window_rows = [[
        make_cell("Start"),
        make_cell("End"),
        make_cell("AI likelihood"),
        make_cell("Verdict"),
    ]]

    for score in report.window_scores:
        window_rows.append([
            make_cell(seconds(score.start_seconds)),
            make_cell(seconds(score.end_seconds)),
            make_cell(f"{score.p_fake:.1%}"),
            make_cell(score.verdict.capitalize()),
        ])

    if len(window_rows) == 1:
        window_rows.append([make_cell("No windows analysed"), "", "", ""])

    window_table = Table(
        window_rows,
        colWidths=[35 * mm, 35 * mm, 48 * mm, 49 * mm],
        repeatRows=1,
        hAlign="LEFT",
    )
    window_table.setStyle(
        TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), CREAM),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("GRID", (0, 0), (-1, -1), 0.5, BORDER),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("LEFTPADDING", (0, 0), (-1, -1), 6),
            ("RIGHTPADDING", (0, 0), (-1, -1), 6),
            ("TOPPADDING", (0, 0), (-1, -1), 5),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ])
    )
    story.append(window_table)

    story.append(Paragraph("Flagged time ranges", heading_style))
    if report.flagged_ranges:
        range_rows = [[make_cell("Start"), make_cell("End")]]
        for item in report.flagged_ranges:
            range_rows.append([
                make_cell(seconds(item.start_seconds)),
                make_cell(seconds(item.end_seconds)),
            ])
        range_table = Table(
            range_rows,
            colWidths=[83.5 * mm, 83.5 * mm],
            repeatRows=1,
        )
        range_table.setStyle(
            TableStyle([
                ("BACKGROUND", (0, 0), (-1, 0), CREAM),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("GRID", (0, 0), (-1, -1), 0.5, BORDER),
                ("LEFTPADDING", (0, 0), (-1, -1), 7),
                ("RIGHTPADDING", (0, 0), (-1, -1), 7),
                ("TOPPADDING", (0, 0), (-1, -1), 5),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
            ])
        )
        story.append(range_table)
    else:
        story.append(Paragraph("No windows were flagged in this analysis.", body_style))

    story.extend([
        Spacer(1, 10),
        Paragraph("Important limitation", heading_style),
        Table(
            [[Paragraph(
                "This report contains automated model estimates, not proof that a "
                "recording is genuine or AI-generated. Scores can be affected by "
                "background noise, compression, short speech, recording quality, "
                "and voice generators not represented in the model's training data. "
                "Review the recording and consider independent verification before "
                "making important decisions.",
                warning_style,
            )]],
            colWidths=[167 * mm],
            style=TableStyle([
                ("BACKGROUND", (0, 0), (-1, -1), AMBER_BG),
                ("BOX", (0, 0), (-1, -1), 1, colors.HexColor("#B85C00")),
                ("LEFTPADDING", (0, 0), (-1, -1), 9),
                ("RIGHTPADDING", (0, 0), (-1, -1), 9),
                ("TOPPADDING", (0, 0), (-1, -1), 8),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
            ]),
        ),
    ])

    doc.build(story, onFirstPage=_footer, onLaterPages=_footer)
    return buffer.getvalue()
'''

# Back up the two files before applying the verified replacements.
Path("app.py.before_call_pdf.py").write_text(app_text)
Path("evidence_report.py.before_call_pdf.py").write_text(pdf_text)

app_path.write_text(app_text.replace(old_download, new_download, 1))
pdf_path.write_text(pdf_text + pdf_function)

# Verify Python syntax.
py_compile.compile("app.py", doraise=True)
py_compile.compile("evidence_report.py", doraise=True)

print("SUCCESS: Call Analysis PDF generator added.")
print("SUCCESS: Call Analysis CSV download replaced with PDF download.")
print("SUCCESS: Existing Single Clip PDF code was not changed.")
print("SUCCESS: app.py and evidence_report.py compile without errors.")
print("Backups created:")
print(" - app.py.before_call_pdf.py")
print(" - evidence_report.py.before_call_pdf.py")

/content/indic-fake-voice-detection
SUCCESS: Call Analysis PDF generator added.
SUCCESS: Call Analysis CSV download replaced with PDF download.
SUCCESS: Existing Single Clip PDF code was not changed.
SUCCESS: app.py and evidence_report.py compile without errors.
Backups created:
 - app.py.before_call_pdf.py
 - evidence_report.py.before_call_pdf.py


In [12]:
%cd /content/indic-fake-voice-detection
!pip install "reportlab>=4.0,<5"

/content/indic-fake-voice-detection
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 13.6 MB/s  0:00:00


In [15]:
%cd /content/indic-fake-voice-detection

from pathlib import Path

pdf_path = Path("evidence_report.py")
source = pdf_path.read_text()

if "from pathlib import Path" not in source:
    source = source.replace(
        "from io import BytesIO\n",
        "from io import BytesIO\nfrom pathlib import Path\n",
        1,
    )
    pdf_path.write_text(source)

import py_compile
py_compile.compile("evidence_report.py", doraise=True)

print("SUCCESS: Path import added and PDF module compiles.")

/content/indic-fake-voice-detection
SUCCESS: Path import added and PDF module compiles.


In [17]:
%cd /content/indic-fake-voice-detection

import importlib
import evidence_report

importlib.reload(evidence_report)

print("Path import available:", hasattr(evidence_report, "Path"))

pdf_data = evidence_report.generate_call_analysis_report(
    test_report,
    "test_call_recording.mp3",
)

print("PDF generated:", isinstance(pdf_data, bytes))
print("PDF size:", len(pdf_data), "bytes")
print("Valid PDF header:", pdf_data[:5] == b"%PDF-")

/content/indic-fake-voice-detection
Path import available: True
PDF generated: True
PDF size: 3189 bytes
Valid PDF header: True


In [18]:
%cd /content/indic-fake-voice-detection

!git diff --check
!git status --short
!grep -n "generate_call_analysis_report\|call_csv_button\|call_analysis.pdf" app.py evidence_report.py

/content/indic-fake-voice-detection
 M app.py
 M evidence_report.py
?? app.py.before_call_pdf.py
?? evidence_report.py.before_call_pdf.py
app.py:583:        call_report_pdf = evidence_report.generate_call_analysis_report(
app.py:594:            file_name=f"voice_guard_{safe_stem}_call_analysis.pdf",
evidence_report.py:280:def generate_call_analysis_report(report: Any, filename: str = "call_recording") -> bytes:


In [20]:
%cd /content/indic-fake-voice-detection

!python --version
!pip show streamlit
!head -20 requirements.txt

/content/indic-fake-voice-detection
Python 3.13.18
# Runtime dependencies for app.py and analyze_bytes(). Upper bounds prevent a
# new major release from silently changing the public Kaggle demo. Torch and
# torchaudio must resolve to the same release family.
streamlit>=1.40,<2
torch>=2.2,<3
torchaudio>=2.2,<3
transformers>=4.40,<6
numpy>=1.26,<3
soundfile>=0.12,<1
pydub>=0.25,<1
reportlab>=4.0,<5
truststore>=0.9,<1


In [21]:
!pip install "streamlit>=1.40,<2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 27.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 38.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [22]:
import streamlit
import reportlab

print("Streamlit:", streamlit.__version__)
print("ReportLab:", reportlab.Version)

Streamlit: 1.67.0
ReportLab: 4.5.1


In [26]:
%cd /content/indic-fake-voice-detection
!streamlit run app.py --server.port 8501 --server.headless true

/content/indic-fake-voice-detection


2026-11-18 14:36:10.629 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.13.144.77:8501

2026-11-18 14:36:35.283 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-euw4b2-mstp47jswici-b.europe-west4.prod.r.colab.dev, host=m-s-euw4b2-mstp47jswici.europe-west4-b.c.codatalab-user-runtimes.internal:8007
2026-11-18 14:36:37.951 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-euw4b2-mstp47jswici-b.europe-west4.prod.r.colab.dev, host=m-s-euw4b2-mstp47jswici.europe-west4-b.c.codatalab-user-runtimes.internal:8007
2026-11-18 14:36:39.560 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-euw4b2-mstp47jswici-b.europe-west4.prod.r.colab.dev, host=m-s-euw4b2-mstp47jswici.europe-west4-b.c.coda

In [25]:
from google.colab import output

url = output.eval_js("google.colab.kernel.proxyPort(8501)")
print("Open Streamlit in a separate browser tab:")
print(url)

from IPython.display import HTML, display
display(HTML(f'<a href="{url}" target="_blank">Open Voice Guard Streamlit App</a>'))

Open Streamlit in a separate browser tab:
https://8501-m-s-euw4b2-mstp47jswici-b.europe-west4.prod.r.colab.dev


In [27]:
import requests

url = "http://localhost:8501/_stcore/health"

try:
    response = requests.get(url, timeout=10)
    print("Status code:", response.status_code)
    print("Response:", response.text)
except Exception as e:
    print("Connection error:", e)

Connection error: HTTPConnectionPool(host='localhost', port=8501): Max retries exceeded with url: /_stcore/health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7a4f49300c20>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [28]:
import subprocess

print("Checking Streamlit process...")
subprocess.run(["pgrep", "-af", "streamlit"], text=True)

print("\nChecking port 8501...")
subprocess.run(["bash", "-lc", "ss -ltnp | grep 8501 || echo 'Nothing listening on port 8501'"])

Checking Streamlit process...

Checking port 8501...


CompletedProcess(args=['bash', '-lc', "ss -ltnp | grep 8501 || echo 'Nothing listening on port 8501'"], returncode=0)

In [29]:
import subprocess
from pathlib import Path

%cd /content/indic-fake-voice-detection

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("Streamlit process started. PID:", process.pid)

/content/indic-fake-voice-detection
Streamlit process started. PID: 7816


In [30]:
import time
import requests
import subprocess

time.sleep(5)

try:
    response = requests.get("http://127.0.0.1:8501/_stcore/health", timeout=10)
    print("Server status:", response.status_code)
    print("Response:", response.text)
except Exception as e:
    print("Server not responding:", e)
    print("\nRecent Streamlit logs:")
    subprocess.run(["tail", "-n", "40", "/content/streamlit.log"])

Server status: 200
Response: ok


In [31]:
from google.colab import output
from IPython.display import HTML, display

url = output.eval_js("google.colab.kernel.proxyPort(8501)")

print("Streamlit URL:", url)
display(HTML(
    f'<a href="{url}" target="_blank" '
    'style="font-size:18px;">Open Voice Guard in a new tab</a>'
))

Streamlit URL: https://8501-m-s-euw4b2-mstp47jswici-b.europe-west4.prod.r.colab.dev


In [32]:
!tail -n 80 /content/streamlit.log



2026-11-18 14:38:56.092 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.13.144.77:8501

2026-11-18 14:38:56.914 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-euw4b2-mstp47jswici-b.europe-west4.prod.r.colab.dev, host=m-s-euw4b2-mstp47jswici.europe-west4-b.c.codatalab-user-runtimes.internal:8007
2026-11-18 14:39:12.185 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-euw4b2-mstp47jswici-b.europe-west4.prod.r.colab.dev, host=m-s-euw4b2-mstp47jswici.europe-west4-b.c.codatalab-user-runtimes.internal:8007
2026-11-18 14:39:31.000 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-euw4b2-mstp47jswici-b.europe-west4.prod.r.colab.dev, host=m-s-euw4b2-mstp47jswici.europe-west4-b.c.codatalab-user-runtimes.internal:80

In [33]:
import subprocess
import os
import time

# Stop the existing Streamlit process
subprocess.run(["pkill", "-f", "streamlit run app.py"])

time.sleep(2)

# Restart with Colab proxy compatibility
log = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("New Streamlit PID:", process.pid)

New Streamlit PID: 8642


In [34]:
import time
import requests

time.sleep(5)

response = requests.get(
    "http://127.0.0.1:8501/_stcore/health",
    timeout=10
)

print("Status:", response.status_code)
print("Response:", response.text)

Status: 200
Response: ok


In [35]:
from google.colab import output
from IPython.display import HTML, display

url = output.eval_js("google.colab.kernel.proxyPort(8501)")

display(HTML(
    f'<a href="{url}" target="_blank">Open Voice Guard</a>'
))

In [36]:
%cd /content/indic-fake-voice-detection

!git branch --show-current
!git status --short
!git diff --check
!git diff --stat

/content/indic-fake-voice-detection
feature/long-call-analysis
 M app.py
 M evidence_report.py
?? app.py.before_call_pdf.py
?? evidence_report.py.before_call_pdf.py
 app.py             |  47 ++++--------
 evidence_report.py | 209 +++++++++++++++++++++++++++++++++++++++++++++++++++++
 2 files changed, 225 insertions(+), 31 deletions(-)


In [37]:
%cd /content/indic-fake-voice-detection

!git add app.py evidence_report.py
!git commit -m "Add PDF reports for Call Analysis"

/content/indic-fake-voice-detection
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@7b36cd087342.(none)')


In [38]:
%cd /content/indic-fake-voice-detection

!git config user.name "Ashitha"
!git config user.email "rajashitha310@gmail.com"

!git config user.name
!git config user.email

/content/indic-fake-voice-detection
Ashitha
rajashitha310@gmail.com


In [39]:
%cd /content/indic-fake-voice-detection

!git commit -m "Add PDF reports for Call Analysis"

/content/indic-fake-voice-detection
[feature/long-call-analysis f06f0bb] Add PDF reports for Call Analysis
 2 files changed, 225 insertions(+), 31 deletions(-)


In [40]:
import getpass
import os
import subprocess
import tempfile

token = getpass.getpass("Enter your GitHub Personal Access Token: ")

with tempfile.NamedTemporaryFile(mode="w", delete=False) as f:
    f.write("""#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")
    askpass_path = f.name

os.chmod(askpass_path, 0o700)

env = os.environ.copy()
env["GIT_ASKPASS"] = askpass_path
env["GITHUB_TOKEN"] = token
env["GIT_TERMINAL_PROMPT"] = "0"

try:
    result = subprocess.run(
        ["git", "push", "origin", "feature/long-call-analysis"],
        capture_output=True,
        text=True,
        env=env
    )
    print(result.stdout)
    print(result.stderr)
    print("Push exit code:", result.returncode)
finally:
    os.remove(askpass_path)
    token = None
    env.pop("GITHUB_TOKEN", None)

Enter your GitHub Personal Access Token: ··········

To https://github.com/Shruti-sivaramakrishnan/indic-fake-voice-detection.git
   4ec3ecd..f06f0bb  feature/long-call-analysis -> feature/long-call-analysis

Push exit code: 0


In [41]:
%cd /content/indic-fake-voice-detection

print("CURRENT BRANCH")
!git branch --show-current

print("\nLATEST COMMIT")
!git log -1 --oneline

print("\nWORKING TREE STATUS")
!git status --short

print("\nLATEST COMMITS")
!git log -5 --oneline

/content/indic-fake-voice-detection
CURRENT BRANCH
feature/long-call-analysis

LATEST COMMIT
f06f0bb (HEAD -> feature/long-call-analysis, origin/feature/long-call-analysis) Add PDF reports for Call Analysis

WORKING TREE STATUS
?? app.py.before_call_pdf.py
?? evidence_report.py.before_call_pdf.py

LATEST COMMITS
f06f0bb (HEAD -> feature/long-call-analysis, origin/feature/long-call-analysis) Add PDF reports for Call Analysis
4ec3ecd Fix Call Analysis layout and short-window scoring
7723489 Add Call Analysis Streamlit interface
6af6647 Add long-call analysis core
55570bb (origin/main, origin/HEAD) transmission-robustness


In [42]:
%cd /content/indic-fake-voice-detection

import os
import py_compile

files = [
    "app.py",
    "call_analysis.py",
    "evidence_report.py",
    "model_pipeline.py",
    "voice_guard_core.py",
    "i18n.py",
]

print("PYTHON SYNTAX CHECK")
for file in files:
    if os.path.exists(file):
        py_compile.compile(file, doraise=True)
        print(f"PASS: {file}")
    else:
        print(f"MISSING: {file}")

print("\nPROJECT FILES")
!ls -lah

/content/indic-fake-voice-detection
PYTHON SYNTAX CHECK
PASS: app.py
PASS: call_analysis.py
PASS: evidence_report.py
PASS: model_pipeline.py
PASS: voice_guard_core.py
PASS: i18n.py

PROJECT FILES
total 1.3M
drwxr-xr-x 13 root root 4.0K Nov 18 14:23  .
drwxr-xr-x  1 root root 4.0K Nov 18 14:38  ..
-rw-r--r--  1 root root  26K Nov 18 14:23  app.py
-rw-r--r--  1 root root  26K Nov 18 14:23  app.py.before_call_pdf.py
-rw-r--r--  1 root root  11K Nov 18 14:16  call_analysis.py
drwxr-xr-x  2 root root 4.0K Nov 18 14:16  checkpoints
-rw-r--r--  1 root root 9.2K Nov 18 14:16  codec_robustness.py
-rw-r--r--  1 root root  51K Nov 18 14:16  confusion_matrix_Assamese.png
-rw-r--r--  1 root root  48K Nov 18 14:16  confusion_matrix_Bengali.png
-rw-r--r--  1 root root  47K Nov 18 14:16  confusion_matrix_Bodo.png
-rw-r--r--  1 root root  48K Nov 18 14:16  confusion_matrix_Dogri.png
-rw-r--r--  1 root root  46K Nov 18 14:16  confusion_matrix_English.png
-rw-r--r--  1 root root  48K Nov 18 14:16  confus

In [43]:
%cd /content/indic-fake-voice-detection

!python -m pytest -q

/content/indic-fake-voice-detection

==================================== ERRORS ====================================
________________ ERROR collecting tests/test_evidence_report.py ________________
ImportError while importing test module '/content/indic-fake-voice-detection/tests/test_evidence_report.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_evidence_report.py:3: in <module>
    from pypdf import PdfReader
E   ModuleNotFoundError: No module named 'pypdf'
=========================== short test summary info ============================
ERROR tests/test_evidence_report.py
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error in 3.47s


In [44]:
%cd /content/indic-fake-voice-detection

!grep -n -i "pypdf\|reportlab" requirements.txt requirements-training.txt requirements-kaggle.txt
!sed -n '1,100p' tests/test_evidence_report.py

/content/indic-fake-voice-detection
requirements.txt:11:reportlab>=4.0,<5
requirements-training.txt:8:pypdf>=5,<7
from types import SimpleNamespace

from pypdf import PdfReader

from evidence_report import generate_analysis_report


def test_analysis_report_is_one_page_and_uses_safe_language() -> None:
    result = SimpleNamespace(
        verdict="fake",
        confidence=0.99,
        raw_score=4.2,
        audio_hash="a" * 64,
        metadata={
            "p_fake_calibrated": 0.97,
            "duration_seconds": 4.2,
            "sample_rate_in": 16000,
            "frontend": "frontend",
            "checkpoint": "best_model.pt",
            "checkpoint_epoch": 11,
            "temperature": 1.26,
            "decision_policy_id": "legacy-symmetric-abstention-v1",
            "audio_quality": {
                "active_speech_seconds_estimate": 3.5,
                "rms": 0.05,
                "peak": 0.7,
                "quality_warning": None,
            },
        },
    )


In [45]:
%pip install "pypdf>=5,<7"


In [46]:
%cd /content/indic-fake-voice-detection

!python -m pytest -q

/content/indic-fake-voice-detection
.................................                                        [100%]
33 passed in 3.32s


In [47]:
import requests

try:
    response = requests.get("http://127.0.0.1:8501/_stcore/health", timeout=5)
    print("HTTP status:", response.status_code)
    print("Response:", response.text)
except requests.RequestException as error:
    print("Streamlit is not responding:", error)

HTTP status: 200
Response: ok


In [48]:
from google.colab import output
from IPython.display import HTML, display

url = output.eval_js("google.colab.kernel.proxyPort(8501)")
display(HTML(f'<a href="{url}" target="_blank">Open Voice Guard</a>'))

In [49]:
%cd /content/indic-fake-voice-detection

print("=== DEPENDENCIES ===")
!cat requirements.txt
!echo "\n=== TRAINING DEPENDENCIES ==="
!cat requirements-training.txt

print("\n=== MODEL CHECKPOINTS ===")
!find checkpoints model_pipeline_artifacts -maxdepth 3 -type f -printf '%p (%s bytes)\n' 2>/dev/null

print("\n=== PROJECT MODULES ===")
!find data_pipeline evaluation tests docs -maxdepth 2 -type f 2>/dev/null | sort

print("\n=== MAIN MODULE IMPORTS ===")
!python - <<'PY'
import ast

for filename in [
    "app.py",
    "call_analysis.py",
    "evidence_report.py",
    "model_pipeline.py",
    "voice_guard_core.py",
    "i18n.py",
]:
    print(f"\n--- {filename} ---")
    with open(filename, encoding="utf-8") as f:
        tree = ast.parse(f.read())
    imports = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            imports.append(ast.unparse(node))
    print("\n".join(sorted(set(imports))))
PY

/content/indic-fake-voice-detection
=== DEPENDENCIES ===
# Runtime dependencies for app.py and analyze_bytes(). Upper bounds prevent a
# new major release from silently changing the public Kaggle demo. Torch and
# torchaudio must resolve to the same release family.
streamlit>=1.40,<2
torch>=2.2,<3
torchaudio>=2.2,<3
transformers>=4.40,<6
numpy>=1.26,<3
soundfile>=0.12,<1
pydub>=0.25,<1
reportlab>=4.0,<5
truststore>=0.9,<1
\n=== TRAINING DEPENDENCIES ===
# Install after requirements.txt for training, evaluation, and tests.
-r requirements.txt
datasets>=3.0,<5
scikit-learn>=1.4,<2
pandas>=2.1,<3
matplotlib>=3.8,<4
pytest>=8,<9
pypdf>=5,<7

=== MODEL CHECKPOINTS ===
checkpoints/step6_summary.json (3237 bytes)
checkpoints/per_language_eer.json (349 bytes)
checkpoints/best_model.pt (5174479 bytes)
model_pipeline_artifacts/split_metadata_verified.json (1148119 bytes)

=== PROJECT MODULES ===
data_pipeline/build_group_safe_split.py
data_pipeline/degraded/.gitkeep
data_pipeline/degraded/musan_

NameError: name 'PY' is not defined

In [50]:
%cd /content/indic-fake-voice-detection

from pathlib import Path
import ast

# Read project documentation
for filename in [
    "README.md",
    "docs/TEAM_SYNC_LOG.md",
    "docs/TEST_SET_RESULTS.md",
    "docs/KAGGLE_DEPLOYMENT.md",
    "docs/V2_TRAINING.md",
]:
    path = Path(filename)
    print(f"\n{'='*25} {filename} {'='*25}")
    if path.exists():
        print(path.read_text(encoding="utf-8")[:10000])
    else:
        print("File not found")

# Extract classes and function names from the main modules
for filename in [
    "app.py",
    "call_analysis.py",
    "evidence_report.py",
    "model_pipeline.py",
    "voice_guard_core.py",
    "codec_robustness.py",
    "i18n.py",
]:
    path = Path(filename)
    if not path.exists():
        continue

    tree = ast.parse(path.read_text(encoding="utf-8"))
    print(f"\n{'='*25} STRUCTURE: {filename} {'='*25}")

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            print(f"Function: {node.name}")
        elif isinstance(node, ast.ClassDef):
            print(f"Class: {node.name}")
            for child in node.body:
                if isinstance(child, ast.FunctionDef):
                    print(f"  Method: {child.name}")

/content/indic-fake-voice-detection

========================= README.md =========================
# Voice Guard - Indic fake voice detection

Voice Guard is a research/demo pipeline for detecting synthetic speech across 16 Indian languages. It uses a frozen `facebook/wav2vec2-xls-r-300m` frontend and a trainable graph-attention classifier head, exposed through a Streamlit upload UI.

The head is **AASIST-inspired, not an implementation of the full AASIST architecture**. The current checkpoint is a historical v1 model and must not be treated as production-validated.

## Current status

- The upload path now rejects empty, silent, too-short, too-long, non-finite, and oversized audio before model inference.
- The UI returns `Likely genuine`, `Inconclusive`, or `Likely synthetic`, and labels its number as a model score rather than proof.
- Results and PDF downloads are bound to the exact SHA-256 hash of the currently selected audio.
- Automatic language identification is disabled. The rem

In [51]:
%cd /content/indic-fake-voice-detection

from pathlib import Path
import json

print("=== README SUMMARY ===")
readme = Path("README.md").read_text(errors="replace")
print(readme[:2500])

print("\n=== TEST SET RESULTS ===")
for f in [
    "evaluation/clean/per_language_test_metrics.json",
    "evaluation/robustness/robustness_summary.json",
    "checkpoints/step6_summary.json",
    "checkpoints/per_language_eer.json",
]:
    p = Path(f)
    print(f"\n{f}")
    if p.exists():
        try:
            data = json.loads(p.read_text())
            print(json.dumps(data, indent=2)[:1800])
        except Exception:
            print(p.read_text()[:1800])
    else:
        print("Not found")

print("\n=== MAIN FUNCTIONS ===")
import ast

for filename in [
    "call_analysis.py",
    "evidence_report.py",
    "voice_guard_core.py",
    "codec_robustness.py",
]:
    tree = ast.parse(Path(filename).read_text())
    names = [
        n.name for n in tree.body
        if isinstance(n, (ast.FunctionDef, ast.ClassDef))
    ]
    print(filename, ":", ", ".join(names))

/content/indic-fake-voice-detection
=== README SUMMARY ===
# Voice Guard - Indic fake voice detection

Voice Guard is a research/demo pipeline for detecting synthetic speech across 16 Indian languages. It uses a frozen `facebook/wav2vec2-xls-r-300m` frontend and a trainable graph-attention classifier head, exposed through a Streamlit upload UI.

The head is **AASIST-inspired, not an implementation of the full AASIST architecture**. The current checkpoint is a historical v1 model and must not be treated as production-validated.

## Current status

- The upload path now rejects empty, silent, too-short, too-long, non-finite, and oversized audio before model inference.
- The UI returns `Likely genuine`, `Inconclusive`, or `Likely synthetic`, and labels its number as a model score rather than proof.
- Results and PDF downloads are bound to the exact SHA-256 hash of the currently selected audio.
- Automatic language identification is disabled. The removed SpeechBrain path did not correctly 

In [52]:
%cd /content/indic-fake-voice-detection
!git diff HEAD origin/feature/long-call-analysis -- README.md

/content/indic-fake-voice-detection


In [53]:
%cd /content/indic-fake-voice-detection
!git diff HEAD origin/feature/long-call-analysis -- README.md


/content/indic-fake-voice-detection


In [54]:
%cd /content/indic-fake-voice-detection

!git fetch origin
!git diff origin/feature/long-call-analysis -- README.md

/content/indic-fake-voice-detection


In [55]:
import subprocess

repo = "/content/indic-fake-voice-detection"

subprocess.run(["git", "fetch", "origin"], cwd=repo)

result = subprocess.run(
    ["git", "diff", "HEAD", "origin/feature/long-call-analysis", "--", "README.md"],
    cwd=repo,
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout else "No README differences found.")

No README differences found.
